# Establishing Connections to Databases

In [1]:
bridge_sfn = '0130192'

## TIMS 'doticsqlp31' Data Source

In [2]:
import pandas as pd
import sqlalchemy as db
from sqlalchemy import create_engine, MetaData, text, inspect
from sqlalchemy.engine import URL

In [3]:
connection_string = (
    r"Driver=ODBC Driver 17 for SQL Server;"
    r"Server=doticsqlp31;"
    r"Database=TIMS;"
    r"Trusted_Connection=yes;"
)

connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

engine = create_engine(connection_url)
conn = engine.connect()

## 'DOTWarehousePrd' Data Source

In [4]:
connection_string = (
    r"Driver=SQL Server;"
    r"Server=DOTWarehousePrd;"
    r"Database=Warehouse;"
    r"Trusted_Connection=yes;"
)
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
engine = create_engine(connection_url)
conn = engine.connect()

In [ ]:
# Get data from 1985-1990 table - Huge, don't do this.
# hist85_90DataQuery = text("SELECT * from BMS_1985_1990")

In [ ]:
# hist_bridge_data = pd.read_sql(hist85_90DataQuery, conn)
# hist_bridge_data

## Oracle Load Rating Database

In [7]:
from sqlalchemy import create_engine, text
import json

In [8]:
def load_secrets(file_path):
    with open(file_path, 'r') as file:
        user_info = json.load(file)
    return user_info

In [9]:
file_path = '../secrets.json'
dane = load_secrets(file_path)

In [10]:
# Oracle connection string using service name instead of SID
oracle_connection_string = (
    f"oracle+cx_oracle://{dane['BRR_USN']}:{dane['BRR_PASS']}@{dane['BRR_SERVER']}:{dane['BRR_PORT']}/?service_name={dane['BRR_SERVICE']}"
)

In [11]:
# Create the engine
oracle_engine = create_engine(oracle_connection_string)

In [12]:
# Establish connection
oracle_conn = oracle_engine.connect()
print("Connection successful!")

Connection successful!


In [13]:
from sqlalchemy import inspect

# Create an inspector from the engine
inspector = inspect(oracle_engine)

# Get a list of table names
tables = inspector.get_table_names()
print("Tables in the database:")
for table in tables:
    print(table)

Tables in the database:


In [14]:
query = text("""
  SELECT DISTINCT owner
  FROM all_tables
  ORDER BY owner
""")

result = oracle_conn.execute(query)
schemas = [row[0] for row in result]
print("Accessible schemas:")

for schema in schemas:
     print(schema)

Accessible schemas:
BRIDGEWARE
CTXSYS
MDSYS
ODOTREF
SYS
SYSTEM
XDB


In [15]:
schema_list = ["BRIDGEWARE", "CTXSYS", "MDSYS", "ODOTREF", "SYS", "SYSTEM", 
               "XDB"]  # Add other schemas if you want to search deeper

for schema_name in schema_list:
    print(f"\nChecking for tables in schema: {schema_name}\n")
    query = text(f"""
    SELECT table_name
    FROM all_tables
    WHERE owner = '{schema_name}'
    """)
    result = oracle_conn.execute(query)
    tables = [row[0] for row in result]
    if tables:
        print(f"Tables in schema {schema_name}:")
        for table in tables:
            print(table)
    else:
        print(f"No tables found in schema {schema_name}.")


Checking for tables in schema: BRIDGEWARE

Tables in schema BRIDGEWARE:
ABW_BRIDGE_SUB_LRFDDSN_SETTING
ABW_MA_STRUSS_ELEM_LOSS_RANGE
ABW_MCB_SEG_DECK
ABW_MATL_CONC
ABW_MATL_PS_BAR
ABW_MATL_PS_STRAND
ABW_FL_FLOORBEAM_TRAVELWAY
ABW_FLINE_MBR
ABW_MCB_TEND_PROF_DEF_DETAIL
ABW_STL_ANGLE
ABW_STL_ANGLE_CONN
ABW_STL_BEAM_ASSEMBLY
ABW_WEB_DISTRIB_LOAD
BRIDGE
COPTIONS
MULTIMEDIA
ABW_TRUSS_LINE_MBR
ABW_TRUSS_LINE_SUPPORT
ABW_TRUSSDEF_ELEMENT_CONC_LOAD
ABW_RESULTS_CONC_LS_SUMMARY
ABW_RESULTS_CONC_XSEC_PROP
ABW_LIB_MATL_TIMBER_SAWN_ITEM
ABW_SYS_LRFD_LS
ABW_SUBSDEF_MODEL_SETTING
ABW_SUPER_BR_FORCE_SUB
ABW_CONC_CURB_SIDEWALK
ABW_CONC_RAILING
ABW_CONC_RAILING_LOC
ABW_BRIDGE_DIAPHRAGM_DEF_CON
ABW_SUBSDEF_FOUND_FK
ABW_STL_BUILTUP_IBEAM_DEF
ABW_STL_LONG_STIFF
ABW_EVENT_VEHICLE_TEMPLATE
ABW_LEG_ANAL_PT_REINF_CONC
ABW_RESULTS_CRIT_LOAD_LRFD
ABW_RESULTS_DL_ACTION
ABW_RESULTS_LL_ACTION
ABW_RESULTS_LS_SUMMARY_DETAIL
ABW_RESULTS_PS_CONC_STRESS
ABW_RESULTS_RC_SERVICE_STRESS
ABW_RESULTS_SPNG_MBR_ALT_XY
ABW_RESU

In [16]:
import pandas as pd
from sqlalchemy.sql import text

# Query to fetch the entire table
query = text("SELECT * FROM BRIDGEWARE.ABW_GIRDER_MBR")

# Execute the query and fetch the data into a DataFrame
result = oracle_conn.execute(query)
df = pd.DataFrame(result.fetchall(), columns=result.keys())

In [17]:
df

,bridge_id,struct_def_id,super_struct_mbr_id,struct_def_ref_line_id,settlement_load_case_id
0,15974,1,1,5,NaN
1,15974,1,2,6,NaN
2,15974,1,3,7,NaN
3,15974,1,4,8,NaN
4,15974,1,5,9,NaN
...,...,...,...,...,...
73449,17240,1,11,15,NaN
73450,17241,1,1,5,NaN
73451,17241,1,2,6,NaN
73452,17241,1,3,7,NaN


In [18]:
def get_table(schema, table):
    query = text(f"SELECT * FROM {schema}.{table}")
    result = oracle_conn.execute(query)
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    return df

In [19]:
get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFDDSN_SETTING")

,bridge_id,lrfd_design_setting_id,name,descr,bridge_lrfd_factor_id,preliminary_design_setting_ind,final_design_setting_ind,dynamic_load_allow_method_type,dla_simple_fatigue_frac_impact,dla_simple_other_ls_impact,vehicle_summary_display_type,last_change_timestamp,limit_num_dsgn_load_combo_ind,cw_num_loadcombo_axial_bending,ftg_num_loadcombo_brg_capacity,lrfd_analysis_module_guid,lrfd_spec_edition_guid
0,3332,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2016-01-29 16:55:09.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
1,3717,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2016-10-28 16:40:47.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
2,5564,1,Final Design Setting (US),Final Design Setting (US),2,F,T,42301,15.0,33.0,43502,2018-11-27 18:47:28.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
3,5478,1,Final Design Setting (US),Final Design Setting (US),2,F,T,42301,15.0,33.0,43502,2018-09-25 17:49:03.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
4,13095,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2020-05-18 17:07:42.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
5,18262,1,Preliminary Design Setting (US),Preliminary Design Setting (US),26,T,F,42301,15.0,33.0,43502,2024-08-15 13:11:39.103,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,70A01C5A-21D9-4C12-9A23-1FD3431D583B


### BRIDGEWARE

#### ABW_BRIDGE_SUB_LRFDDSN_SETTING

In [20]:
get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFDDSN_SETTING")

,bridge_id,lrfd_design_setting_id,name,descr,bridge_lrfd_factor_id,preliminary_design_setting_ind,final_design_setting_ind,dynamic_load_allow_method_type,dla_simple_fatigue_frac_impact,dla_simple_other_ls_impact,vehicle_summary_display_type,last_change_timestamp,limit_num_dsgn_load_combo_ind,cw_num_loadcombo_axial_bending,ftg_num_loadcombo_brg_capacity,lrfd_analysis_module_guid,lrfd_spec_edition_guid
0,3332,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2016-01-29 16:55:09.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
1,3717,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2016-10-28 16:40:47.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
2,5564,1,Final Design Setting (US),Final Design Setting (US),2,F,T,42301,15.0,33.0,43502,2018-11-27 18:47:28.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
3,5478,1,Final Design Setting (US),Final Design Setting (US),2,F,T,42301,15.0,33.0,43502,2018-09-25 17:49:03.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
4,13095,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2020-05-18 17:07:42.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
5,18262,1,Preliminary Design Setting (US),Preliminary Design Setting (US),26,T,F,42301,15.0,33.0,43502,2024-08-15 13:11:39.103,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,70A01C5A-21D9-4C12-9A23-1FD3431D583B


#### ABW_MA_STRUSS_ELEM_LOSS_RANGE

In [21]:
get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFDDSN_SETTING")

,bridge_id,lrfd_design_setting_id,name,descr,bridge_lrfd_factor_id,preliminary_design_setting_ind,final_design_setting_ind,dynamic_load_allow_method_type,dla_simple_fatigue_frac_impact,dla_simple_other_ls_impact,vehicle_summary_display_type,last_change_timestamp,limit_num_dsgn_load_combo_ind,cw_num_loadcombo_axial_bending,ftg_num_loadcombo_brg_capacity,lrfd_analysis_module_guid,lrfd_spec_edition_guid
0,3332,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2016-01-29 16:55:09.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
1,3717,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2016-10-28 16:40:47.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
2,5564,1,Final Design Setting (US),Final Design Setting (US),2,F,T,42301,15.0,33.0,43502,2018-11-27 18:47:28.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
3,5478,1,Final Design Setting (US),Final Design Setting (US),2,F,T,42301,15.0,33.0,43502,2018-09-25 17:49:03.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
4,13095,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2020-05-18 17:07:42.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
5,18262,1,Preliminary Design Setting (US),Preliminary Design Setting (US),26,T,F,42301,15.0,33.0,43502,2024-08-15 13:11:39.103,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,70A01C5A-21D9-4C12-9A23-1FD3431D583B


#### ABW_MCB_SEG_DECK

In [22]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_DECK")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,segment_deck_id,super_ref_line_type,super_ref_line_ref_type,super_ref_line_ref_web_id
0,3958,1,1,1,1,63803,63902,None
1,4275,1,1,1,1,63803,63902,None
2,4275,2,1,1,1,63803,63902,None
3,5236,1,1,1,1,63803,63902,None
4,3496,1,1,1,1,63801,63902,None


#### ABW_MATL_CONC

In [23]:
get_table("BRIDGEWARE", "ABW_MATL_CONC")

,bridge_id,conc_id,name,descr,si_or_us_type,comp_strength_28,initial_comp_strength,thermal_exp_coeff,density_dl,density_e,...,shear_factor,std_initial_mod_of_elast,last_change_timestamp,shrinkage_coef,lrfd_mod_of_elast,lrfd_initial_mod_of_elast,splitting_tensile_strength,lrfd_max_aggregate_size,std_mod_of_rupture,lrfd_mod_of_rupture
0,752,1,Class S OH 1951-1980,Class S Cement Concrete,10401,27.579028,NaN,0.000011,2402.809922,2322.716258,...,1.0,NaN,2011-03-14 18:18:48.000000,None,25125.523576,NaN,None,None,NaN,3.308500
1,776,1,Class C (US),Class C cement concrete,10401,27.579032,NaN,0.000011,2402.809922,2322.716258,...,1.0,NaN,2010-10-06 13:19:31.000000,None,25125.523576,NaN,None,None,NaN,3.308500
2,777,1,Class C (US),Class C cement concrete,10401,27.579032,NaN,0.000011,2402.809922,2322.716258,...,1.0,NaN,2010-10-06 13:24:15.000000,None,25125.523576,NaN,None,None,NaN,3.308500
3,781,1,f'c 4000,Based on year of construction,10401,27.579028,24.131650,0.000011,2402.809922,2322.716258,...,1.0,NaN,2010-10-12 18:51:58.000000,None,25125.511010,NaN,None,None,NaN,NaN
4,789,1,f'c 3000,Based on year of construction,10401,20.684271,17.236892,0.000011,2402.809922,2322.716258,...,1.0,NaN,2019-01-08 14:49:16.000000,None,21759.330818,NaN,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16183,18507,1,Class S OH 1951-1980,Class S Cement Concrete,10401,27.579028,NaN,0.000011,2402.809922,2322.716258,...,1.0,0.000000,2024-09-19 13:56:13.996000,None,27486.282900,0.000000,None,None,NaN,3.309483
16184,18508,1,Class S OH 1931-1950,Class S Cement Concrete,10401,20.684271,NaN,0.000011,2402.809922,2322.716258,...,1.0,0.000000,2024-07-02 16:46:33.742000,None,25125.523578,0.000000,None,None,NaN,3.308500
16185,18524,1,PS 7.0 ksi 5.5 Release,release 5500psi,10401,48.263299,37.921163,0.000011,2402.809922,2402.809922,...,1.0,30999.246508,2025-05-30 15:22:19.582129,None,34970.207504,30999.241157,None,None,NaN,4.378035
16186,18524,2,Deck Concrete 4.5 ksi,Deck Concrete,10401,31.026407,NaN,0.000011,2402.809922,2322.716258,...,1.0,0.000000,2025-05-30 15:22:19.581337,None,28575.664911,0.000000,None,None,NaN,3.510237


#### ABW_MATL_PS_BAR

In [24]:
get_table("BRIDGEWARE", "ABW_MATL_PS_BAR")

,bridge_id,ps_bar_id,name,descr,bar_diameter,bar_area,bar_type,ultimate_tensile_strength,yield_strength,modulus_of_elasticity,unit_load_per_length,epoxy_coated_ind
0,3430,1,Tie Rod,"1"" Type I Plain Bar",25.4,503.2248,55201,744.633756,689.475700,206842.71,3.973465,F
1,4383,1,"1"" Type I","1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
2,13098,1,"1"" Type I","1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
3,3914,1,None,None,NaN,NaN,55201,NaN,NaN,NaN,NaN,F
4,3583,1,None,None,NaN,NaN,55201,NaN,NaN,NaN,NaN,F
5,5484,1,Tie Rod,"1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
6,6351,1,"1"" Type I","1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
7,12916,1,Tie Rod,"1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
8,16051,1,"1"" Type I","1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081517,206842.71,3.973465,F


#### ABW_MATL_PS_STRAND

In [25]:
get_table("BRIDGEWARE", "ABW_MATL_PS_STRAND")

,bridge_id,matl_ps_strand_id,name,descr,si_or_us_type,strand_type,strand_diameter,strand_area,weight_per_length,ultimate_tensile_strength,yield_strength,mod_of_elast,transfer_length_lrfd,transfer_length_std,epoxy_coated_ind,last_change_timestamp
0,20,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.7,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2003-01-01 12:00:00.000000
1,21,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.7,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2003-01-01 12:00:00.000000
2,22,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.7,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2003-01-01 12:00:00.000000
3,23,1,"1/2"" (7W-270)","1/2""/Seven Wire/fpu = 270",10401,34301,12.7,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2003-01-01 12:00:00.000000
4,19464,1,"1/2"" (7W-270) SR","Stress relieved 1/2""/Seven Wire/fpu = 270",10401,34302,12.7,98.70948,0.773858,1861.58439,1582.346731,196500.5745,762.0,635.0,F,2024-09-13 20:54:07.337984
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2996,20421,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.7,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2025-04-22 14:05:11.857000
2997,20424,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.7,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2019-05-08 21:18:25.000000
2998,20427,1,"1/2"" (7W-270) SR","Stress relieved 1/2""/Seven Wire/fpu = 270",10401,34302,12.7,98.70948,0.773858,1861.58439,1582.346731,196500.5745,762.0,635.0,F,2024-10-04 00:15:56.696000
2999,20427,2,"1/2"" (7W-270K) LR 0.167 sq inch","LR 1/2""/Seven Wire/fpu = 270 special",10401,34301,12.7,107.74172,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2024-10-04 00:10:11.477000


#### ABW_FL_FLOORBEAM_TRAVELWAY

In [26]:
get_table("BRIDGEWARE", "ABW_FL_FLOORBEAM_TRAVELWAY")

,bridge_id,struct_def_id,super_struct_mbr_id,travelway_id,dist_left_girder_travelway,travelway_width
0,8322,4,1,1,5.449885,10.05840
1,8322,4,1,2,17.489485,10.05840
2,11883,2,4,1,-0.533400,9.55039
3,13078,2,2,1,0.914400,0.99060


#### ABW_FLINE_MBR

In [27]:
get_table("BRIDGEWARE", "ABW_FLINE_MBR")

,bridge_id,struct_def_id,super_struct_mbr_id,fline_location_type,length,girder_spacing,deck_crack_control_param_z


#### ABW_MCB_TEND_PROF_DEF_DETAIL

In [28]:
get_table("BRIDGEWARE", "ABW_MCB_TEND_PROF_DEF_DETAIL")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,tendon_profile_id,detail_id,profile_type,inflect_point_left_percent,inflect_point_low_percent,inflect_point_right_percent,...,vert_offset_left_end_dist,vert_offset_le_meas_from_type,vert_offset_left_dist,vert_offset_l_meas_from_type,vert_offset_low_dist,vert_offset_lo_meas_from_type,vert_offset_right_dist,vert_offset_r_meas_from_type,vert_offset_right_end_dist,vert_offset_re_meas_from_type
0,5236,1,1,1,1,11,57101,NaN,NaN,NaN,...,215.9,57201,None,57201,101.6,57202,None,57201,215.9,57201
1,3958,1,1,1,1,4,57102,NaN,NaN,NaN,...,444.5,57201,None,57201,812.8,57202,None,57201,444.5,57201
2,4275,1,1,1,1,7,57103,NaN,40.0,10.0,...,855.0,57201,None,57201,380.0,57202,None,57201,584.0,57201
3,4275,1,1,1,2,1,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,380.0,57202,None,57201,584.0,57201
4,4275,1,1,1,2,2,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,380.0,57202,None,57201,584.0,57201
5,4275,1,1,1,2,3,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,380.0,57202,None,57201,584.0,57201
6,4275,1,1,1,2,4,57104,10.0,60.0,NaN,...,584.0,57201,None,57201,380.0,57202,None,57201,855.0,57201
7,4275,2,1,1,1,7,57103,NaN,40.0,10.0,...,855.0,57201,None,57201,425.0,57202,None,57201,584.0,57201
8,4275,2,1,1,1,8,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,425.0,57202,None,57201,584.0,57201
9,4275,2,1,1,1,9,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,425.0,57202,None,57201,584.0,57201


#### ABW_STL_ANGLE

In [29]:
get_table("BRIDGEWARE", "ABW_STL_ANGLE")

,bridge_id,stl_shape_id,angle_size_1,angle_size_2,angle_thick,nominal_weight_or_mass,k,area,ixx,yxx,iyy,xyy,rzz,zztantheta
0,18682,2,152.4,152.4,9.5250,22.174016,NaN,2825.8008,6.409964e+06,41.148,6.409964e+06,41.1480,30.2260,NaN
1,18994,3,101.6,101.6,7.9502,12.203150,NaN,1548.3840,1.527569e+06,28.194,1.527569e+06,28.1940,19.8374,NaN
2,116,1,76.2,76.2,7.9502,9.077953,NaN,1148.3848,6.243471e+05,21.844,6.243471e+05,21.8440,14.8082,NaN
3,16402,3,76.2,76.2,7.9375,9.077953,NaN,1148.3848,6.285095e+05,21.971,6.285095e+05,21.9710,14.9606,NaN
4,18682,1,152.4,101.6,7.9502,15.328347,NaN,1954.8348,4.745038e+06,48.260,1.719036e+06,23.0632,22.1996,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8170,19148,4,101.6,101.6,7.9502,12.203150,NaN,1548.3840,1.527569e+06,28.194,1.527569e+06,28.1940,19.8374,NaN
8171,19149,1,127.0,127.0,9.5250,18.304725,NaN,2354.8340,3.646187e+06,34.798,3.646187e+06,34.7980,25.0444,NaN
8172,19149,2,101.6,101.6,9.5250,14.584252,NaN,1845.1576,1.798120e+06,28.702,1.798120e+06,28.7020,19.7866,NaN
8173,19162,3,76.2,76.2,7.9502,9.077953,NaN,1148.3848,6.243471e+05,21.844,6.243471e+05,21.8440,14.8082,NaN


#### ABW_STL_ANGLE_CONN

In [30]:
get_table("BRIDGEWARE", "ABW_STL_ANGLE_CONN")

,bridge_id,struct_def_id,stl_component_id,stl_shape_id,length,y_offset,position_ind,leg_ind


#### ABW_STL_BEAM_ASSEMBLY

In [31]:
get_table("BRIDGEWARE", "ABW_STL_BEAM_ASSEMBLY")

,bridge_id,struct_def_id,spng_mbr_def_id,stl_assembly_id,stl_component_id,dist
0,17084,2,1,6,24,18.739104
1,17084,2,2,1,2,0.000000
2,17084,2,2,2,11,18.281599
3,17084,2,2,6,29,18.281599
4,17084,2,3,1,3,0.000000
...,...,...,...,...,...,...
230902,16597,1,4,5,29,29.260800
230903,16597,1,4,6,30,7.315200
230904,16597,1,4,7,31,29.260800
230905,16597,1,5,1,32,0.000000


#### ABW_WEB_DISTRIB_LOAD

In [32]:
get_table("BRIDGEWARE", "ABW_WEB_DISTRIB_LOAD")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,web_id,web_distr_load_id,direction_type,load_case_id,dist,length,load_start,load_end,local_axis_ind,ws_field_measured_ind,flexure_percent_of_load,shear_percent_of_load,description,reference_point_type


#### BRIDGE

In [33]:
get_table("BRIDGEWARE", "BRIDGE")

,brkey,bridge_id,struct_num,strucname,featint,fhwa_regn,district,county,facility,location,...,userkey14,userkey15,btrigger,traceflag,createdatetime,createuserkey,modtime,userkey,docrefkey,notes


#### COPTIONS

In [34]:
get_table("BRIDGEWARE", "COPTIONS")

,optionname,optionval,defaultval,helpid,description
0,MULTISERVER,C:\,C:\,1580,This specifies the directory path to multimedi...


#### MULTIMEDIA

In [35]:
get_table("BRIDGEWARE", "MULTIMEDIA")

,docrefkey,sequence,context,fileloc,fileref,filetype,agencytype,status,reportflag,userkey1,userkey2,userkey3,userkey4,userkey5,createdatetime,createuserkey,modtime,userkey,notes
0,6701213::15A2EBD9-5B6A-4298-AA40-E393E2E520A1,1,Virtis/Opis,X:\POR-43-1375\,DSC_1690.JPG,JPG,D,None,N,None,None,None,None,None,2006-06-21 09:16:00,VOU,None,None,None


#### ABW_TRUSS_LINE_MBR

In [36]:
get_table("BRIDGEWARE", "ABW_TRUSS_LINE_MBR")

,bridge_id,struct_def_id,super_struct_mbr_id,truss_location_type,length,truss_spacing,settlement_load_case_id,deck_crack_control_param_z,deck_exposure_factor
0,12853,6,1,31801,None,None,None,None,None


#### ABW_TRUSS_LINE_SUPPORT

In [37]:
get_table("BRIDGEWARE", "ABW_TRUSS_LINE_SUPPORT")

,bridge_id,struct_def_id,super_struct_mbr_id,truss_line_support_id,dist,x_translation_type,y_translation_type,z_rotation_type,x_translation_spring_constant,y_translation_spring_constant,z_rotation_spring_constant,x_translation_settlement,y_translation_settlement,z_rotation_settlement,override_zrot_spring_const_ind


#### ABW_TRUSSDEF_ELEMENT_CONC_LOAD

In [38]:
get_table("BRIDGEWARE", "ABW_TRUSSDEF_ELEMENT_CONC_LOAD")

,bridge_id,struct_def_id,spng_mbr_def_id,truss_def_element_id,element_load_id,dist,force_x,force_y,force_z,moment_z


#### ABW_RESULTS_CONC_LS_SUMMARY

In [39]:
get_table("BRIDGEWARE", "ABW_RESULTS_CONC_LS_SUMMARY")

,spng_mbr_alt_event_id,point_id,conc_ls_summary_id,flex_type,load_type_id,vehicle_type,summary_type,event_lrfd_factor_id,event_ls_id,stage_id,shear,moment,horz_shear


#### ABW_RESULTS_CONC_XSEC_PROP

In [40]:
get_table("BRIDGEWARE", "ABW_RESULTS_CONC_XSEC_PROP")

,spng_mbr_alt_event_id,point_id,conc_xsec_prop_id,moment_sign_type,depth,gross_xsection_area,gross_moment_of_inertia,cracked_neutral_axis_loc,cracked_moment_of_inertia,section_modulus_conc_top,section_modulus_conc_bot,section_modulus_reinf_top,section_modulus_reinf_bot


#### ABW_LIB_MATL_TIMBER_SAWN_ITEM

In [41]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TIMBER_SAWN_ITEM")

,timber_matl_id,sawn_timber_matl_item_id,timber_commercial_grade_id,timber_size_classification_id,timber_grading_rule_agency_id,unit_asd_bending_stress,unit_asd_tens_stress_parallel,unit_asd_shear_stress_parallel,unit_asd_comp_stress_perp,unit_asd_comp_stress_parallel,mod_of_elast,density,unit_lrfd_bending_stress,unit_lrfd_tens_stress_parallel,unit_lrfd_shear_stres_parallel,unit_lrfd_comp_stress_perp,unit_lrfd_comp_stress_parallel,lrfd_mod_of_elast,asd_notes,lrfd_notes
0,6,2,4,4,1,6.894757,3.964485,0.758423,6.101860,6.377650,9652.6598,800.93664,6.894757,3.964485,1.516847,6.101860,6.377650,9652.660,None,None
1,6,3,7,4,1,6.722388,3.964485,0.758423,6.101860,4.998699,8963.1841,800.93664,6.722388,3.964485,1.516847,6.101860,4.998699,8963.185,None,None
2,6,4,8,8,1,11.031611,6.550019,0.723949,6.101860,6.550019,8963.1841,800.93664,11.031610,6.550019,1.413425,6.101860,6.550019,8963.185,None,None
3,6,5,4,8,1,9.307922,4.653961,0.723949,6.101860,5.515806,8963.1841,800.93664,9.307922,4.653961,1.413425,6.101860,5.515806,8963.185,None,None
4,6,6,7,8,1,6.032912,2.930272,0.723949,6.101860,3.447379,6894.7570,800.93664,6.032912,2.930272,1.413425,6.101860,3.447378,6894.757,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,12,20,4,9,6,5.515806,3.792116,0.448159,2.309744,4.309223,8273.7084,800.93664,5.515806,3.792116,0.861845,2.309744,4.309223,8273.708,None,None
148,12,21,7,9,6,2.413165,1.551320,0.448159,2.309744,1.551320,6894.7570,800.93664,3.275010,2.240796,0.861845,2.309744,2.930272,6894.757,None,None
149,13,1,8,4,1,6.894757,3.964485,0.517107,2.895798,6.205281,10342.1355,800.93664,6.894757,3.964485,0.999740,2.895798,6.205281,10342.140,None,None
150,13,2,4,4,1,4.998699,2.930272,0.517107,2.895798,4.998699,9652.6598,800.93664,4.998699,2.930272,0.999740,2.895798,4.998699,9652.660,None,None


#### ABW_SYS_LRFD_LS

In [42]:
get_table("BRIDGEWARE", "ABW_SYS_LRFD_LS")

,sys_lrfd_ls_id,name,descr
0,1,STRENGTH-I,STRENGTH-I
1,2,STRENGTH-II,STRENGTH-II
2,3,STRENGTH-III,STRENGTH-III
3,4,STRENGTH-IV,STRENGTH-IV
4,5,STRENGTH-V,STRENGTH-V
5,6,SERVICE-I,SERVICE-I
6,7,SERVICE-II,SERVICE-II
7,8,SERVICE-III,SERVICE-III
8,9,SERVICE-IV,SERVICE-IV
9,10,FATIGUE-I,FATIGUE-I


#### ABW_SUBSDEF_MODEL_SETTING

In [43]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_MODEL_SETTING")

,bridge_id,struct_def_id,model_setting_id,default_ind,num_equal_element_cap_span,num_equal_element_col_seg,footing_element_type,use_cracked_moi_ind,cracked_moi_top_col_wall_pct,cracked_moi_bot_col_wall_pct,use_long_stl_reinf_ei_calc_ind,prov_moment_rel_col_cap_ind
0,18675,5,1,None,NaN,NaN,43901,None,None,None,None,None
1,4904,3,1,None,NaN,NaN,43901,None,None,None,None,None
2,7533,2,1,None,NaN,NaN,43901,None,None,None,None,None
3,7534,3,1,None,NaN,NaN,43901,None,None,None,None,None
4,5564,3,1,None,1.0,1.0,43901,F,None,None,None,F
5,3717,2,1,None,1.0,1.0,43901,F,None,None,None,F
6,5564,4,1,None,NaN,NaN,43901,None,None,None,None,None
7,5564,5,1,None,NaN,NaN,43901,None,None,None,None,None
8,5564,6,1,None,NaN,NaN,43901,None,None,None,None,None
9,6370,2,1,None,NaN,NaN,43901,None,None,None,None,None


#### ABW_SUPER_BR_FORCE_SUB

In [44]:
get_table("BRIDGEWARE", "ABW_SUPER_BR_FORCE_SUB")

,bridge_id,sub_struct_def_id,lane_set_id,num_lanes,braking_force,braking_force_ul


#### ABW_CONC_CURB_SIDEWALK

In [45]:
get_table("BRIDGEWARE", "ABW_CONC_CURB_SIDEWALK")

,bridge_id,struct_def_id,curb_id,load_case_id,use_type,offset_ref_type,conc_id,bridge_ref_line_id,straight_ind,offset_start,offset_end,bot_width_start,top_width_start,bot_width_end,top_width_end,avg_thick,conc_density,measured_to_left_face_ind,pedestrian_load
0,13690,1,2,1.0,20102,20202,1,None,None,-0.050810,-0.050810,2006.6,None,None,None,234.95000,None,F,NaN
1,13763,1,1,1.0,20102,20201,1,None,None,0.000000,0.000000,990.6,None,None,None,320.67500,None,T,NaN
2,13763,1,2,1.0,20102,20202,1,None,None,0.000000,0.000000,990.6,None,None,None,320.67500,None,F,NaN
3,16749,1,1,2.0,20101,20202,1,None,None,0.050800,0.050800,3962.4,None,None,None,203.20000,None,F,NaN
4,16928,4,1,2.0,20101,20201,4,None,None,0.000000,0.000000,2133.6,None,None,None,203.20000,None,T,3591.019792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1174,18584,2,2,2.0,20101,20202,1,None,None,0.304800,0.304800,1524.0,None,None,None,233.36250,None,F,NaN
1175,19381,1,1,2.0,20101,20202,1,None,None,0.000000,0.000000,1574.8,None,None,None,160.33750,None,F,NaN
1176,16731,1,2,1.0,20101,20202,3,None,None,0.304800,1.828800,1524.0,None,None,None,203.20000,None,F,3.591020
1177,16731,1,1,1.0,20101,20201,3,None,None,0.304800,1.828800,1524.0,None,None,None,203.20000,None,T,3.591020


#### ABW_CONC_RAILING

In [46]:
get_table("BRIDGEWARE", "ABW_CONC_RAILING")

,bridge_id,conc_railing_id,name,descr,si_or_us_type,barrier_type,conc_id,x1,x2,x3,x4,x5,y1,y2,y3,y4,dist_to_cg,additional_load,conc_density,last_change_timestamp
0,1200,1,Barrier,Barrier,10401,26804,None,457.2,NaN,NaN,NaN,NaN,914.4,NaN,NaN,NaN,228.6,9.340107,NaN,2012-02-01 21:16:58.000
1,1201,1,Pipe Rail,Hand Rail,10401,26804,None,50.8,NaN,NaN,NaN,NaN,609.6,NaN,NaN,NaN,25.4,0.145939,NaN,2012-02-03 16:55:59.000
2,1274,1,"OH-Deflector Parapet Type 42""",based upon plan detail,10401,26801,None,228.6,50.8,177.8,NaN,NaN,0.0,736.6,254.00,76.2,NaN,NaN,2402.809922,2017-12-01 14:34:50.000
3,1269,1,LEFT PARAPET,None,10401,26801,None,228.6,76.2,228.6,NaN,NaN,0.0,431.8,330.20,114.3,0.0,0.656726,2402.809922,2012-04-06 12:02:47.000
4,1269,2,SIDEWALK BARRIER,None,10401,26801,None,304.8,0.0,0.0,NaN,NaN,0.0,685.8,400.05,0.0,0.0,0.612945,2402.809922,2012-04-06 12:02:47.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8590,20433,1,BR-1-79 (1988 Rehab),User defined BR-1\r\n,10401,26801,None,228.6,50.8,177.8,NaN,NaN,0.0,482.6,285.75,76.2,NaN,NaN,2402.809922,2025-06-20 16:12:03.840
8591,20434,1,"OH-Deflector Parapet Type 42""",Section B-B Std.Dwg.: BR1 (7-19-02),10401,26801,None,203.2,76.2,177.8,NaN,NaN,0.0,736.6,254.00,76.2,0.0,0.437817,2402.809922,2025-06-17 21:01:45.630
8592,20435,1,"OH-Deflector Parapet Type 42""",Section B-B Std.Dwg.: BR1 (7-19-02),10401,26801,None,203.2,76.2,177.8,NaN,NaN,0.0,736.6,254.00,76.2,0.0,0.437817,2402.809922,2025-06-17 21:01:45.630
8593,20437,1,"OH-Deflector Parapet Type 36""",Section B-B Std.Dwg.:BR1 (7-19-02),10401,26801,None,215.0,65.0,178.0,NaN,NaN,0.0,585.0,254.00,76.0,0.0,0.401000,2402.809922,2025-06-23 18:08:11.580


#### ABW_CONC_RAILING_LOC

In [47]:
get_table("BRIDGEWARE", "ABW_CONC_RAILING_LOC")

,bridge_id,struct_def_id,conc_railing_location_id,load_case_id,offset_ref_type,stl_railing_id,conc_railing_id,bridge_ref_line_id,conc_railing_offset_start,conc_railing_offset_end,steel_railing_offset,straight_ind,conc_railing_face_left_ind,stl_railing_face_left_ind,measured_to_front_face_ind
0,3158,1,1,1.0,20201,None,1,None,0.0,0.0,None,None,F,None,F
1,3158,1,2,1.0,20202,None,1,None,0.0,0.0,None,None,T,None,F
2,3158,2,1,1.0,20201,None,1,None,0.0,0.0,None,None,F,None,F
3,3158,2,2,1.0,20202,None,1,None,0.0,0.0,None,None,T,None,F
4,3159,1,1,1.0,20201,None,1,None,0.0,0.0,None,None,F,None,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16812,18065,1,2,4.0,20202,None,1,None,0.0,0.0,None,None,T,None,F
16813,18066,1,1,4.0,20201,None,1,None,0.0,0.0,None,None,F,None,F
16814,18066,1,2,4.0,20202,None,1,None,0.0,0.0,None,None,T,None,F
16815,18072,1,1,4.0,20201,None,1,None,0.0,0.0,None,None,F,None,F


#### ABW_BRIDGE_DIAPHRAGM_DEF_CON

In [48]:
get_table("BRIDGEWARE", "ABW_BRIDGE_DIAPHRAGM_DEF_CON")

,bridge_id,diaphragm_def_id,connection_id,name,support_type,y_dist,meas_from_type
0,2505,2,4,B,52401,NaN,52501.0
1,2505,3,1,A,52401,50.8,52501.0
2,2505,3,2,B,52401,50.8,52501.0
3,2505,3,3,C,52401,50.8,52502.0
4,2505,3,4,D,52401,50.8,52502.0
...,...,...,...,...,...,...,...
35995,20430,1,126,A,52401,355.6,52501.0
35996,20430,1,127,B,52401,355.6,52501.0
35997,20430,1,128,C,52401,254.0,52502.0
35998,20430,1,129,D,52401,254.0,52502.0


#### ABW_SUBSDEF_FOUND_FK

In [49]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_FOUND_FK")

,bridge_id,struct_def_id,subsdef_found_id,as_built_found_alt_id,current_found_alt_id
0,7533,2,1,NaN,NaN
1,7534,3,1,NaN,NaN
2,18675,5,1,1.0,1.0
3,4904,3,1,1.0,1.0
4,5564,3,1,1.0,1.0
5,3717,2,1,1.0,1.0
6,5564,4,1,1.0,1.0
7,5564,5,1,1.0,1.0
8,5564,6,1,1.0,1.0
9,6370,2,1,NaN,NaN


#### ABW_STL_BUILTUP_IBEAM_DEF

In [50]:
get_table("BRIDGEWARE", "ABW_STL_BUILTUP_IBEAM_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id
0,8528,2,1
1,8528,2,2
2,8528,2,3
3,8528,2,4
4,8528,2,5
...,...,...,...
519,7870,3,7
520,7960,1,1
521,7960,1,2
522,8145,2,1


#### ABW_STL_LONG_STIFF

In [51]:
get_table("BRIDGEWARE", "ABW_STL_LONG_STIFF")

,bridge_id,struct_def_id,stl_component_id,stl_angle_shape_id,web_weld_id,length,width,vert_dist,vert_dist_by_web_fraction,dist_ref_type,thick,short_leg_attachment_ind,stl_long_stiff_type,last_change_timestamp,long_stiff_side_type
0,12769,2,252,NaN,1.0,33680.400000,152.4,609.6,NaN,27002,15.875,None,22201,2019-11-06 18:37:22.000,NaN
1,12769,2,253,NaN,1.0,38252.400000,152.4,609.6,NaN,27001,15.875,None,22201,2019-11-06 18:37:22.000,NaN
2,12769,2,254,NaN,1.0,33680.400000,152.4,609.6,NaN,27002,15.875,None,22201,2019-11-06 18:37:22.000,NaN
3,12769,2,255,NaN,1.0,33171.384000,152.4,609.6,NaN,27001,15.875,None,22201,2019-11-06 18:37:22.000,NaN
4,4760,7,57,NaN,NaN,20624.800102,152.4,NaN,20.0,27004,9.525,None,22201,2017-12-11 15:34:32.000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3068,20329,1,918,NaN,NaN,16857.085920,114.3,NaN,20.0,27004,9.525,None,22201,2024-12-17 16:23:59.110,57301.0
3069,20329,1,919,NaN,NaN,17512.131600,114.3,NaN,20.0,27004,9.525,None,22201,2024-12-17 16:23:59.110,57301.0
3070,20329,1,920,NaN,NaN,17512.131600,114.3,NaN,20.0,27004,9.525,None,22201,2024-12-17 16:23:59.110,57301.0
3071,20329,1,921,NaN,NaN,17512.131600,114.3,NaN,20.0,27004,9.525,None,22201,2024-12-17 16:23:59.110,57301.0


#### ABW_EVENT_VEHICLE_TEMPLATE

In [52]:
get_table("BRIDGEWARE", "ABW_EVENT_VEHICLE_TEMPLATE")

,template_id,template_vehicle_id,vehicle_id,inventory_ind,operating_ind,design_ind,permit_ind,fatigue_ind,scale_factor,single_lane_ind,...,lrfr_operating_ind,legal_pair_ind,lrfr_fatigue_ind,legal_live_load_override_ind,legal_ll_override_factor,lrfr_legal_haul_ind,asr_lfr_permit_inventory_ind,asr_lfr_permit_operating_ind,asr_lfr_legal_inventory_ind,asr_lfr_legal_operating_ind
0,95,12,44,F,T,F,F,F,1.0,F,...,F,F,F,F,0.0,F,F,F,F,F
1,95,13,7,F,T,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,F
2,95,14,9,F,T,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,F
3,95,15,8,F,T,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,F
4,96,1,994,F,T,F,F,F,1.0,F,...,F,F,F,F,0.0,F,F,F,F,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
418,95,11,43,F,T,F,F,F,1.0,F,...,F,F,F,F,0.0,F,F,F,F,F
419,89,106,3,T,T,F,F,F,1.0,F,...,F,F,F,F,0.0,F,F,F,F,F
420,89,107,15,F,T,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,F
421,104,1,2,F,F,T,F,F,1.0,None,...,F,None,F,None,NaN,F,F,F,F,F


#### ABW_LEG_ANAL_PT_REINF_CONC

In [53]:
get_table("BRIDGEWARE", "ABW_LEG_ANAL_PT_REINF_CONC")

,bridge_id,struct_def_id,super_struct_mbr_id,leg_id,frame_mbr_alt_leg_id,anal_pt_id,lrfd_shear_comp_method_type,override_shear_reinf_ind,percent_shear,shear_dist,shear_beta,shear_theta,shear_sx,stirrup_matl_stl_reinf_id,stirrup_rebar_id,stirrup_num_legs,stirrup_area,lfr_ignore_shear_ind


#### ABW_RESULTS_CRIT_LOAD_LRFD

In [54]:
get_table("BRIDGEWARE", "ABW_RESULTS_CRIT_LOAD_LRFD")

,spng_mbr_alt_event_id,point_id,crit_load_lrfd_id,stage_id,event_lrfd_factor_id,event_ls_id,moment_max,moment_max_cvehicle_id,moment_max_crit_dl_type,moment_min,...,reaction_max_crit_dl_type,reaction_min,reaction_min_cvehicle_id,reaction_min_crit_dl_type,y_defl_max,y_defl_max_cvehicle_id,y_defl_max_crit_dl_type,y_defl_min,y_defl_min_cvehicle_id,y_defl_min_crit_dl_type


#### ABW_RESULTS_DL_ACTION

In [55]:
get_table("BRIDGEWARE", "ABW_RESULTS_DL_ACTION")

,spng_mbr_alt_event_id,point_id,dl_action_id,dead_load_case_id,side_type,moment,axial,shear,reaction,x_defl,y_defl,top_flange_moment,bot_flange_moment,torsion
0,862,22,4,2193,58301,9.741623,0.0,-105.200300,NaN,0.0,-8.372860,0.0,0.0,NaN
1,862,23,4,2193,58301,-61.609489,0.0,-114.672080,NaN,0.0,-6.758536,0.0,0.0,NaN
2,862,24,7,2193,58302,-178.612077,0.0,-128.704060,NaN,0.0,-4.459940,0.0,0.0,NaN
3,862,24,8,2193,58303,-178.612077,0.0,-128.704060,NaN,0.0,-4.459940,0.0,0.0,NaN
4,862,25,4,2193,58301,-305.082018,0.0,-142.323982,NaN,0.0,-2.483374,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340310,1149,14,6,2956,58301,187.127317,0.0,-3.601851,NaN,0.0,-1.291407,0.0,0.0,NaN
340311,1149,15,6,2956,58301,186.531070,0.0,-3.733869,NaN,0.0,-1.286499,0.0,0.0,NaN
340312,1149,16,6,2956,58301,162.097363,0.0,-7.323108,NaN,0.0,-1.089735,0.0,0.0,NaN
340313,1149,17,11,2956,58302,146.280424,0.0,-8.906086,NaN,0.0,-0.966901,0.0,0.0,NaN


#### ABW_RESULTS_LL_ACTION

In [56]:
get_table("BRIDGEWARE", "ABW_RESULTS_LL_ACTION")

,spng_mbr_alt_event_id,point_id,ll_action_id,stage_id,ll_vehicle_id,ll_vehicle_type,moment_pos,moment_neg,axial_pos,axial_neg,...,concurrent_moment_pos,concurrent_moment_neg,concurrent_shear_pos,concurrent_shear_neg,top_flange_moment_pos,top_flange_moment_neg,bot_flange_moment_pos,bot_flange_moment_neg,torsion_pos,torsion_neg
0,854,75,3,3,2,32802,351.517908,-1380.899441,0.0,0.0,...,241.242298,351.517908,-11.257291,44.223030,0.0,0.0,0.0,0.0,None,None
1,854,76,3,3,2,32802,349.517570,-1373.041331,0.0,0.0,...,271.251192,349.517570,-11.257291,44.223030,0.0,0.0,0.0,0.0,None,None
2,854,77,3,3,2,32802,623.043067,-1284.337721,0.0,0.0,...,602.539143,561.016654,251.212485,44.223030,0.0,0.0,0.0,0.0,None,None
3,854,78,3,3,2,32802,819.880821,-1227.466170,0.0,0.0,...,801.635219,752.368391,242.180532,44.223030,0.0,0.0,0.0,0.0,None,None
4,854,79,3,3,2,32802,948.704476,-1188.441926,0.0,0.0,...,931.429974,879.619589,235.715187,44.223030,0.0,0.0,0.0,0.0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444684,1149,182,41,5,9,32802,2254.976259,-518.627849,0.0,0.0,...,1973.563138,1973.563228,-54.183619,44.034948,0.0,0.0,0.0,0.0,None,None
444685,1149,183,41,5,9,32802,2145.790084,-686.530944,0.0,0.0,...,1730.951198,1968.366702,-101.842643,-44.176474,0.0,0.0,0.0,0.0,None,None
444686,1149,184,41,5,9,32802,2129.029885,-697.565551,0.0,0.0,...,1708.794975,1957.535332,-104.926198,-44.176474,0.0,0.0,0.0,0.0,None,None
444687,1149,185,41,5,9,32802,2127.889277,-698.283458,0.0,0.0,...,1707.308907,1956.779835,-105.126913,-44.176387,0.0,0.0,0.0,0.0,None,None


#### ABW_RESULTS_LS_SUMMARY_DETAIL

In [57]:
get_table("BRIDGEWARE", "ABW_RESULTS_LS_SUMMARY_DETAIL")

,spng_mbr_alt_event_id,point_id,stl_ls_summary_id,ls_summary_detail_id,summary_type,flex_moment_resist,flex_stress_resist,shear_force,service_stress_top,service_stress_bot,bearing_force


#### ABW_RESULTS_PS_CONC_STRESS

In [58]:
get_table("BRIDGEWARE", "ABW_RESULTS_PS_CONC_STRESS")

,spng_mbr_alt_event_id,point_id,ps_conc_stress_id,flex_type,load_type_id,vehicle_type,summary_type,load_combo_type,stress_type,event_lrfd_factor_id,event_ls_id,stage_id,top_flng_stress,bot_flng_stress,deck_stress


#### ABW_RESULTS_RC_SERVICE_STRESS

In [59]:
get_table("BRIDGEWARE", "ABW_RESULTS_RC_SERVICE_STRESS")

,spng_mbr_alt_event_id,point_id,rc_service_stress_id,flex_type,load_type_id,vehicle_type,summary_type,spec_id,spec_article_id,event_lrfd_factor_id,event_ls_id,stage_id,stress,moment,check_required_ind


#### ABW_RESULTS_SPNG_MBR_ALT_XY

In [60]:
get_table("BRIDGEWARE", "ABW_RESULTS_SPNG_MBR_ALT_XY")

,spng_mbr_alt_event_id,report_id,spng_mbr_alt_xy_id,point_id,y
0,27,20,69,1738,-8.267467e+00
1,27,20,70,1739,-6.288441e+00
2,27,20,71,1741,-8.604776e-01
3,27,20,72,1742,-1.896382e-01
4,27,20,73,1743,-6.406247e-12
...,...,...,...,...,...
21807,825,1,25,25,-1.144289e+00
21808,825,1,26,26,-3.785004e-01
21809,825,1,27,27,-3.366784e-01
21810,825,1,28,28,-2.683945e-01


#### ABW_RESULTS_STL_XSEC_PROP

In [61]:
get_table("BRIDGEWARE", "ABW_RESULTS_STL_XSEC_PROP")

,spng_mbr_alt_event_id,xsection_report_id,stl_xsec_prop_id,point_id,effective_slab_width,effective_slab_thickness,depth,area,moment_of_inertia,c_bot,section_modulus_top,section_modulus_bot,section_modulus_conc,section_modulus_reinf


#### ABW_RIVET_DEF

In [62]:
get_table("BRIDGEWARE", "ABW_RIVET_DEF")

,bridge_id,rivet_def_id,name,descr,rivet_type,undriven_rivet_diameter,hole_diameter,fu,last_change_timestamp
0,10820,1,"7/8"" Rivet",None,58504,22.225,22.2250,NaN,2019-07-17 17:12:22
1,10866,1,"7/8"" Rivets","Original 7/8"" Rivets for Spans 1-5",58501,22.225,23.8125,413.685420,2019-07-17 18:48:39
2,4898,1,Rivet,None,58501,NaN,NaN,124.105626,2017-12-21 16:50:18
3,7545,1,"Standard 7/8"" Rivet-ASTM A141",Rivets for 1956 constructed steel plate girder...,58501,22.225,25.4000,358.527364,2019-04-26 12:31:09
4,12858,1,Rivets,Used for built up sections,58504,25.400,25.4000,413.685420,2019-12-23 19:36:21
5,13078,1,LAW7 Rivets,None,58504,22.225,25.4000,344.737850,2020-04-23 15:13:21
6,12768,1,"7/8"" Rivets",None,58501,22.225,23.8125,399.895906,2019-11-06 18:35:52


#### ABW_RSLT_CONCURRENT_LL_ACTION

In [63]:
get_table("BRIDGEWARE", "ABW_RSLT_CONCURRENT_LL_ACTION")

,spng_mbr_alt_event_id,point_id,concurrent_ll_action_id,ll_vehicle_type,basis_envelope_type,envelope_pos_neg_type,ll_vehicle_id,vehicle_lead_axle_loc,vehicle_direction_type,moment,shear,axial,y_defl


#### ABW_SHEAR_CONN_CHANNEL_FIELD

In [64]:
get_table("BRIDGEWARE", "ABW_SHEAR_CONN_CHANNEL_FIELD")

,bridge_id,struct_def_id,spng_mbr_def_id,shear_conn_id,stl_channel_id,channel_length,long_spacing


#### ABW_FLOORSYS_UNIT_GEOMETRY

In [65]:
get_table("BRIDGEWARE", "ABW_FLOORSYS_UNIT_GEOMETRY")

,bridge_id,struct_def_id,unit_geometry_id,unit_num,floorsys_stringer_group_def_id,mirror_stringer_group_def_type,ref_floor_beam_mbr_loc_id,unit_ref_type,dist_start_stringer_group_def,current_mbr_alt_name_template,existing_mbr_alt_name_template
0,18126,2,1,1,1.0,38301,11.0,38401,NaN,None,None
1,18126,2,2,2,1.0,38301,12.0,38402,NaN,None,None
2,18126,2,3,3,1.0,38301,13.0,38402,NaN,None,None
3,18126,2,4,4,1.0,38301,14.0,38402,NaN,None,None
4,18126,2,5,5,1.0,38301,15.0,38402,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...
566,20260,7,12,12,9.0,38301,NaN,38402,0.0,None,None
567,20260,7,13,13,10.0,38301,NaN,38402,0.0,None,None
568,20260,7,14,14,11.0,38301,NaN,38402,0.0,None,None
569,20260,7,15,15,12.0,38301,NaN,38402,0.0,None,None


#### ABW_FLRBM_STRINGER_DL_REACTION

In [66]:
get_table("BRIDGEWARE", "ABW_FLRBM_STRINGER_DL_REACTION")

,bridge_id,struct_def_id,floor_beam_mbr_id,dl_reaction_id,stringer_mbr_id,stage_id,user_defined_reaction,override_computed_ind,up_to_date_ind
0,12957,2,75,87,46,1,NaN,F,F
1,12957,2,75,88,46,2,NaN,F,F
2,12957,2,75,89,47,1,NaN,F,F
3,12957,2,75,90,47,2,NaN,F,F
4,12957,2,76,1,12,1,NaN,F,F
...,...,...,...,...,...,...,...,...,...
21985,20416,18,21,6,7,2,NaN,None,F
21986,20416,18,21,7,8,1,NaN,None,F
21987,20416,18,21,8,8,2,NaN,None,F
21988,20416,18,21,9,9,1,NaN,None,F


#### ABW_PIER_WSHFT_SHEAR_REINF

In [67]:
get_table("BRIDGEWARE", "ABW_PIER_WSHFT_SHEAR_REINF")

,bridge_id,struct_def_id,wall_shaft_id,shear_reinf_id,rebar_id,stl_reinf_id,trans_num_legs,long_num_legs,start_dist,num_spaces,bar_spacing


#### ABW_MULTIMEDIA

#### ABW_LIB_MATL_SOIL

#### ABW_LIB_MATL_ALUMINUM

#### ABW_MATL_TIMBER

#### ABW_MATL_SOIL

#### ABW_METAL_PIPE_CULVERT_DEF

#### ABW_CULVERT

#### ABW_SYS_LRFR_LOADING

#### ABW_SYS_LRFR_LOAD_GROUP

#### ABW_SYS_REPORT

#### ABW_SYS_REPORT_KEYWORD

#### ABW_SYS_SUBTYPE_TABLES

#### ABW_SUPER_FR_FORCE_SUB

#### ABW_SUPER_LL_PATTERN_SUB

#### ABW_SUPER_LOAD_SCENARIO

#### ABW_SUPER_LOAD_SCENARIO_ITEM

#### ABW_SUPER_PED_LL_REACT_SUB

#### ABW_PIER_WSHFT_FLEX_REINF

#### ABW_PIER_WSHFT_RECT_XSECT

#### ABW_PIER_WSHFT_REINF_PATT_DET

#### ABW_SUPER_STRUCT_DIAPH_MBR

#### ABW_SUPER_STRUCT_LOADING_PATH

#### ABW_PIER_COL_REINF_PATTERN_DEF

#### ABW_STL_TRUSS_XSECTION_CPLATE

#### ABW_STL_WEB_COVER_PLATE

#### ABW_STL_WEB_LOSS_RANGE

#### ABW_STL_WEB_PLATE

#### ABW_STL_XSECTION

#### ABW_ANALYSIS_EVENT

#### ABW_STL_THREADED_BAR

#### ABW_STL_TRANS_STIFF_ANGLE

#### ABW_STL_TRANS_STIFF_GNRL_RANGE

#### ABW_STL_TRANS_STIFF_LOC_RANGE

#### ABW_CHECKED_OUT_BRIDGE

#### ABW_STL_TRANS_STIFF_PLATE

#### ABW_ADV_CONC_TENDON_RANGE

#### ABW_DECK_PANEL_RATING_RESULTS

#### ABW_DEF_ANAL_OPTION_SUBS_LOAD

#### ABW_DESIGNRATIO_RESULT

#### ABW_DIAPH_LOC

#### ABW_MBR_ALT_SUPPORT

#### ABW_BMDEF_ANAL_PT_STL

#### ABW_METAL_BOX_CULVERT_DEF

#### ABW_MBR_ALT_TIMBER_DECK_RANGE

#### ABW_MBR_ALT_TRUSS_LINE_SUPPORT

#### ABW_MBR_COLUMN_STIFFNESS

#### ABW_MBR_CONCENT_LOAD

#### ABW_FRAME_COLUMN_MBR

#### ABW_FRAME_LEG_BRACE_LOC

#### ABW_FRAME_LEG_MBR_DEF

#### ABW_FRAME_MBR_LEG

#### ABW_MCB_SEG_CONC_CURB_SIDEWALK

#### ABW_DECK_PANEL_EVENT_ERRORS

#### ABW_FLNG_LAT_BEND_STRESS_SUP

#### ABW_STRGRP_LL_DISTFACT_RANGE

#### ABW_STRINGER_DL_REACT_DETAIL

#### ABW_CONC_BMDEF_SECTION_PROFILE

#### ABW_CONC_BMDEF_VOID_RANGE

#### ABW_CONC_BMDEF_WEB_PROFILE

#### ABW_PS_PRECAST_BEAM_DEF

#### ABW_CONC_BMDEF_WEB_WIDTH

#### ABW_CONC_BMDF_VOID_SEC_PAT_DET

#### ABW_STL_STRUCT_TEE

#### ABW_SUB_STRUCT_WA_BUOYANCY

#### ABW_SUB_STRUCT_WA_STREAM_PRESS

#### ABW_SUB_STRUCT_WS_LOAD

#### ABW_SUB_STRUCT_WS_SKEW

#### ABW_SUBS_STREAM_FLOW

#### ABW_SUBSALT_DLA_LIMIT_STATE

#### ABW_MCB_WEB_SHEAR_REINF_RANGE

#### ABW_MCELL_BOX_BDEF_SLAB_REINF

#### ABW_MULTI_CELL_BOX_BDEF_WEB_FK

#### ABW_RESULTS_CRIT_LOAD_LFR

#### ABW_SUPER_STRUCT_ALT

#### ABW_GUSSET_PARTIAL_SHEAR_PLANE

#### ABW_SUPER_STRUCT_DL_REACT_SUB

#### ABW_LIB_ANAL_OPTION_SUBS_LOAD

#### ABW_SUPER_STRUCT_LL_REACT_SUB

#### ABW_SUPER_STRUCT_MBR

#### ABW_LIB_BOLT

#### ABW_SUPSTRUCTDEF_BRACING_MBR

#### ABW_LIB_BOLT_DIAMETER

#### ABW_SYS_DB_MAINTENANCE

#### ABW_LIB_MATL_PS_STRAND

#### ABW_SYS_DB_MAINTENANCE_STAGE

#### ABW_LIB_MATL_WEARING_SURFACE

#### ABW_TIMBER_BEAM_SHAPE

#### ABW_LOAD_PALETTE_TEMPL_LOADING

#### ABW_FLNG_LAT_BEND_STRESS

#### ABW_SYS_BRM_SYNC_SETTING

#### ABW_WSS_WL_SUPER_MBR_WIND

#### ABW_DETAILED_TRUSS_DEF

#### ABW_WSS_WL_SUPER_WIND_EFFECT

#### ABW_SYS_METAL_BOX_COLUMN_LABEL

#### ABW_EVENT_VEHICLE

#### ABW_TIMBER_RECT_SAWN_BEAM_DEF

#### ABW_PARAMETER

#### ABW_ADV_CONC_XSECTION

#### ABW_BMDEF_DECK_REINF_RANGE

#### ABW_BMDEF_SHEAR_CONN_RANGE

#### ABW_BMDEF_SIDEWALK_LL

#### ABW_BMDEF_STL_FATIGUE_LOC

#### ABW_BOLT_DEF

#### ABW_BRIDGE_DESIGN_PARAM

#### ABW_BRIDGE_DIAPHRAGM_DEF

#### ABW_TRUSS_DEF_ELEMENT

#### ABW_TRUSS_DEF_INTERM_SUPPORT

#### ABW_TRUSS_DEF_PANEL_POINT

#### ABW_TRUSS_DEF_SPAN

#### ABW_SUPER_LL_REACT_SUB_DETAIL

#### ABW_SUPER_LOAD_CASE

#### ABW_SUPER_MBR_ENVIRONMENT_LOAD

#### ABW_SUPER_PED_LL_SUB_DETAIL

#### ABW_BOT_FLANGE_LAT_BRACING_DEF

#### ABW_ANAL_PT_REINF_CONC

#### ABW_BEAM_DEF

#### ABW_TEMP_LOAD

#### ABW_TENDON_PROFILE_ANCH_REINF

#### ABW_TENDON_PROFILE_DEF

#### ABW_TENDON_PROFILE_DEF_DETAIL

#### ABW_TIMBER_BEAM_DEF

#### ABW_LIB_MATL_STL_STRUCTURAL

#### ABW_CULVERT_STRUCT_FK

#### ABW_GENERAL_PREFERENCE

#### ABW_GENERAL_PREFERENCE_CAT

#### ABW_RC_CULVERT_DEF

#### ABW_TRUSS_DEF_SUPPORT

#### ABW_ADV_CONC_BDEF_GRD_SPACING

#### ABW_SUPER_STRUCT_DEF_FK

#### ABW_SUBSDEF_FOUND_DEF_DISTLOAD

#### ABW_SUPER_STRUCT_FK

#### ABW_SUPER_STRUCT_FRAME_DEF

#### ABW_SUPER_STRUCT_LL_DIST_SUB

#### ABW_SUPER_STRUCT_SPNG_MBR_FK

#### ABW_SUBSDEF_FOUND_DEF_UNIFLOAD

#### ABW_SUBSDEF_MODEL_SET_FIXITY

#### ABW_SUBSDEF_MODEL_SET_MBR_REL

#### ABW_SUBSDEF_MODEL_SET_POI

#### ABW_SYS_DESIGN_WATER_LEVEL

#### ABW_SYS_LRFD_LOADING

#### ABW_FLOORSYS_STRUCT_DEF

#### ABW_LIB_SUB_LRFD_DS_VEH_LOAD

#### ABW_LIB_SUB_LRFD_DSGN_SETTING

#### ABW_LIB_TIMBER_RECT_BEAM_SHAPE

#### ABW_LIB_VEHICLE

#### ABW_LIB_VEHICLE_AXLE

#### ABW_BMDEF_HAUNCH_RANGE

#### ABW_BRIDGE_BOLT_RIVET_FIELD

#### ABW_BRIDGE_DIAPH_DEF_LOC_CON

#### ABW_ADV_CONC_BDEF_FLEX_REINF

#### ABW_BMDEF_DISTRIB_LOAD

#### ABW_BRIDGE_REF_LINE_HORZ

#### ABW_BRIDGE_ENVIRONMENTAL_PARAM

#### ABW_LRFD_LS

#### ABW_LRFR_FACTOR

#### ABW_LRFR_LEGAL_LOADS

#### ABW_LRFR_LEGAL_LOADS_HAUL

#### ABW_LRFR_LF_TABLE_VALUE

#### ABW_LRFR_LOAD_FACTOR

#### ABW_LRFR_LS

#### ABW_LRFR_PERMIT_LIMITED

#### ABW_LRFR_PERMIT_ROUTINE

#### ABW_STL_CHANNEL

#### ABW_STL_COVER_PLATE

#### ABW_STL_COVER_PLATE_LOSS_RANGE

#### ABW_STL_EYE_BAR

#### ABW_STL_FATIGUE_LOC

#### ABW_STL_FLNG_ANGLE_LOSS_RANGE

#### ABW_STL_FLNG_FLAT_LOSS_RANGE

#### ABW_STL_FLNG_PLATE

#### ABW_STL_GENERAL_PLATE

#### ABW_STL_ISHAPE

#### ABW_CULVERT_DEF_RCBOX_SEG_CELL

#### ABW_CULVERT_DEF_RCBOX_SEG_WALL

#### ABW_CULVERT_VEHICLE_PATH

#### ABW_DECK_CONC_POUR

#### ABW_DECK_CORR_METAL_PANEL

#### ABW_DECK_GENERIC_PANEL

#### ABW_LIB_PS_SHAPE_STRAND_GRID

#### ABW_LIB_PS_TEE_BEAM

ABW_LIB_PS_UBEAM


ABW_LIB_REBAR


ABW_LIB_STAY_IN_PLACE_FORM


ABW_LIB_STL_ANGLE


ABW_LIB_STL_CHANNEL


ABW_LIB_STL_ISHAPE


ABW_LIB_STL_RAILING


ABW_LIB_STL_STRUCT_TEE


ABW_LIB_SUB_LRFD_DS_LS


ABW_CULVERT_DEF_RC_SEGMENT

ABW_CULVERT_DEF_RC_SEGMENT


ABW_ADV_CONC_PRECAST_XSECTION


ABW_ADV_CONC_XSECT_RANGE


ABW_AGENCY_WIND_EFFECT


ABW_BMDEF_SPAN


ABW_FSYS_STRGRP_FLOOR_BEAM


ABW_FSYS_SUPPORT


ABW_GIRDER_SYS_FRAME_CONN


ABW_GIRDER_SYS_INT_DIAPH


ABW_GIRDER_SYS_STRUCT_DEF


ABW_GLINE_MBR


ABW_BMDEF_DIAPH_LOC


ABW_GLINE_SUPPORT


ABW_LIB_VEHICLE_AXLE_WHEEL


ABW_LIB_WELDED_WIRE_REINF


ABW_LL_DISTFACTOR_RANGE


ABW_LRFD_FACTOR


ABW_LRFD_LF_TABLE_VALUE


ABW_LRFD_LOAD_FACTOR


ABW_SYS_BRM_SYNC_SETTING_VALUE


ABW_BEAM_DEF_KNEE_BRACE


ABW_BEAM_HINGE_LOC


ABW_MCELL_BDEF_TRANS_REF_LINE


ABW_SYS_DATABASE


ABW_WEB_CONCENT_LOAD


ABW_BMDEF_ANAL_PT_COMPONENT


ABW_CULVERT_COMPONENT


ABW_DECK_PANEL_COMPONENT


ABW_ABUTMENT_SUB_STRUCT


ABW_BEAM_DEF_FK


ABW_BEARING_ELASTOMERIC


ABW_BEARING_POT


ABW_BEARING_ROCKER


ABW_SUB_STRUCT_EARTH_LOAD


ABW_SUB_STRUCT_ICE_LOAD


ABW_SUB_STRUCT_PILE_DEF


ABW_SUB_STRUCT_PS_PILE_DEF


ABW_BMDEF_CONCENT_LOAD


ABW_LIB_TIMBER_SIZE_CLASS


ABW_LIB_TIMBER_SPECIES


ABW_LIB_VEHICLE_LFD


ABW_ADV_CONC_CIP_XSECTION


ABW_LOADCOMBO_SETTING_TEMPLATE


ABW_LOAD_PALETTE_TEMPLATE


ABW_MBRALT_SCHCPLT_LOSS_RANGE


ABW_STL_XSEC_CPLATE_LOSS_RANGE


ABW_STRINGER_MBR


ABW_STRUCT_DEF_STRINGER_DLCASE


ABW_SUBALT_ANALYSIS_OPTIONS


ABW_SUBALT_ANALYSIS_OPTIONS_LS


ABW_SUBALT_ANAL_OPTION_VEHICLE


ABW_SUBSALT_LOADCOMBO_SETTING


ABW_SUBSALT_LOAD_PALETTE


ABW_SUBSALT_LRFD_LOAD_COMBO


ABW_SUBSALT_WATERLEVEL_SETTING


ABW_SUBSDEF_FOUND


ABW_SUPPORT_LINE


ABW_SUPSTRUCTDEF_BRAC_MBR_LOSS


ABW_SYS_DATA_DICTIONARY


ABW_SYS_UNIT


ABW_LIB_TIMBER_COMMERCIALGRADE


ABW_LIB_TIMBER_GRAD_RULE_AGNCY


ABW_MBRALT_XSCCPLT_LOSS_RANGE


ABW_MBR_ALT_IMPORT_MESSAGE


ABW_PS_SHAPE_STRAND_GRID


ABW_MBR_CONN_DEF


ABW_MODIFICATION_EVENT


ABW_PS_TEE_BEAM


ABW_PARTIAL_SUPER_STRUCT_DEF


ABW_PERSON


ABW_PIER_CAP_BOT_COLUMN_BAY


ABW_PIER_CAP_COL


ABW_PIER_CAP_ENCASEMENT_WALL


ABW_PIER_CAP_PILE


ABW_PIER_COL_CIRC_XSEC


ABW_PIER_COL_FOUNDATION


ABW_PIER_COL_RECT_XSEC


ABW_PS_UBEAM


ABW_STL_BEAM_LOSS_RANGE


ABW_GENPREF_TEMPLATE_ITEM


ABW_CHECK_IN_OUT_EVENT


ABW_COMMENT


ABW_COMMENT_ITEM


ABW_CONC_LEG_DEF


ABW_STL_BEARING_STIFF_LOC


ABW_CREATION_EVENT


ABW_SUPER_STRUCT


ABW_MULTI_CELL_BOX_BEAM_DEF


ABW_FLOORSYS_STRINGER_MBR


ABW_LIB_CONC_RAILING


ABW_LIB_CORR_METAL_PANEL


ABW_LIB_LRFD_FACTOR


ABW_LIB_LRFD_LF_TABLE_VALUE


ABW_LIB_LRFD_LOAD_FACTOR


ABW_LIB_LRFD_LS


ABW_RESULTS_CRIT_STRESSES_ASR


ABW_LIB_CORR_MP_CULV_ITEM


ABW_LIB_CORRUGATED_MP_CULV


ABW_LIB_SPRL_RIB_MP_CULV_ITEM


ABW_RATING_TOOL_PERM_SCENARIO


ABW_RC_BOX_CULVERT_DEF


ABW_RC_BOX_CULVERT_DEF_CELL


ABW_RC_LEG_CIRC_XSECT_REINF


ABW_RC_LEG_RECT_XSECT_REINF


ABW_RC_LEG_XSECTION


ABW_RC_LEG_XSECTION_RANGE


ABW_RC_VARIABLE_XSECTION_RANGE


ABW_RC_XSECTION


ABW_RC_XSECTION_RANGE


ABW_RC_XSECTION_REINF


ABW_PS_BOX_BEAM


ABW_PS_BOX_INT_DIAPH_LOC


ABW_PS_CONC_STRESS_LIMIT


ABW_PS_CONC_STRESS_LIMIT_RANGE


ABW_PS_IBEAM


ABW_PS_PRECAST_BEAM_CONT_REBAR


ABW_PS_PRECAST_BEAM_SPAN


ABW_PS_PRECAST_MILD_STL_LAYOUT


ABW_PS_PRECAST_STRAND_LAYOUT


ABW_PS_PROPERTIES


ABW_PS_SHAPE_MILD_STEEL


ABW_CHECKED_OUT_STRUCT_DEF


ABW_BMDEF_LL_DISTFACTOR_RANGE


ABW_BF_LAT_BRACING_PROPERTIES


ABW_CHECKOUT_AUTHORIZATION


ABW_BRIDGE_SUB_LRFD_DSVEH_LOAD


ABW_BRIDGE_WATER_LEVEL


ABW_SYS_TABLE_PROPERTIES


ABW_CONC_BMDEF_EFF_SUPPORT


ABW_CONC_BMDEF_FRAME_CONN


ABW_CONC_BMDEF_REINF_PROFILE


ABW_SYS_BRDR_TABLE


ABW_BRIDGE_ALT


ABW_SUB_DR_SHAFT_QW


ABW_SUB_DR_SHAFT_REINF_BAR


ABW_SUB_DR_SHAFT_SHEAR_REINF


ABW_SUB_DR_SHAFT_TZ


ABW_SUB_DRILL_SHAFT_FOUND_DEF


ABW_SUB_PILE_DEF_CONCENT_LOAD


ABW_DECK_PANEL


ABW_SYS_TYPE


ABW_DECK_TIMBER_PANEL


ABW_GROUP_BRIDGE


ABW_GROUP_USER


ABW_GROUP_ITEM_USER


ABW_GROUP_ITEM_BRIDGE


ABW_STL_TRANS_STIFF


ABW_GUSSET_PLATE_DEF


ABW_GUSSET_PLATE_MEMBER_DEF


ABW_LANE_POSITION


ABW_LEG_ANAL_PT


ABW_LEG_CONCENT_LOAD


ABW_LEG_DISTRIB_LOAD


ABW_LEG_SHEAR_REINF_LOC


ABW_LIB_CONC_COORD_SHAPE_COORD


ABW_BRIDGE_EXCHANGE


ABW_PIER_DEF_CAP_FLEX_REINF


ABW_PIER_DEF_CAP_SHEAR_REINF


ABW_PIER_DEF_COL_CONCENT_LOAD


ABW_PIER_DEF_COL_DISTRIB_LOAD


ABW_PIER_DEF_ENCASEMENT_WALL


ABW_PIER_DEF_WALL_SHAFT


ABW_PIER_DEF_WALL_SHAFT_SEG


ABW_PIER_DEF_WALL_TOP


ABW_PIER_DEF_WALL_TOP_SEG


ABW_PIER_DEF_WEB_WALL


ABW_PIER_ENCSW_CONCENT_LOAD


ABW_PIER_ENCSW_DISTRIB_LOAD


ABW_PIER_ENCSW_RECT_XSECT


ABW_PIER_ENCSW_ROUNDNOSE_XSECT


ABW_SHEAR_CONN_FIELD

ABW_SHEAR_CONN_SPIRAL_FIELD


ABW_IMPORT_EVENT


ABW_IMPORT_MESSAGE


ABW_SHEAR_CONN_STUD_FIELD


ABW_SHEAR_CONNECTOR


ABW_INSPECTION_EVENT


ABW_SHEAR_REINF_DEF


ABW_SLAB_SYS_STRUCT_DEF


#### IABW_PIER_RC_COL_RECT_XSEC

In [69]:
# get_table("BRIDGEWARE", "IABW_PIER_RC_COL_RECT_XSEC")

#### ABW_PIER_RC_COL_ROUNDNOSE_XSEC

In [70]:
get_table("BRIDGEWARE", "ABW_PIER_RC_COL_ROUNDNOSE_XSEC")

,bridge_id,struct_def_id,pier_column_id,column_xsection_id,conc_id,width,depth,left_radius,right_radius


#### ABW_PIER_RC_COL_USR_XSEC_COORD 

In [71]:
get_table("BRIDGEWARE", "ABW_PIER_RC_COL_USR_XSEC_COORD")

,bridge_id,struct_def_id,pier_column_id,column_xsection_id,vertex_coord_id,x_coord,y_coord


#### ABW_PIER_RC_COL_WEDGENOSE_XSEC

In [72]:
get_table("BRIDGEWARE", "ABW_PIER_RC_COL_WEDGENOSE_XSEC")

,bridge_id,struct_def_id,pier_column_id,column_xsection_id,conc_id,width,depth,left_wedge_width,right_wedge_width,left_straight_depth,right_straight_depth


#### ABW_PIER_SUB_STRUCT_DEF

In [73]:
get_table("BRIDGEWARE", "ABW_PIER_SUB_STRUCT_DEF")

,bridge_id,struct_def_id,pier_sub_struct_def_type,web_wall_ind,web_wall_structural_ind,long_axis_sidesway_brace_type,long_axis_unbraced_length,long_axis_eff_length_factor,long_axis_slndr_uptodate_ind,long_axis_slndr_mom_inertia,long_axis_slndreff_anal_type,trans_axis_sidesway_brace_type,trans_axis_unbraced_length,trans_axis_eff_length_factor,trans_axis_slndr_uptodate_ind,trans_axis_slndr_mom_inertia,trans_axis_slndreff_anal_type,slender_gross_area
0,18675,5,25424,None,None,41501,NaN,0.65,None,None,41601,41502,NaN,2.0,None,None,41601,None
1,7533,2,25420,None,None,41501,NaN,0.65,None,None,41601,41502,NaN,2.0,None,None,41601,None
2,7534,3,25420,None,None,41501,NaN,0.65,None,None,41601,41502,NaN,2.0,None,None,41601,None
3,4904,3,25424,None,None,41501,NaN,0.65,None,None,41601,41502,NaN,2.0,None,None,41601,None
4,5564,3,25420,None,None,41502,4.78789,0.65,None,None,41601,41502,5.70229,2.0,None,None,41601,None
5,3717,2,25424,None,None,41501,2.43840,0.65,None,None,41601,41502,3.04800,2.0,None,None,41601,None
6,5564,4,25420,None,None,41502,4.89585,0.65,None,None,41601,41502,5.81025,2.0,None,None,41601,None
7,5564,5,25420,None,None,41502,4.98793,0.65,None,None,41601,41502,5.90233,2.0,None,None,41601,None
8,5564,6,25420,None,None,41502,4.95300,0.65,None,None,41601,41502,5.86740,2.0,None,None,41601,None
9,6370,2,25420,None,None,41501,NaN,0.65,None,None,41601,41502,NaN,2.0,None,None,41601,None


#### ABW_PIER_SUBSDEF_COLUMN

In [74]:
get_table("BRIDGEWARE", "ABW_PIER_SUBSDEF_COLUMN")

,bridge_id,struct_def_id,pier_column_id,name,top_dist,top_offset,bot_dist,bot_offset,exposure_factor,lock_reinf_ind,lock_geometry_ind,shear_reinf_type
0,7533,2,1,Column1,-8.343900,0.0,-8.343900,0.0,NaN,None,None,47201
1,7533,2,2,Column2,-4.819650,0.0,-4.819650,0.0,NaN,None,None,47201
2,7533,2,3,Column3,-1.301749,0.0,-1.301749,0.0,NaN,None,None,47201
3,7533,2,4,Column4,2.216152,0.0,2.216152,0.0,NaN,None,None,47201
4,7533,2,5,Column5,5.734053,0.0,5.734053,0.0,NaN,None,None,47201
5,7533,2,6,Column6,9.258303,0.0,9.258303,0.0,NaN,None,None,47201
6,7534,3,1,Column1,-8.823960,0.0,-8.823960,0.0,NaN,None,None,47201
7,7534,3,2,Column2,-5.299710,0.0,-5.299710,0.0,NaN,None,None,47201
8,7534,3,3,Column3,-1.781830,0.0,-1.781830,0.0,NaN,None,None,47201
9,7534,3,4,Column4,1.736049,0.0,1.736049,0.0,NaN,None,None,47201


#### ABW_PIER_SUBSDEF_COLUMN_SEG

In [75]:
get_table("BRIDGEWARE", "ABW_PIER_SUBSDEF_COLUMN_SEG")

,bridge_id,struct_def_id,pier_column_id,pier_column_seg_id,pier_subsdef_column_seg_type,bot_elevation,height,angle,xsect_variation_type,start_column_xsection_id,end_column_xsection_id
0,7533,2,1,1,39101,NaN,294.423441,None,26202,1,1
1,7533,2,2,1,39101,NaN,294.484935,None,26202,1,1
2,7533,2,3,1,39101,NaN,294.546317,None,26202,1,1
3,7533,2,4,1,39101,NaN,294.607699,None,26202,1,1
4,7533,2,5,1,39101,NaN,294.669081,None,26202,1,1
5,7533,2,6,1,39101,NaN,294.730575,None,26202,1,1
6,7534,3,1,1,39101,NaN,0.000000,None,26202,1,1
7,7534,3,2,1,39101,NaN,0.000000,None,26202,1,1
8,7534,3,3,1,39101,NaN,0.000000,None,26202,1,1
9,7534,3,4,1,39101,NaN,0.000000,None,26202,1,1


#### ABW_PIER_TOP_CAP_SEG

In [76]:
get_table("BRIDGEWARE", "ABW_PIER_TOP_CAP_SEG")

,bridge_id,struct_def_id,pier_cap_id,top_cap_seg_id,segment_loc_type,dist,length,start_elevation,end_elevation
0,18675,5,1,5,41702,-6.85800,13.716000,0.000000,0.000000
1,18675,5,1,6,41701,-6.85800,13.716000,0.000000,0.000000
2,3717,2,1,9,41702,-5.35305,10.706100,0.609600,0.609600
3,3717,2,1,10,41701,-5.35305,10.706100,0.609600,0.609600
4,7533,2,1,1,41702,-8.80110,18.516603,295.482264,295.805352
5,7533,2,1,2,41701,-8.80110,18.516603,295.482264,295.805352
6,7534,3,1,1,41702,-9.28116,18.516539,NaN,NaN
7,7534,3,1,2,41701,-9.28116,18.516539,NaN,NaN
8,5564,3,1,5,41702,-5.02920,10.058400,232.327704,232.327704
9,5564,3,1,6,41701,-5.02920,10.058400,232.327704,232.327704


#### ABW_PIER_WALL_SHAFT_CONCLOAD

In [77]:
get_table("BRIDGEWARE", "ABW_PIER_WALL_SHAFT_CONCLOAD")

,bridge_id,struct_def_id,wall_shaft_id,concent_load_id,load_case_type,load_direction_type,name,dist,offset,concent_force,concent_moment


#### ABW_PIER_WALL_SHAFT_DISTLOAD

In [78]:
get_table("BRIDGEWARE", "ABW_PIER_WALL_SHAFT_DISTLOAD")

,bridge_id,struct_def_id,wall_shaft_id,distrib_load_id,load_case_type,load_direction_type,name,dist,offset,length,start_force,end_force


#### ABW_PS_UBEAM_INT_DIAPH_LOC

In [79]:
get_table("BRIDGEWARE", "ABW_PS_UBEAM_INT_DIAPH_LOC")

,bridge_id,struct_def_id,spng_mbr_def_id,loc_id,dist,spacing,num_spaces,thick,weight


#### ABW_PT_LOSS_PROPERTIES

In [80]:
get_table("BRIDGEWARE", "ABW_PT_LOSS_PROPERTIES")

,bridge_id,struct_def_id,pt_properties_id,name,loss_method_type,anchor_set,coeff_friction,wobble_coeff,ps_trans_stress_ratio,transfer_time,age_deck_placement,final_age,lump_sum_initial_loss,lump_sum_final_loss
0,18358,1,1,PT loss,35004,9.652,0.25,6.561680e-07,NaN,18.0,45.0,10000.0,NaN,NaN
1,5236,1,1,Post-Tension Tendon,35003,9.525,0.25,2.000000e-07,0.8,24.0,7.0,18250.0,203.658711,319.730566
2,3958,1,1,PT Losses,35004,9.525,0.25,6.400000e-07,NaN,NaN,28.0,18250.0,NaN,NaN
3,4275,1,1,AASHTO refined losses,35004,10.000,0.20,6.600000e-07,NaN,24.0,30.0,20000.0,NaN,NaN
4,4275,2,1,AASHTO refined losses,35004,10.000,0.20,6.600000e-07,NaN,24.0,30.0,20000.0,NaN,NaN
5,19736,1,2,PT Loss,35004,152.400,0.25,6.561680e-07,NaN,18.0,45.0,10000.0,NaN,NaN
6,18765,1,1,PT loss,35004,9.652,0.25,6.561680e-07,NaN,18.0,45.0,10000.0,NaN,NaN
7,18795,3,2,Wizard PS Properties,35004,9.500,0.25,6.000000e-07,NaN,18.0,45.0,10000.0,NaN,NaN
8,18796,3,2,Wizard PS Properties,35004,9.500,0.25,6.000000e-07,NaN,18.0,45.0,10000.0,NaN,NaN


#### ABW_RATING_RESULTS

In [81]:
get_table("BRIDGEWARE", "ABW_RATING_RESULTS")

,spng_mbr_alt_event_id,point_id,rating_results_id,vehicle_id,vehicle_type,design_method_type,rating_success_type,impact_loading_type,lane_loading_type,inv_capacity,...,legal_inv_capacity,legal_opr_capacity,permit_inv_rf,permit_opr_rf,legal_inv_rf,legal_opr_rf,permit_inv_limit_state,permit_opr_limit_state,legal_inv_limit_state,legal_opr_limit_state
0,849,29,11,2,32810,21701,0,38803,38903,49.882213,...,None,None,None,NaN,None,None,None,None,None,None
1,849,29,12,2,32810,21701,0,38803,38903,134.983038,...,None,None,None,NaN,None,None,None,None,None,None
2,849,29,13,2,32809,21701,0,38803,38903,158.800181,...,None,None,None,NaN,None,None,None,None,None,None
3,849,29,14,2,32809,21701,0,38803,38903,71.063958,...,None,None,None,NaN,None,None,None,None,None,None
4,849,29,15,2,32809,21701,0,38803,38903,149.534950,...,None,None,None,NaN,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311122,1148,369,12,42,32802,21701,0,38803,38903,NaN,...,None,None,None,NaN,None,None,None,None,None,None
311123,1148,369,13,43,32802,21701,0,38803,38903,NaN,...,None,None,None,NaN,None,None,None,None,None,None
311124,1148,369,14,44,32802,21701,0,38803,38903,NaN,...,None,None,None,NaN,None,None,None,None,None,None
311125,1148,369,15,7,32802,21701,0,38803,38903,NaN,...,None,None,None,NaN,None,None,None,None,None,None


#### ABW_RATING_RESULTS_SUMMARY

In [82]:
get_table("BRIDGEWARE", "ABW_RATING_RESULTS_SUMMARY")

,spng_mbr_alt_event_id,rating_results_summary_id,vehicle_id,vehicle_type,design_method_type,rating_success_type,inv_capacity,opr_capacity,post_capacity,safe_capacity,...,legal_inv_location,legal_opr_location,permit_inv_limit_state,permit_opr_limit_state,legal_inv_limit_state,legal_opr_limit_state,permit_inv_element_name,permit_opr_element_name,legal_inv_element_name,legal_opr_element_name
0,856,3,2,32808,21704,24401,43.747688,56.709966,NaN,NaN,...,None,NaN,None,None,None,None,None,None,None,None
1,108,210,5,32802,21702,24402,0.000000,0.000000,NaN,NaN,...,None,NaN,None,None,None,None,None,None,None,None
2,108,211,5,32802,21702,24402,0.000000,0.000000,NaN,NaN,...,None,NaN,None,None,None,None,None,None,None,None
3,108,212,5,32802,21702,24402,0.000000,0.000000,NaN,NaN,...,None,NaN,None,None,None,None,None,None,None,None
4,108,213,5,32802,21702,24402,0.000000,0.000000,NaN,NaN,...,None,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6557,1156,12,43,32802,21704,24401,NaN,NaN,55.882674,NaN,...,None,NaN,None,None,None,None,None,None,None,None
6558,1156,13,44,32802,21704,24401,NaN,NaN,58.223378,NaN,...,None,NaN,None,None,None,None,None,None,None,None
6559,1156,14,7,32802,21704,24401,NaN,NaN,55.509899,NaN,...,None,NaN,None,None,None,None,None,None,None,None
6560,1156,15,9,32802,21704,24401,NaN,NaN,107.681947,NaN,...,None,NaN,None,None,None,None,None,None,None,None


#### ABW_SUBSDEF_FOUND_DEF

In [83]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_FOUND_DEF")

,bridge_id,struct_def_id,found_def_id,subsdef_found_def_type,name,descr,found_seal_bot_elevation,found_seal_conc_id,found_seal_width,found_seal_length,...,reinf_long_top_dev_ind,reinf_long_bot_dev_ind,reinf_trans_top_dev_ind,reinf_trans_bot_dev_ind,subsurface_type,factored_bearing_resistance,reinf_long_top_head_ind,reinf_trans_top_head_ind,reinf_long_bot_head_ind,reinf_trans_bot_head_ind
0,18675,5,1,39501,Pile Foundation Def,None,None,NaN,None,None,...,None,None,None,None,47701,NaN,None,None,None,None
1,4904,3,1,39501,Pile Foundation Def,None,None,NaN,None,None,...,None,None,None,None,47701,NaN,None,None,None,None
2,5564,3,1,39503,Empty Name,None,None,1.0,None,None,...,T,T,T,T,47701,NaN,None,None,None,None
3,3717,2,1,39501,Pile Foundation Def,None,None,NaN,None,None,...,None,None,None,None,47701,3351.617986,None,None,None,None
4,5564,4,1,39503,Empty Name,None,None,1.0,None,None,...,T,T,T,T,47701,NaN,None,None,None,None
5,5564,5,1,39503,Empty Name,None,None,1.0,None,None,...,T,T,T,T,47701,NaN,None,None,None,None
6,5564,6,1,39503,Empty Name,None,None,1.0,None,None,...,T,T,T,T,47701,NaN,None,None,None,None


#### ABW_SUBSDEF_FOUND_DEF_CONCLOAD

In [84]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_FOUND_DEF_CONCLOAD")

,bridge_id,struct_def_id,found_def_id,concent_load_id,name,load_case_type,load_direction_type,local_longitudinal_coord,local_transverse_coord,concent_force,concent_moment


#### ABW_ADV_CONC_RC_BEAM_DEF

In [85]:
get_table("BRIDGEWARE", "ABW_ADV_CONC_RC_BEAM_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id
0,13878,6,1


#### ABW_TRUSS_GUSSET_PLATE_DEF

In [86]:
get_table("BRIDGEWARE", "ABW_TRUSS_GUSSET_PLATE_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id,gusset_plate_def_id,name,descr,location_type,stl_structural_id,double_plate_ind,plate_thick,length_a,length_b,length_c,max_unsupported_edge_length,fastener_min_diameter,fastener_shear_capacity,eccentricity_a,eccentricity_b,eccentricity_c


#### ABW_ABUTMENT_SUB_STRUCT_DEF

In [87]:
get_table("BRIDGEWARE", "ABW_ABUTMENT_SUB_STRUCT_DEF")

,bridge_id,struct_def_id


#### ABW_ACCESS_PRIVILEGE

In [88]:
get_table("BRIDGEWARE", "ABW_ACCESS_PRIVILEGE")

,access_privilege_id,name,descr,creation_event_id,modification_event_id
0,24,Bridge Locking and Unlocking,The user is allowed to perform bridge protecti...,NaN,NaN
1,19,System Defaults,The user is allowed to alter System Defaults a...,12.0,NaN
2,1,Bridge Description,The user is allowed to perform various actions...,12.0,17.0
3,2,Public Bridge Folders,The user is allowed to alter Public Bridge fol...,12.0,NaN
4,3,Design Events,The user is allowed to alter Design events.,12.0,18.0
5,4,Rating Events,The user is allowed to alter Rating events.,12.0,19.0
6,5,Access Rights,The user is allowed to alter Access rights.,12.0,16.0
7,6,Parameters,The user is allowed to alter Access Rights,12.0,NaN
8,7,Log Events,Reserved for future use.,12.0,NaN
9,8,Libraries,The user is allowed to alter Libraries data.,12.0,14.0


#### ABW_ANALYSIS_REPORTS_TEMPLATE

In [89]:
get_table("BRIDGEWARE", "ABW_ANALYSIS_REPORTS_TEMPLATE")

,template_id,report_type,report_output_type
0,85,34202,49201
1,85,34206,49201
2,85,34213,49201
3,85,34214,49201
4,85,34231,49202
...,...,...,...
318,96,34234,49202
319,96,34236,49202
320,96,34237,49202
321,96,34238,49202


#### ABW_ANAL_PT_CONC_REINF

In [90]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_CONC_REINF")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,anal_pt_id,anal_pt_conc_reinf_id,spng_mbr_def_id,reinf_set_id,fully_developed_ind
0,2087,1,1,1,1,1,1,1,T
1,2087,1,1,1,1,2,1,4,T
2,17789,1,1,1,1,2,1,4,F
3,17789,1,1,1,2,1,1,1,F
4,17789,1,1,1,2,2,1,4,F
...,...,...,...,...,...,...,...,...,...
290,5334,1,11,1,2,1,2,1,None
291,5334,1,11,1,3,1,2,1,None
292,5334,1,11,1,4,1,2,1,None
293,5334,1,11,1,5,1,2,1,None


#### ABW_APPROVAL_EVENT

In [91]:
get_table("BRIDGEWARE", "ABW_APPROVAL_EVENT")

,event_id,approved_ind


#### ABW_SUPPORT

In [92]:
get_table("BRIDGEWARE", "ABW_SUPPORT")

,bridge_id,struct_def_id,super_struct_mbr_id,support_id,support_line_id,x_translation_type,y_translation_type,z_translation_type,x_rotation_type,y_rotation_type,...,stg_thr_y_rtn_spring_const,stg_thr_z_rtn_spring_const,stg_thr_x_trans_settlement,stg_thr_y_trans_settlement,stg_thr_z_trans_settlement,stg_thr_x_rotation_settlement,stg_thr_y_rotation_settlement,stg_thr_z_rotation_settlement,stg_thr_ovr_zrot_sprg_cnst_ind,skew_for_skew_adjustment
0,18651,1,1,5,5,25501,25502,25502,25502,25502,...,None,None,None,None,None,None,None,None,None,None
1,1905,1,3,6729,1,25502,25502,25502,25502,25502,...,None,None,None,None,None,None,None,None,None,None
2,1905,1,6,6735,1,25502,25502,25502,25502,25502,...,None,None,None,None,None,None,None,None,None,None
3,1903,1,7,6780,3,25501,25502,25502,25502,25502,...,None,None,None,None,None,None,None,None,None,None
4,1904,1,1,6787,1,25502,25502,25502,25501,25501,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253113,20425,1,6,3,3,25501,25502,25502,25501,25501,...,None,None,None,None,None,None,None,None,None,None
253114,20425,1,6,4,4,25501,25502,25502,25501,25501,...,None,None,None,None,None,None,None,None,None,None
253115,20425,1,6,5,5,25501,25502,25502,25501,25501,...,None,None,None,None,None,None,None,None,None,None
253116,20425,1,7,1,1,25502,25502,25502,25501,25501,...,None,None,None,None,None,None,None,None,None,None


#### ABW_GUSSET_PLATE_PANEL_POINT

In [93]:
get_table("BRIDGEWARE", "ABW_GUSSET_PLATE_PANEL_POINT")

,bridge_id,struct_def_id,spng_mbr_def_id,panel_point_id,panel_point_name,gusset_def_id,flipped_ind,include_in_analysis_ind
0,3192,1,1,1,L1,4.0,F,T
1,3192,1,1,2,L2,8.0,F,T
2,3192,1,1,3,L3,5.0,F,T
3,3192,1,1,4,L4,8.0,F,T
4,3192,1,1,5,L5,6.0,F,T
...,...,...,...,...,...,...,...,...
1690,13078,1,2,23,M6',7.0,T,F
1691,13078,1,2,24,M4',NaN,None,None
1692,13078,1,2,25,M3',NaN,None,None
1693,13871,2,1,1,U1,NaN,F,F


#### ABW_FSYS_STRGRP_DEF_TEMPLATE

In [94]:
get_table("BRIDGEWARE", "ABW_FSYS_STRGRP_DEF_TEMPLATE")

,bridge_id,struct_def_id,floorsys_stringer_group_def_id,template_id,stringer_mbr_loc_id,std_single_ll_fact_moment,std_single_ll_fact_shear,std_single_ll_fact_shear_supp,std_single_ll_fact_deflection,std_multi_ll_fact_moment,std_multi_ll_fact_shear,std_multi_ll_fact_shear_supp,std_multi_ll_fact_deflection,name,std_dist_fact_perm_load_ind,lrfd_dist_fact_perm_load_ind
0,18126,2,1,1,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T147315,T,F
1,18126,2,1,2,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T147317,None,None
2,18126,2,1,3,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T147319,None,None
3,18126,2,1,4,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T147321,None,None
4,18126,2,1,5,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T147323,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,13052,2,1,4,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stringer120111,None,None
362,13052,2,1,5,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stringer120112,None,None
363,13052,2,1,6,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stringer120113,None,None
364,13052,2,1,7,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stringer120114,None,None


#### ABW_DECK_STL_PANEL

In [95]:
get_table("BRIDGEWARE", "ABW_DECK_STL_PANEL")

,bridge_id,struct_def_id,deck_panel_id


#### ABW_DEFAULT_ANALYSIS_OPTIONS

In [96]:
get_table("BRIDGEWARE", "ABW_DEFAULT_ANALYSIS_OPTIONS")

,bridge_id,bridge_design_alt_id,lib_analysis_options_id,lrfd_factor_id,name,descr


#### ABW_DEF_ANALYSIS_OPTIONS_LS

In [97]:
get_table("BRIDGEWARE", "ABW_DEF_ANALYSIS_OPTIONS_LS")

,bridge_id,bridge_design_alt_id,analysis_option_limit_state_id,lrfd_factor_id,ls_id,include_in_analysis_ind


#### ABW_DEF_ANAL_OPTION_LS_VEHICLE

In [98]:
get_table("BRIDGEWARE", "ABW_DEF_ANAL_OPTION_LS_VEHICLE")

,bridge_id,bridge_design_alt_id,analysis_option_limit_state_id,limit_state_vehicle_id,vehicle_id,include_in_analysis_ind,adjacent_lane_vehicle_id


#### ABW_DEF_ANAL_OPTION_VEHICLE

In [99]:
get_table("BRIDGEWARE", "ABW_DEF_ANAL_OPTION_VEHICLE")

,bridge_id,bridge_design_alt_id,vehicle_id,consider_tandem_pair_ind


#### ABW_DESIGN_EVENT

In [100]:
get_table("BRIDGEWARE", "ABW_DESIGN_EVENT")

,event_id


#### ABW_DIAPH_DEF

In [101]:
get_table("BRIDGEWARE", "ABW_DIAPH_DEF")

,bridge_id,struct_def_id,diaph_def_id


#### ABW_ERROR_EVENT

In [102]:
get_table("BRIDGEWARE", "ABW_ERROR_EVENT")

,event_id


#### ABW_ERROR_MESSAGE

In [103]:
get_table("BRIDGEWARE", "ABW_ERROR_MESSAGE")

,event_id,error_mess_id,error_type,error_short_descr,error_long_descr,prev_error_mess_id


#### ABW_FATIGUE_GROSS_VEHICLE

In [104]:
get_table("BRIDGEWARE", "ABW_FATIGUE_GROSS_VEHICLE")

,bridge_id,fatigue_gross_vehicle_id,min_gross_weight,max_gross_weight,num_2_axle,num_3_axle,num_4_axle,num_3_axle_comb,num_4_axle_comb,num_5_axle_comb


#### ABW_FLINE_STRUCT_DEF

In [105]:
get_table("BRIDGEWARE", "ABW_FLINE_STRUCT_DEF")

,bridge_id,struct_def_id,shared_legs_type,single_lane_ind,mbr_type


#### ABW_LIB_LFR_LOADING

In [106]:
get_table("BRIDGEWARE", "ABW_LIB_LFR_LOADING")

,lfr_factor_id,lfr_loading_id,name,descr,inv_beta_factor,opr_beta_factor,sl_beta_factor,pst_beta_factor
0,2,1,D,Dead Load,1.00,1.0,None,None
1,2,2,(L+I)n,Live Load plus impact for AASHTO Highway H or ...,1.67,1.0,None,None
2,2,3,(L+I)p,Live Load plus impact consistent with the over...,0.00,0.0,None,None
3,2,4,CF,Centrifugal Force,1.00,1.0,None,None
4,2,5,E,Earth pressure,1.00,1.0,None,None
5,2,6,B,Buoyancy,1.00,1.0,None,None
6,2,7,SF,Stream Flow pressure,1.00,1.0,None,None
7,2,8,W,Wind load on structure,0.00,0.0,None,None
8,2,9,WL,Wind load on Live load,0.00,0.0,None,None
9,2,10,LF,Longitudinal Force from live load,0.00,0.0,None,None


#### ABW_RESULTS_CRIT_LOAD_ASR 

In [107]:
get_table("BRIDGEWARE", "ABW_RESULTS_CRIT_LOAD_ASR")

,spng_mbr_alt_event_id,point_id,crit_load_asr_id,load_group_id,stage_id,moment_max,moment_max_cvehicle_id,moment_min,moment_min_cvehicle_id,axial_max,...,shear_min,shear_min_cvehicle_id,reaction_max,reaction_max_cvehicle_id,reaction_min,reaction_min_cvehicle_id,y_defl_max,y_defl_max_cvehicle_id,y_defl_min,y_defl_min_cvehicle_id


#### ABW_FLOORSYS_FLOOR_BEAM_MBR

In [108]:
get_table("BRIDGEWARE", "ABW_FLOORSYS_FLOOR_BEAM_MBR")

,bridge_id,struct_def_id,super_struct_mbr_id,struct_def_ref_line_id,settlement_load_case_id
0,806,1,7,14,None
1,806,1,8,16,None
2,806,1,9,18,None
3,806,5,8,16,None
4,806,4,15,24,None
...,...,...,...,...,...
1564,20416,18,17,87,None
1565,20416,18,18,89,None
1566,20416,18,19,91,None
1567,20416,18,20,93,None


#### ABW_FLOORSYS_MBR_LOC

In [109]:
get_table("BRIDGEWARE", "ABW_FLOORSYS_MBR_LOC")

,bridge_id,struct_def_id,mbr_loc_id,name,floorsys_mbr_loc_type,struct_def_ref_line_id
0,806,1,5,Floor Beam A,38202,13
1,806,1,6,Floor Beam B,38202,15
2,806,1,7,Floor Beam C,38202,17
3,806,1,1,GirderMbrLocation1,38201,3
4,806,1,2,GirderMbrLocation2,38201,5
...,...,...,...,...,...,...
2514,19659,1,17,Floorbeam10,38202,37
2515,19659,1,18,Floorbeam11,38202,39
2516,19659,1,19,Floorbeam12,38202,41
2517,19659,1,20,Floorbeam13,38202,43


#### ABW_SUPER_STRUCT_DEF_DL_DIST

In [110]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_DEF_DL_DIST")

,bridge_id,struct_def_id,dl_dist_id,percentage,dl_dist_type,member_type
0,3495,1,1,12.500,58202,25705
1,3495,1,2,12.500,58202,25705
2,3495,1,3,12.500,58202,25705
3,3495,1,4,12.500,58202,25705
4,3495,1,5,12.500,58202,25705
...,...,...,...,...,...,...
189,17949,2,20,8.333,58201,25708
190,17949,2,21,8.333,58201,25708
191,17949,2,22,8.333,58201,25708
192,17949,2,23,8.334,58201,25708


#### ABW_SUPER_BR_FORCE_SUB_DETAIL

In [111]:
get_table("BRIDGEWARE", "ABW_SUPER_BR_FORCE_SUB_DETAIL")

,bridge_id,sub_struct_def_id,lane_set_id,br_force_detail_id,spng_mbr_bearing_loc_id,calc_long_br_force,calc_trans_br_force,calc_vert_br_force,ovr_long_br_force,ovr_trans_br_force,ovr_vert_br_force,calc_long_br_force_ul,calc_trans_br_force_ul,calc_vert_br_force_ul,ovr_long_br_force_ul,ovr_trans_br_force_ul,ovr_vert_br_force_ul


#### ABW_SUPER_CE_FORCE_SUB

In [112]:
get_table("BRIDGEWARE", "ABW_SUPER_CE_FORCE_SUB")

,bridge_id,sub_struct_def_id,lane_set_id,num_lanes,centrifugal_force


#### ABW_SUPER_CE_FORCE_SUB_DETAIL

In [113]:
get_table("BRIDGEWARE", "ABW_SUPER_CE_FORCE_SUB_DETAIL")

,bridge_id,sub_struct_def_id,lane_set_id,ce_force_detail_id,spng_mbr_bearing_loc_id,calc_long_ce_force,calc_trans_ce_force,calc_vert_ce_force,ovr_long_ce_force,ovr_trans_ce_force,ovr_vert_ce_force


#### ABW_SUPER_DL_REACT_SUB_DETAIL

In [114]:
get_table("BRIDGEWARE", "ABW_SUPER_DL_REACT_SUB_DETAIL")

,bridge_id,sub_struct_def_id,dl_reaction_set_id,dl_react_detail_id,spng_mbr_bearing_loc_id,stage_id,dead_load_type,reaction,reaction_ul
0,5564,3,1,1,1,1,22602,1818.388936,None
1,5564,3,1,2,2,1,22602,1849.296269,None
2,5564,3,1,3,3,1,22602,1849.296269,None
3,5564,3,1,4,4,1,22602,1849.296269,None
4,5564,3,1,5,5,1,22602,1818.388936,None
5,5564,4,1,1,1,1,22602,1727.713323,None
6,5564,4,1,2,2,1,22602,1757.285779,None
7,5564,4,1,3,3,1,22602,1757.285779,None
8,5564,4,1,4,4,1,22602,1757.285779,None
9,5564,4,1,5,5,1,22602,1727.713323,None


#### ABW_SUPER_DLREACT_SUB_OVERRIDE

In [115]:
get_table("BRIDGEWARE", "ABW_SUPER_DLREACT_SUB_OVERRIDE")

,bridge_id,sub_struct_def_id,dl_reaction_set_id,dl_react_override_id,spng_mbr_bearing_loc_id,dead_load_type,reaction,reaction_ul
0,5564,3,1,1,1,22602,None,None
1,5564,3,1,2,1,22603,None,None
2,5564,3,1,3,2,22602,None,None
3,5564,3,1,4,2,22603,None,None
4,5564,3,1,5,3,22602,None,None
5,5564,3,1,6,3,22603,None,None
6,5564,3,1,7,4,22602,None,None
7,5564,3,1,8,4,22603,None,None
8,5564,3,1,9,5,22602,None,None
9,5564,3,1,10,5,22603,None,None


#### ABW_LIB_LFR_FACT_SPEC_EDITION

In [116]:
get_table("BRIDGEWARE", "ABW_LIB_LFR_FACT_SPEC_EDITION")

,lfr_factor_id,factor_spec_edition_id,specification_edition_guid
0,1,1,EE11A0B0-323B-4D6B-B431-1516A27A2E77
1,2,1,EE11A0B0-323B-4D6B-B431-1516A27A2E77
2,1,2,DB57BAAC-9427-40A8-AC56-E3BB764DFA90
3,2,2,DB57BAAC-9427-40A8-AC56-E3BB764DFA90
4,1,3,4BCE9530-4A05-4AA5-BC70-B06319E80F52
5,2,3,4BCE9530-4A05-4AA5-BC70-B06319E80F52
6,1,4,FFA15C11-B2EF-48A2-B761-A84748222736
7,2,4,FFA15C11-B2EF-48A2-B761-A84748222736
8,1,5,8B564F41-BEA5-4DC6-9C68-663891734163
9,2,5,DC90EA8A-873F-491F-BDA3-9896A4C8DB4C


#### ABW_LIB_LFR_FACTOR

In [117]:
get_table("BRIDGEWARE", "ABW_LIB_LFR_FACTOR")

,lfr_factor_id,name,descr,library_type,conc_rc_flex,conc_rc_shear,conc_ps_flex,conc_ps_shear,conc_ps_non_flex,stl_flex,...,stl_gusset_shear_fracture,stl_gusset_compr,stl_gusset_chord_splice,stl_tens_yield_gross_section,stl_rivet_shear,conc_pt_flex,buried_hel_pipe_min_wall_ab,buried_ann_pipe_min_wall_ab,buried_ann_pipe_min_long_ss,buried_struc_pl_box_plas_mom
0,2,2002 AASHTO Std. Specifications,AASHTO Standard Specifications for Highway Bri...,22901,0.9,0.85,1.0,0.9,0.9,1.0,...,0.85,1.0,1.0,1.0,0.8,0.95,1.0,1.0,0.67,1.0
1,1,1996 AASHTO Std. Specifications,AASHTO Standard Specifications for Highway Bri...,22901,0.9,0.85,1.0,0.9,0.9,1.0,...,NaN,NaN,NaN,NaN,NaN,0.95,1.0,1.0,0.67,1.0


#### ABW_RC_CIRC_LEG_DEF

In [118]:
get_table("BRIDGEWARE", "ABW_RC_CIRC_LEG_DEF")

,bridge_id,struct_def_id,frame_leg_def_id


#### ABW_RC_FRAMESHAFT_PIER_STR_DEF

In [119]:
get_table("BRIDGEWARE", "ABW_RC_FRAMESHAFT_PIER_STR_DEF")

,bridge_id,struct_def_id


#### ABW_RC_FRAME_PIER_STRUCT_DEF

In [120]:
get_table("BRIDGEWARE", "ABW_RC_FRAME_PIER_STRUCT_DEF")

,bridge_id,struct_def_id
0,5564,3
1,5564,4
2,5564,5
3,5564,6
4,6370,2
5,6370,3
6,6370,4
7,7533,2
8,7534,3
9,13291,3


#### ABW_RC_I_BEAM_DEF

In [121]:
get_table("BRIDGEWARE", "ABW_RC_I_BEAM_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id
0,1060,1,3
1,4142,1,1
2,4333,2,1
3,4333,2,5
4,5867,1,1
5,5912,1,1
6,5912,1,2
7,5912,1,3
8,5912,1,4
9,5912,1,5


#### ABW_RC_LEG_DEF

In [122]:
get_table("BRIDGEWARE", "ABW_RC_LEG_DEF")

,bridge_id,struct_def_id,frame_leg_def_id,rc_leg_def_type


#### ABW_RC_PIER_STRUCT_DEF

In [123]:
get_table("BRIDGEWARE", "ABW_RC_PIER_STRUCT_DEF")

,bridge_id,struct_def_id,rc_pier_struct_def_type
0,7533,2,25420
1,7534,3,25420
2,18675,5,25424
3,4904,3,25424
4,5564,3,25420
5,3717,2,25424
6,5564,4,25420
7,5564,5,25420
8,5564,6,25420
9,6370,2,25420


#### ABW_RC_PILEBENT_PIER_STR_DEF

In [124]:
get_table("BRIDGEWARE", "ABW_RC_PILEBENT_PIER_STR_DEF")

,bridge_id,struct_def_id,encasement_wall_structural_ind
0,18675,5,F
1,4904,3,F
2,3717,2,F


#### ABW_RC_RECT_LEG_DEF

In [125]:
get_table("BRIDGEWARE", "ABW_RC_RECT_LEG_DEF")

,bridge_id,struct_def_id,frame_leg_def_id


#### ABW_RC_SLAB_BEAM_DEF

In [126]:
get_table("BRIDGEWARE", "ABW_RC_SLAB_BEAM_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id
0,355,1,1
1,356,1,1
2,357,1,2
3,15782,1,1
4,5536,1,1
...,...,...,...
4351,7335,4,2
4352,14607,3,1
4353,14607,3,2
4354,14608,1,1


#### ABW_RC_SOLIDSHAFT_PIER_STR_DEF

In [127]:
get_table("BRIDGEWARE", "ABW_RC_SOLIDSHAFT_PIER_STR_DEF")

,bridge_id,struct_def_id


#### ABW_RC_TEE_BEAM_DEF

In [128]:
get_table("BRIDGEWARE", "ABW_RC_TEE_BEAM_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id
0,24,1,1
1,24,1,2
2,24,1,3
3,24,1,4
4,24,1,5
...,...,...,...
209,17400,1,7
210,19659,5,3
211,19659,5,4
212,19659,6,1


#### ABW_RC_WALL_PIER_STRUCT_DEF

In [129]:
get_table("BRIDGEWARE", "ABW_RC_WALL_PIER_STRUCT_DEF")

,bridge_id,struct_def_id


#### ABW_MATL_ALUMINUM

In [130]:
get_table("BRIDGEWARE", "ABW_MATL_ALUMINUM")

,bridge_id,aluminum_id,name,descr,library_type,si_or_us_type,yield_strength,tens_strength,thermal_exp_coeff,density,mod_of_elast,last_change_timestamp
0,19499,1,Structural Plate 0.100-0.175,"Structural plate (thickness 0.100""-0.175"")",22901,10401,165.474168,241.316495,0.000023,2803.278242,68947.57,2024-12-03 18:21:16.143000
1,16049,1,Structural Plate 0.176-0.250,"Structural plate (thickness 0.176""-0.250"")",22901,10401,165.474168,234.421738,0.000023,2803.278242,68947.57,2022-05-05 19:29:02.000000
2,16049,2,Structural Plate test,Structural plate test,22901,10401,0.000000,0.000000,0.000000,0.000000,0.00,2022-09-07 23:02:43.000000
3,18086,2,Structural Plate 0.100-0.175,"Structural plate (thickness 0.100""-0.175"")",22901,10401,165.474168,241.316495,0.000023,2803.278242,68947.57,2024-07-22 18:49:41.683543
4,18269,1,Structural Plate 0.176-0.250,"Structural plate (thickness 0.176""-0.250"")",22901,10401,165.474168,234.421738,0.000023,2803.278242,68947.57,2024-08-27 18:54:07.580000


#### ABW_MACHINE_EVENT

In [131]:
get_table("BRIDGEWARE", "ABW_MACHINE_EVENT")

,event_id


#### ABW_CULVERT_BAR_MARK_DEF

In [132]:
get_table("BRIDGEWARE", "ABW_CULVERT_BAR_MARK_DEF")

,bridge_id,struct_def_id,culvert_def_id,bar_mark_def_id,name,bar_shape_type,dim_a,dim_b,dim_c,dim_h,...,wwr_or_nominal_diameter,wwr_or_nominal_area,wwr_or_nominal_load,wwr_or_cent_to_cent_spacing,wwr_or_area,dim_w,start_hook_diameter,end_hook_diameter,head_at_start_ind,head_at_end_ind
0,2517,1,3,4,A7-8,52301,4.776216,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
1,2517,1,4,1,A1,52304,2.642616,1.600200,1.6002,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
2,2517,1,4,2,A2-3,52301,4.367784,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
3,2517,1,4,3,A4,52301,2.234184,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
4,2517,1,4,4,A7-8,52301,4.776216,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7793,20436,1,1,1,C501,52301,2.895600,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
7794,20436,1,1,8,C904A,52303,1.117610,0.673099,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
7795,20436,1,1,9,C904B,52301,2.819400,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
7796,20436,1,1,10,C904C,52301,1.143000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None


#### ABW_GLINE_MBR_FRAME_CONN

In [133]:
get_table("BRIDGEWARE", "ABW_GLINE_MBR_FRAME_CONN")

,bridge_id,struct_def_id,super_struct_mbr_id,gline_support_id,column_conc_id,bent_cap_width,num_columns,column_length,percent_fixity_at_column_base,column_xsection_type,top_column_width,bot_column_width,top_column_depth,bot_column_depth,top_column_diameter,bot_column_diameter,num_girders,perc_bent_stiff_per_girder
0,15927,5,2,3,NaN,NaN,NaN,NaN,NaN,37101,NaN,NaN,NaN,NaN,None,None,NaN,NaN
1,5311,1,2,1,NaN,NaN,1.0,NaN,NaN,37101,NaN,NaN,NaN,NaN,None,None,1.0,NaN
2,5311,1,2,2,NaN,NaN,1.0,NaN,NaN,37101,NaN,NaN,NaN,NaN,None,None,1.0,NaN
3,5311,1,3,1,NaN,NaN,1.0,NaN,NaN,37101,NaN,NaN,NaN,NaN,None,None,1.0,NaN
4,5311,1,3,2,NaN,NaN,1.0,NaN,NaN,37101,NaN,NaN,NaN,NaN,None,None,1.0,NaN
5,5311,2,1,1,NaN,NaN,1.0,NaN,NaN,37101,NaN,NaN,NaN,NaN,None,None,1.0,NaN
6,5311,2,1,2,NaN,NaN,1.0,NaN,NaN,37101,NaN,NaN,NaN,NaN,None,None,1.0,NaN
7,4142,1,1,2,1.0,10922.00,1.0,7.972349,100.0,37101,10922.00,10922.00,1219.2,800.1,None,None,1.0,6.98
8,4142,1,1,3,1.0,10922.00,1.0,7.972349,100.0,37101,10922.00,10922.00,1219.2,800.1,None,None,1.0,6.98
9,5360,1,1,1,1.0,9144.00,1.0,3.200400,100.0,37101,9144.00,9144.00,457.2,457.2,None,None,1.0,3.23


#### ABW_GLINE_STRUCT_DEF

In [134]:
get_table("BRIDGEWARE", "ABW_GLINE_STRUCT_DEF")

,bridge_id,struct_def_id,single_lane_ind,mbr_type,truck_traffic_frac_single_lane,num_lanes_avail_truck,override_truck_traffic_ind,deck_type
0,18666,1,F,34501.0,NaN,NaN,F,32701.0
1,3503,2,F,NaN,NaN,NaN,None,32701.0
2,18141,1,F,NaN,NaN,NaN,F,32701.0
3,14679,3,F,34501.0,NaN,NaN,F,32701.0
4,1998,1,F,NaN,NaN,NaN,F,32701.0
...,...,...,...,...,...,...,...,...
1869,20225,1,T,34501.0,NaN,NaN,F,32702.0
1870,16735,1,None,NaN,NaN,NaN,None,32701.0
1871,18929,1,None,NaN,NaN,NaN,None,32701.0
1872,20408,4,T,NaN,NaN,1.0,F,32701.0


#### ABW_MULTI_CELL_BOX_STRUCT_DEF

In [135]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_STRUCT_DEF")

,bridge_id,struct_def_id,structure_type,num_cells,post_tensioned_ind,dist_left_edge_stdef_refline,stagger_perpend_diaphragms_ind,cons_sub_skew_fe_secprop_ind,standalone_struct_type,left_conn_adj_struct_ind,right_conn_adj_struct_ind,start_left_conn_spacing,end_left_conn_spacing,start_right_conn_spacing,end_right_conn_spacing,analyze_webs_only_ind,mcb_structure_type,xsection_orientation_type
0,5236,1,54503,None,T,None,F,T,61101,None,None,None,None,None,None,F,63501,64201
1,3496,1,54503,None,F,None,F,T,61101,None,None,None,None,None,None,F,63501,64201
2,3958,1,54503,None,T,None,F,T,61101,None,None,None,None,None,None,F,63501,64201
3,4275,1,54502,None,T,None,F,T,61101,None,None,None,None,None,None,F,63501,64201
4,4275,2,54502,None,T,None,F,T,61101,None,None,None,None,None,None,F,63501,64201


#### ABW_SLAB_SYS_STRUCT_DEF_FK

In [136]:
get_table("BRIDGEWARE", "ABW_SLAB_SYS_STRUCT_DEF_FK")

,bridge_id,struct_def_id,wearing_surface_load_case_id
0,2091,2,NaN
1,2612,2,2.0
2,3108,1,NaN
3,3167,1,2.0
4,1671,2,1.0
...,...,...,...
2756,20429,1,5.0
2757,20433,1,3.0
2758,20434,1,3.0
2759,20435,1,3.0


#### ABW_PIER_WALL_SHAFT_FOUNDATION

In [137]:
get_table("BRIDGEWARE", "ABW_PIER_WALL_SHAFT_FOUNDATION")

,bridge_id,struct_def_id,wall_shaft_id,wall_shaft_foundation_id,subsdef_found_id


#### ABW_PIER_WALL_TOP_WALL_SHAFT

In [138]:
get_table("BRIDGEWARE", "ABW_PIER_WALL_TOP_WALL_SHAFT")

,bridge_id,struct_def_id,pier_wall_top_id,wall_top_wall_shaft_id,wall_shaft_id


#### ABW_PIER_WSHFT_REINF_PATT_DEF

In [139]:
get_table("BRIDGEWARE", "ABW_PIER_WSHFT_REINF_PATT_DEF")

,bridge_id,struct_def_id,wall_shaft_id,reinf_pattern_def_id,name,bundled_bars_ind


#### ABW_BMDEF_CONC_DECK_RANGE

In [140]:
get_table("BRIDGEWARE", "ABW_BMDEF_CONC_DECK_RANGE")

,bridge_id,struct_def_id,spng_mbr_def_id,deck_range_id,dist,length,conc_id,actual_thick,eff_thick,tributary_width_start,tributary_width_end,eff_width_lrfd,eff_width_std,modular_ratio
0,20117,1,1,1,0.0,169.165587,1,NaN,222.25,NaN,NaN,2768.6,2667.0,7.50
1,20117,1,7,1,0.0,169.165587,1,NaN,222.25,NaN,NaN,2768.6,2667.0,7.50
2,20117,1,12,1,0.0,169.165587,1,NaN,222.25,NaN,NaN,2768.6,2667.0,7.50
3,20117,1,13,1,0.0,169.165587,1,NaN,222.25,NaN,NaN,2768.6,2667.0,7.50
4,20117,1,14,1,0.0,169.165587,1,NaN,222.25,NaN,NaN,2768.6,2667.0,7.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,16040,8,2,1,0.0,20.040600,1,NaN,215.90,NaN,NaN,1562.1,1562.1,6.50
101,16742,3,2,1,0.0,5.334000,1,NaN,165.10,NaN,NaN,1371.6,1333.5,8.00
102,16742,8,2,1,0.0,5.334000,1,NaN,165.10,NaN,NaN,1371.6,1333.5,8.00
103,19659,1,3,1,0.0,53.492400,1,NaN,247.65,NaN,NaN,1574.8,1574.8,7.25


#### ABW_FLOORBEAM_MBR_ALT

In [141]:
get_table("BRIDGEWARE", "ABW_FLOORBEAM_MBR_ALT")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id
0,806,1,7,1
1,806,1,8,1
2,806,1,9,1
3,806,4,15,1
4,806,4,16,1
...,...,...,...,...
1563,20416,18,19,1
1564,20416,18,20,1
1565,20416,18,21,1
1566,20416,18,21,2


#### ABW_LIB_LRFD_LOAD_FACTOR_TABLE

In [142]:
get_table("BRIDGEWARE", "ABW_LIB_LRFD_LOAD_FACTOR_TABLE")

,lrfd_factor_id,load_factor_table_id,table_name
0,9,1,Table 3.4.1-4
1,5,1,Table 3.4.1-4
2,8,1,Table 3.4.1-4
3,3,1,Table 3.4.1-4
4,1,1,Table 3.4.1-4
5,2,1,Table 3.4.1-4
6,6,1,Table 3.4.1-4
7,7,1,Table 3.4.1-4
8,4,1,Table 3.4.1-4
9,12,1,Table 3.4.1-4


#### ABW_CULVERT_STRUCT_DEF

In [143]:
get_table("BRIDGEWARE", "ABW_CULVERT_STRUCT_DEF")

,bridge_id,struct_def_id,impact_factor_type,lrfd_dla_type,lrfd_dla_override,skew_angle,left_opening_rotation,right_opening_rotation,box_len_struct_def_ref_len,soil_id,water_unit_load
0,8119,1,31401,51001,None,NaN,NaN,NaN,NaN,1.0,999.55
1,8117,1,31401,51001,None,NaN,NaN,NaN,NaN,1.0,999.55
2,8120,1,31401,51001,None,NaN,NaN,NaN,NaN,1.0,999.55
3,10604,1,31401,51001,None,0.0,135.0,135.0,20.7264,1.0,999.55
4,10605,1,31401,51001,None,0.0,135.0,135.0,20.7264,1.0,999.55
...,...,...,...,...,...,...,...,...,...,...,...
1269,7689,1,31401,51001,None,NaN,45.0,45.0,NaN,1.0,999.55
1270,8005,1,31401,51001,None,15.0,90.0,90.0,26.8224,1.0,999.55
1271,7691,1,31401,51001,None,NaN,45.0,45.0,NaN,1.0,999.55
1272,8006,1,31401,51001,None,7.5,90.0,90.0,12.1920,1.0,999.55


#### ABW_LIB_LRFR_LOAD_FACTOR_TABLE

In [144]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_LOAD_FACTOR_TABLE")

,lrfr_factor_id,load_factor_table_id,table_name
0,8,1,Table 6A.4.2.2-2
1,5,1,Table 6A.4.2.2-2
2,3,1,Table 6A.4.2.2-2
3,1,1,Table 6A.4.2.2-2
4,2,1,Table 6A.4.2.2-2
5,6,1,Table 6A.4.2.2-2
6,7,1,Table 6A.4.2.2-2
7,4,1,Table 6A.4.2.2-2
8,12,1,Table 6A.4.2.2-2
9,9,1,Table 6A.4.2.2-2


#### ABW_LRFD_LOAD_FACTOR_TABLE

In [145]:
get_table("BRIDGEWARE", "ABW_LRFD_LOAD_FACTOR_TABLE")

,bridge_id,lrfd_factor_id,load_factor_table_id,table_name
0,2,1,1,Table 3.4.1-4
1,3,1,1,Table 3.4.1-4
2,8,1,1,Table 3.4.1-4
3,10,1,1,Table 3.4.1-4
4,18,1,1,Table 3.4.1-4
...,...,...,...,...
2872,18837,1,1,Table 3.4.1-4
2873,18838,1,1,Table 3.4.1-4
2874,18840,1,1,Table 3.4.1-4
2875,18846,1,1,Table 3.4.1-4


#### ABW_FLINE_NODE

In [146]:
get_table("BRIDGEWARE", "ABW_FLINE_NODE")

,bridge_id,struct_def_id,super_struct_mbr_id,node_id,x,y,z,x_translation_type,y_translation_type,z_rotation_type,x_translation_spring_constant,y_translation_spring_constant,z_rotation_spring_constant,x_translation_settlement,y_translation_settlement,z_rotation_settlement


#### ABW_FLOORLINE_FLOOR_BEAM_MBR

In [147]:
get_table("BRIDGEWARE", "ABW_FLOORLINE_FLOOR_BEAM_MBR")

,bridge_id,struct_def_id,super_struct_mbr_id,flrbm_share_stringer_spans_ind,settlement_load_case_id,deck_crack_control_param_z,length,dist_to_prev_adj_floor_beam,dist_to_next_adj_floor_beam,deck_exposure_factor
0,8322,4,1,F,None,None,27.403415,NaN,NaN,None
1,11883,2,4,F,None,None,4.241801,NaN,NaN,None
2,13078,2,2,T,None,None,1.905000,NaN,NaN,None
3,12853,6,2,None,None,None,7.620000,7.62,7.62,None


#### ABW_SYS_ANAL_MODULE_MBR_TYPE

In [148]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MODULE_MBR_TYPE")

,module_guid,anal_module_mbr_type_id,mbr_type,mbr_use_type
0,240C916E-0B52-44CF-B74C-1AB3059953F5,481,24119,37901
1,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,484,24102,37901
2,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,485,24103,37901
3,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,486,24104,37901
4,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,487,24106,37901
...,...,...,...,...
470,E8A0F6D6-6AD8-439b-83AF-567BD522BE89,476,24120,37904
471,E8A0F6D6-6AD8-439b-83AF-567BD522BE89,477,24121,37901
472,E8A0F6D6-6AD8-439b-83AF-567BD522BE89,478,24121,37902
473,E8A0F6D6-6AD8-439b-83AF-567BD522BE89,479,24121,37903


#### ABW_FLOORLINE_MBR_SUPPORT

In [149]:
get_table("BRIDGEWARE", "ABW_FLOORLINE_MBR_SUPPORT")

,bridge_id,struct_def_id,super_struct_mbr_id,floorline_support_id,dist,x_translation_type,y_translation_type,z_rotation_type,x_translation_spring_constant,y_translation_spring_constant,z_rotation_spring_constant,x_translation_settlement,y_translation_settlement,z_rotation_settlement
0,8322,4,2,1,0.000000,25502,25502,25501,None,None,None,None,None,None
1,8322,4,2,2,13.887450,25501,25502,25501,None,None,None,None,None,None
2,8322,4,2,3,31.272175,25501,25502,25501,None,None,None,None,None,None
3,8322,4,2,4,48.645775,25501,25502,25501,None,None,None,None,None,None
4,8322,4,2,5,66.019375,25501,25502,25501,None,None,None,None,None,None
5,8322,4,2,6,83.392975,25501,25502,25501,None,None,None,None,None,None
6,8322,4,2,7,100.766575,25501,25502,25501,None,None,None,None,None,None
7,8322,4,2,8,114.634975,25501,25502,25501,None,None,None,None,None,None
8,8322,4,1,5,0.304800,25502,25502,25501,None,None,None,None,None,None
9,8322,4,1,6,12.395210,25502,25502,25501,None,None,None,None,None,None


#### ABW_FLOORLINE_STRINGER_MBR

In [150]:
get_table("BRIDGEWARE", "ABW_FLOORLINE_STRINGER_MBR")

,bridge_id,struct_def_id,super_struct_mbr_id,stringer_location_type,length,settlement_load_case_id,deck_crack_control_param_z,stringer_spacing,deck_exposure_factor
0,8322,4,2,31801,114.634975,None,None,2.400300,None
1,11883,2,3,31801,79.225780,None,None,2.120900,None
2,13078,2,1,31801,2.743200,None,None,1.929384,None


#### ABW_FLOORLINE_STRUCT_DEF

In [151]:
get_table("BRIDGEWARE", "ABW_FLOORLINE_STRUCT_DEF")

,bridge_id,struct_def_id,main_member_config_type,floorline_struct_def_type,main_members_support_deck_ind,stringer_frame_into_flrbm_ind,single_lane_ind,truck_traffic_frac_single_lane,num_lanes_avail_truck,override_truck_traffic_ind
0,20416,2,38101,25416,T,T,F,None,None,F
1,20416,3,38101,25416,T,T,F,None,None,F
2,20416,15,38101,25416,T,T,F,None,None,F
3,20416,16,38101,25416,T,T,F,None,None,F
4,20416,17,38101,25416,T,T,F,None,None,F
5,8322,4,38101,25418,None,None,F,None,None,F
6,11883,2,38101,25416,T,F,F,None,None,F
7,5624,2,38101,25416,T,F,F,None,None,F
8,13078,2,38101,25416,T,T,F,None,None,F
9,12853,6,38101,25429,None,None,F,None,None,F


#### ABW_GIRDER_MBR_ALT

In [152]:
get_table("BRIDGEWARE", "ABW_GIRDER_MBR_ALT")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id
0,18347,1,1,1
1,18347,1,2,2
2,18347,1,3,1
3,18347,1,4,1
4,18347,1,5,1
...,...,...,...,...
48295,19453,1,2,1
48296,19454,1,1,1
48297,19454,1,2,1
48298,19455,1,1,1


#### ABW_TRUSS_MBR_ALT

In [153]:
get_table("BRIDGEWARE", "ABW_TRUSS_MBR_ALT")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id
0,1904,1,1,1
1,1904,1,2,1
2,2163,3,1,1
3,2163,3,2,1
4,2163,4,1,1
...,...,...,...,...
96,13871,2,2,1
97,13880,5,1,1
98,13880,5,2,1
99,18126,2,1,1


#### ABW_STRINGER_MBR_ALT

In [154]:
get_table("BRIDGEWARE", "ABW_STRINGER_MBR_ALT")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id
0,806,1,3,1
1,806,1,4,1
2,806,1,5,1
3,806,1,6,1
4,806,4,1,1
...,...,...,...,...
3497,20416,18,6,1
3498,20416,18,7,1
3499,20416,18,7,2
3500,20416,18,8,1


#### ABW_STL_XSECTION_COVER_PLATE

In [155]:
get_table("BRIDGEWARE", "ABW_STL_XSECTION_COVER_PLATE")

,bridge_id,struct_def_id,spng_mbr_def_id,xsection_id,cover_plate_id,width,thick,relative_position,stl_structural_id,side_weld_id,end_weld_id,bolt_hole_size,bolt_hole_effective_num,connector_type,bolt_hole_pitch,bolt_hole_gage,development_length_start,percent_developed_start,development_length_end,percent_developed_end
0,3437,1,4,3,3,457.2,9.525,-1,1,NaN,NaN,22.225,2.0,26902,127.0,279.4,0.0,100.0,0.0,100.0
1,3437,1,4,3,4,457.2,9.525,-2,1,NaN,NaN,22.225,2.0,26902,127.0,279.4,0.0,100.0,0.0,100.0
2,3437,1,4,4,1,457.2,12.700,1,1,NaN,NaN,22.225,2.0,26902,127.0,279.4,0.0,100.0,0.0,100.0
3,3437,1,4,4,2,457.2,9.525,2,1,NaN,NaN,22.225,2.0,26902,127.0,279.4,0.0,100.0,0.0,100.0
4,3437,1,4,4,3,457.2,12.700,-1,1,NaN,NaN,22.225,2.0,26902,127.0,279.4,0.0,100.0,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14811,20416,12,20,30,7,457.2,25.400,-2,11,NaN,NaN,NaN,NaN,26902,NaN,NaN,0.0,100.0,0.0,100.0
14812,20416,12,20,30,8,457.2,25.400,-3,11,NaN,NaN,NaN,NaN,26902,NaN,NaN,0.0,100.0,0.0,100.0
14813,20416,12,20,30,9,457.2,25.400,-4,11,NaN,NaN,NaN,NaN,26902,NaN,NaN,0.0,100.0,0.0,100.0
14814,20416,12,20,30,10,457.2,15.875,-5,12,NaN,NaN,NaN,NaN,26902,NaN,NaN,0.0,100.0,0.0,100.0


#### ABW_STL_XSECTION_ELEMENT

In [156]:
get_table("BRIDGEWARE", "ABW_STL_XSECTION_ELEMENT")

,bridge_id,spng_mbr_def_id,struct_def_id,xsection_id,element_id,element_type,h,x,y,dx,dy


#### ABW_STL_XSECTION_RANGE

In [157]:
get_table("BRIDGEWARE", "ABW_STL_XSECTION_RANGE")

,bridge_id,struct_def_id,spng_mbr_def_id,range_id,dist,start_section_id,end_section_id,length,web_variation_type
0,5447,1,1,1,0.000000,1,1,25.298400,26202
1,5447,1,1,2,25.298400,2,2,24.384000,26202
2,5447,1,1,3,49.682400,3,3,22.555200,26202
3,5447,1,1,4,72.237600,4,4,30.480000,26202
4,5447,1,1,5,102.717600,5,5,24.384000,26202
...,...,...,...,...,...,...,...,...,...
16699,20416,18,12,5,29.187648,4,4,5.486400,26202
16700,20416,18,12,6,34.674048,3,3,2.840736,26202
16701,20416,18,12,7,37.514784,1,1,5.922264,26202
16702,20416,18,12,8,43.437048,2,2,12.344400,26202


#### ABW_STL_XSECTION_REINF

In [158]:
get_table("BRIDGEWARE", "ABW_STL_XSECTION_REINF")

,bridge_id,struct_def_id,spng_mbr_def_id,xsection_id,row_id,vert_dist,num_bars_std,row_ind,rebar_id,stl_reinf_id,dist_ref_type,num_bars_lrfd,bar_spacing
0,18141,1,2,3,1,73.1520,7.68,None,18,1,21801,NaN,NaN
1,18141,1,2,1,1,76.2000,2.40,None,18,1,21801,NaN,NaN
2,1763,1,1,1,1,76.2000,10.50,None,13,1,21801,10.5,228.6
3,1763,1,1,1,2,46.0375,9.50,None,14,1,21802,9.5,228.6
4,1763,1,1,2,1,76.2000,10.50,None,13,1,21801,10.5,228.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3692,10160,2,9,2,1,71.4375,11.00,None,14,1,21801,11.0,NaN
3693,10160,2,9,2,2,49.2125,6.00,None,14,1,21802,6.0,NaN
3694,10160,2,9,3,1,71.4375,11.00,None,14,1,21801,11.0,NaN
3695,10160,2,9,3,2,49.2125,6.00,None,14,1,21802,6.0,NaN


#### ABW_STLBD_TOPFL_LSUP_LOC_RANGE

In [159]:
get_table("BRIDGEWARE", "ABW_STLBD_TOPFL_LSUP_LOC_RANGE")

,bridge_id,struct_def_id,spng_mbr_def_id,top_flng_lat_support_range_id,dist,spacing,num_spaces
0,20416,1,1,1,0.000000,2.28600,9
1,20416,1,3,1,0.000000,2.28600,9
2,20416,1,4,1,0.000000,2.28600,9
3,20416,1,5,1,0.000000,2.28600,9
4,20416,1,6,1,0.000000,2.28600,9
...,...,...,...,...,...,...,...
73,4294,3,2,1,0.228600,0.00000,1
74,13871,2,4,1,0.711098,0.00000,1
75,13871,2,4,2,0.711098,1.11761,5
76,13871,3,4,1,0.000000,0.71120,1


#### ABW_STLBD_TOPFLNG_LATSUP_RANGE

In [160]:
get_table("BRIDGEWARE", "ABW_STLBD_TOPFLNG_LATSUP_RANGE")

,bridge_id,struct_def_id,spng_mbr_def_id,top_flng_lat_support_range_id,dist,length
0,18126,2,3,1,0.0,8.848649
1,18126,2,5,1,0.0,8.848649
2,18126,2,6,1,0.0,8.848649
3,18126,2,7,1,0.0,8.848649
4,18126,2,8,1,0.0,8.848649
...,...,...,...,...,...,...
289,19659,1,3,1,0.0,53.492400
290,19659,1,8,1,0.0,53.492400
291,20260,3,6,1,0.0,15.240000
292,20260,3,7,1,0.0,15.240000


#### ABW_CULVERT_STRUCT_ALT

In [161]:
get_table("BRIDGEWARE", "ABW_CULVERT_STRUCT_ALT")

,bridge_id,bridge_design_alt_id,culvert_id,culvert_struct_alt_id,name,descr,struct_def_id,last_modified_event_id,creation_event_id,last_change_timestamp
0,3610,3,1,1,EX,None,1.0,None,None,2022-05-11 19:28:53.688708
1,19848,1,4,1,West End Rigid Frame,None,2.0,None,None,2023-04-11 11:10:36.167000
2,16893,1,1,1,Culvert,None,1.0,None,None,NaT
3,20194,1,1,1,HUR-18-25.150,None,1.0,None,None,2025-04-12 22:06:01.220000
4,20255,1,1,1,Culvert Structure Alt 1,None,1.0,None,None,NaT
...,...,...,...,...,...,...,...,...,...,...
1207,19155,1,1,1,5502551,None,2.0,None,None,2024-12-03 21:07:57.301000
1208,19156,1,1,1,5502586,None,2.0,None,None,2024-12-04 13:36:24.286000
1209,19157,1,1,1,5502616,None,2.0,None,None,2024-12-04 14:02:02.321000
1210,20118,1,1,1,EX,None,1.0,None,None,2025-05-02 14:00:48.470154


#### ABW_EVENT

In [162]:
get_table("BRIDGEWARE", "ABW_EVENT")

,event_id,prev_event_id,event_type,on_behalf_of_user,entered_by,event_timestamp,event_descr,event_message,how_long_to_keep,event_membership_type
0,6669,NaN,21908,None,7.0,2007-01-03 18:48:21.000000,New member alternative created.,None,None,45501
1,6670,NaN,21908,None,7.0,2007-01-03 18:48:21.000000,New member definition created.,None,None,45501
2,18177,NaN,21909,None,48.0,2010-07-26 13:34:44.000000,Structure definition altered.,None,None,45501
3,8002,NaN,21904,None,7.0,2007-04-12 16:36:09.000000,Import of BARS data file,None,None,45501
4,8003,NaN,21908,None,7.0,2007-04-12 16:36:10.000000,New bridge created.,None,None,45501
...,...,...,...,...,...,...,...,...,...,...
1175836,1201509,1201508.0,21909,None,96.0,2025-03-27 19:50:30.245629,Bridge altered.,None,None,45501
1175837,1201510,1201509.0,21909,None,96.0,2025-03-27 19:51:26.379025,Bridge altered.,None,None,45501
1175838,1201827,1187219.0,21909,None,70.0,2025-04-03 13:30:52.415682,Bridge altered.,None,None,45501
1175839,1201858,1201827.0,21909,None,70.0,2025-04-03 14:10:13.674704,Bridge altered.,None,None,45501


#### ABW_MULTI_CELL_BOX_BDEF_SEG

In [163]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_BDEF_SEG")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,name,descr,include_analysis_ind,post_tens_input_method_type,start_distance,length,...,wearing_surface_matl_name,wearing_surface_desc,wearing_surface_density,wearing_surface_thick,thick_field_mea_ind,wearing_surface_load_case_id,std_dist_fact_computed_date,lrfd_dist_fact_computed_date,edge_reference_type,struct_def_ref_line_id
0,3496,1,1,1,Segment 1,None,T,63601,0.0,116.697125,...,None,None,None,None,F,None,None,None,63701,6
1,3958,1,1,1,Segment 1,None,T,63601,0.0,681.228000,...,None,None,None,None,F,None,None,None,63701,9
2,4275,1,1,1,Segment 1,None,T,63601,0.0,188.760000,...,None,None,None,None,F,None,None,None,63701,8
3,4275,2,1,1,Segment 1,None,T,63601,0.0,177.281000,...,None,None,None,None,F,None,None,None,63701,8
4,5236,1,1,1,Segment 1,None,T,63601,0.0,12.788899,...,None,None,None,None,F,None,None,None,63701,4


#### ABW_MULTI_CELL_BOX_INT_DIAPH

In [164]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_INT_DIAPH")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,interior_diaphragm_id,start_dist,diaphragm_spacing,num_spaces,diaphragm_thick,diaphragm_load
0,4275,1,1,1,1,0.610,0.0,1,1220.0,None
1,4275,1,1,1,2,38.000,0.0,1,2440.0,None
2,4275,1,1,1,3,72.686,0.0,1,2440.0,None
3,4275,1,1,1,4,114.260,0.0,1,2440.0,None
4,4275,1,1,1,5,152.260,0.0,1,2440.0,None
5,4275,1,1,1,6,188.150,0.0,1,1220.0,None
6,4275,2,1,1,1,0.610,0.0,1,1220.0,None
7,4275,2,1,1,2,38.032,0.0,1,2440.0,None
8,4275,2,1,1,3,73.046,0.0,1,2440.0,None
9,4275,2,1,1,4,111.596,0.0,1,2440.0,None


#### ABW_MCB_SEG_LANE_POSITION

In [165]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_LANE_POSITION")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,lane_position_id,dist,length,offset_left_start,offset_right_start,offset_left_end,offset_right_end,straight_ind,num_lanes,wheel_dist_lane_edge,passing_dist
0,3496,1,1,1,2,0.0,116.697125,0.4572,8.3820,0.4572,8.3820,None,2,None,None
1,3958,1,1,1,2,0.0,681.228000,-7.9248,7.9248,-7.9248,7.9248,None,4,None,None
2,4275,1,1,1,4,0.0,188.760000,-6.0055,6.4635,-10.5970,10.5970,None,3,None,None
3,4275,2,1,1,4,0.0,177.281000,-5.5650,-0.4930,-6.4000,0.8000,None,1,None,None
4,5236,1,1,1,2,0.0,12.788899,-5.5626,3.5814,-5.5626,3.5814,None,2,None,None


#### ABW_MCB_SEG_STRIPED_LANE_POS

In [166]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_STRIPED_LANE_POS")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,lane_position_id,dist,length,offset_left_start,offset_right_start,offset_left_end,offset_right_end,straight_ind,num_lanes,wheel_dist_lane_edge,passing_dist


#### ABW_MCB_SEG_WEARING_SURFACE

In [167]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_WEARING_SURFACE")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,wearing_surface_id,descr,dist,length,density,straight_ind,offset_start,offset_end,offset_ref_type,thick,width,future_wearing_surface_ind,load_case_id,thick_field_mea_ind


#### ABW_DIAPH_PROPERTIES

In [168]:
get_table("BRIDGEWARE", "ABW_DIAPH_PROPERTIES")

,bridge_id,struct_def_id,diaph_id,bay_num,diaph_num,perform_td_spec_check_ind
0,702,2,1,1,1,F
1,702,2,2,1,2,F
2,702,2,3,1,3,F
3,702,2,4,1,4,F
4,702,2,5,1,5,F
...,...,...,...,...,...,...
668286,18974,2,10,4,8,F
668287,18974,2,11,4,9,F
668288,18974,2,12,4,10,F
668289,18974,2,13,4,11,F


#### ABW_BRIDGE_STREAM_FLOW_EFFECT

In [169]:
get_table("BRIDGEWARE", "ABW_BRIDGE_STREAM_FLOW_EFFECT")

,bridge_id,environmental_param_id,stream_flow_effect_id,skew_angle_operator_type,skew_angle,lateral_drag_coef
0,154,2,4,42703,20.0,0.9
1,154,2,5,42704,30.0,1.0
2,13721,1,2,42703,5.0,0.5
3,13721,1,3,42703,10.0,0.7
4,13721,1,4,42703,20.0,0.9
...,...,...,...,...,...,...
131965,18034,1,1,42703,0.0,0.0
131966,18034,1,2,42703,5.0,0.5
131967,18034,1,3,42703,10.0,0.7
131968,18034,1,4,42703,20.0,0.9


#### ABW_BRIDGE_SUB_LRFD_DS_LS

In [170]:
get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFD_DS_LS")

,bridge_id,lrfd_design_setting_id,design_setting_ls_info_id,bridge_lrfd_factor_id,bridge_lrfd_ls_id,include_in_design_ind,dla_impact_factor
0,3332,1,1,1,1,T,None
1,3332,1,2,1,2,F,None
2,3332,1,3,1,3,T,None
3,3332,1,4,1,4,T,None
4,3332,1,5,1,5,T,None
...,...,...,...,...,...,...,...
70,18262,1,73,26,9,F,None
71,18262,1,74,26,10,T,None
72,18262,1,75,26,11,T,None
73,18262,1,76,26,12,F,None


#### ABW_LIB_SPIRAL_RIB_MP_CULV

In [171]:
get_table("BRIDGEWARE", "ABW_LIB_SPIRAL_RIB_MP_CULV")

,metal_pipe_culvert_id
0,8
1,9
2,10
3,11


#### ABW_LIB_STR_PL_MP_CULV_ITEM

In [172]:
get_table("BRIDGEWARE", "ABW_LIB_STR_PL_MP_CULV_ITEM")

,metal_pipe_culvert_id,str_pl_mp_culvert_item_id,thickness,area,radius,i,mp,bolt_diameter,seam_strength_four_bpf,seam_strength_stl_five_ot_bpf,seam_strength_alu_five_ot_bpf,seam_strength_sixt_bpf,seam_strength_eight_bpf,gage
0,12,1,2.7686,3293.533333,17.32280,9.899629e+05,11.832271,19.050,627.538417,NaN,NaN,NaN,NaN,12.0
1,12,2,3.5052,4239.683333,17.37360,1.281064e+06,15.301884,19.050,904.822833,NaN,NaN,NaN,NaN,10.0
2,12,3,4.2672,5183.716667,17.42440,1.575836e+06,18.771497,19.050,1182.107250,NaN,NaN,NaN,NaN,8.0
3,12,4,4.7752,5797.550000,17.47520,1.769810e+06,21.040090,19.050,1357.234250,NaN,NaN,NaN,NaN,7.0
4,12,5,5.5372,6771.216667,17.52600,2.079887e+06,24.643150,19.050,1634.518667,NaN,NaN,NaN,NaN,5.0
5,12,6,6.3246,7725.833333,17.57680,2.395340e+06,28.290692,19.050,1926.397000,NaN,NaN,NaN,NaN,3.0
6,12,7,7.1120,8718.550000,17.65300,2.717576e+06,31.938234,19.050,2101.524000,NaN,NaN,2626.905,2831.219833,1.0
7,12,8,8.0772,9886.950000,17.72920,3.113555e+06,NaN,22.225,NaN,NaN,NaN,NaN,3429.570417,NaN
8,12,9,9.6520,11880.850000,17.88160,3.801814e+06,NaN,22.225,NaN,NaN,NaN,NaN,4159.266250,NaN
9,13,1,2.5400,2971.800000,21.43252,1.361197e+06,9.430231,19.050,NaN,408.629667,385.2794,NaN,NaN,NaN


#### ABW_LIB_STRUCT_PLATE_MP_CULV

In [173]:
get_table("BRIDGEWARE", "ABW_LIB_STRUCT_PLATE_MP_CULV")

,metal_pipe_culvert_id
0,12
1,13


#### ABW_LIB_METAL_PIPE_CULVERT

In [174]:
get_table("BRIDGEWARE", "ABW_LIB_METAL_PIPE_CULVERT")

,metal_pipe_culvert_id,name,description,si_or_us_type,library_type,lib_metal_pipe_culvert_type,material_type,seam_type
0,1,1 1/2 x 1/4 Corrugated stl. pipe,None,10401,22901,62601,62101,62701
1,2,2 2/3 x 1/2 Corrugated stl. pipe,None,10401,22901,62601,62101,62701
2,3,3 x 1 Corrugated stl. pipe,None,10401,22901,62601,62101,62701
3,4,5 x 1 Corrugated stl. pipe,None,10401,22901,62601,62101,62701
4,5,1 1/2 x 1/4 Corrugated alum. pipe,None,10401,22901,62601,62102,62701
5,6,2 2/3 x 1/2 Corrugated alum. pipe,None,10401,22901,62601,62102,62701
6,7,3 x 1 Corrugated alum. pipe,None,10401,22901,62601,62102,62701
7,8,3/4 x 3/4 x 7 1/2 Spiral rib stl. pipe,None,10401,22901,62602,62101,62702
8,9,3/4 x 1 x 11 1/2 Spiral rib stl. pipe,None,10401,22901,62602,62101,62702
9,10,3/4 x 3/4 x 7 1/2 Alum. spiral rib pipe,None,10401,22901,62602,62102,62702


#### ABW_LIB_METAL_BOX_CULVERT_ITEM

In [175]:
get_table("BRIDGEWARE", "ABW_LIB_METAL_BOX_CULVERT_ITEM")

,metal_box_culvert_id,metal_box_item_id,rib_thickness,alum_rib_type,rib_spacing,shell_thick_111_label,shell_thick_111_mp,shell_thick_140_label,shell_thick_140_mp,shell_thick_170_label,...,metal_thick_150_label,metal_thick_150_mp,metal_thick_175_label,metal_thick_175_mp,metal_thick_200_label,metal_thick_200_mp,metal_thick_225_label,metal_thick_225_mp,metal_thick_250_label,metal_thick_250_mp
0,1,1,NaN,62801,None,1,NaN,2,NaN,3,...,11,14.145346,12,16.502904,13,18.860461,14,21.218019,15,23.575577
1,1,2,NaN,62802,54,1,NaN,2,NaN,3,...,11,24.287292,12,26.867261,13,29.402747,14,31.893752,15,34.429238
2,1,3,NaN,62802,27,1,NaN,2,NaN,3,...,11,32.249609,12,35.318883,13,38.254709,14,41.146054,15,43.903951
3,1,4,NaN,62802,18,1,NaN,2,NaN,3,...,11,38.521603,12,42.169145,13,45.638758,14,48.930442,15,52.088680
4,1,5,NaN,62802,9,1,NaN,2,NaN,3,...,11,53.956933,12,58.182744,13,62.497519,14,66.856777,15,71.260516
5,1,6,NaN,62803,54,1,NaN,2,NaN,3,...,11,30.336874,12,33.050289,13,35.763705,14,38.388156,15,40.968125
6,1,7,NaN,62803,27,1,NaN,2,NaN,3,...,11,42.658449,12,46.217027,13,49.553193,14,52.711431,15,55.825186
7,1,8,NaN,62803,18,1,NaN,2,NaN,3,...,11,52.933842,12,57.115170,13,61.029606,14,64.810595,15,68.458137
8,1,9,NaN,62803,9,1,NaN,2,NaN,3,...,11,82.114178,12,86.339989,13,90.654764,14,95.058504,15,99.506726
9,1,10,NaN,62804,54,1,NaN,2,NaN,3,...,11,42.302591,12,45.549793,13,48.708031,14,51.777304,15,54.802095


#### ABW_SUB_STRUCT_TIMBER_PILE_DEF

In [176]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_TIMBER_PILE_DEF")

,bridge_id,struct_def_id,found_def_id,pile_def_id,width,depth,timber_id


#### ABW_SUB_STRUCT_TU_LOAD

In [177]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_TU_LOAD")

,bridge_id,struct_def_id,sub_tu_load_id,sub_struct_def_component_id,calc_temp_rise_strain,calc_temp_fall_strain,ovr_temp_rise_strain,ovr_temp_fall_strain


#### ABW_LIB_MATL_TIMBER_GLULAM

In [178]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TIMBER_GLULAM")

,timber_matl_id
0,14
1,15
2,16
3,17


#### ABW_LIB_MATL_TMBR_GLULAM_ITEM

In [179]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TMBR_GLULAM_ITEM")

,timber_matl_id,glulam_timber_matl_item_id,combination_type,timber_species_outer_id,timber_species_core_id,density,asd_perp_tens_zone_stress_tens,asd_perp_comp_zone_stress_tens,asd_perp_tension_face,asd_perp_compression_face,...,lrfd_para_shear_para_grain,lrfd_para_mod_of_elasticity,lrfd_para_tens_para_grain,lrfd_axial_tens_para_grain,lrfd_axial_comp_para_grain,lrfd_axial_mod_of_elasticity,lrfd_fast_top_bottom_face,lrfd_fast_side_face,asd_notes,lrfd_notes
0,14,1,62901,3,3,800.936641,13.789514,6.894757,3.447379,2.585534,...,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,None
1,14,2,62902,1,1,800.936641,13.789514,6.894757,4.481592,3.861064,...,1.585794,10342.1355,None,6.894757,10.686873,11031.6112,3.447379,3.447379,None,None
2,14,3,62903,1,1,800.936641,13.789514,13.789514,4.481592,4.481592,...,1.585794,11031.6112,None,7.239495,11.031611,11031.6112,3.447379,3.447379,None,None
3,14,4,62904,3,3,800.936641,13.789514,13.789514,3.447379,3.447379,...,1.310004,9652.6598,None,6.722388,9.652660,10342.1355,2.964746,2.964746,None,None
4,14,5,62905,1,1,800.936641,16.547417,8.273708,4.481592,4.481592,...,1.585794,11031.6112,None,7.584233,11.376349,11721.0869,3.447379,3.447379,None,None
5,14,6,62906,1,3,800.936641,16.547417,8.273708,4.481592,4.481592,...,1.378951,10342.1355,None,7.584233,9.997398,11031.6112,3.447379,2.964746,None,None
6,14,7,62920,1,1,800.936641,16.547417,16.547417,4.481592,4.481592,...,1.585794,11031.6112,None,7.584233,11.376349,11721.0869,3.447379,3.447379,None,None
7,14,8,62908,1,3,800.936641,16.547417,16.547417,4.481592,4.481592,...,1.378951,10342.1355,None,7.928971,10.686873,11031.6112,3.447379,2.964746,None,None
8,15,1,62909,1,1,800.936641,16.547417,8.273708,4.481592,4.481592,...,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,None
9,15,2,62910,1,1,800.936641,16.547417,16.547417,4.481592,4.481592,...,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,None


#### ABW_RC_BEAM_DEF

In [180]:
get_table("BRIDGEWARE", "ABW_RC_BEAM_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id,rc_beam_def_type,asr_shear_comp_method_type,allow_flange_width_vary_ind,a_value,asr_con_slope_bent_lreinf_ind,lfr_con_slope_bent_lreinf_ind,lrfr_con_slope_bent_lreinf_ind,lrfd_con_slope_bent_lreinf_ind,construct_joint_loc_top_flng
0,6091,1,1,24116,49501.0,None,NaN,None,None,None,None,None
1,6092,1,1,24116,49501.0,None,NaN,None,None,None,None,None
2,6346,1,1,24116,49504.0,None,NaN,F,F,F,F,None
3,505,4,1,24116,49504.0,None,NaN,F,F,F,F,None
4,505,4,2,24116,49504.0,None,NaN,F,F,F,F,None
...,...,...,...,...,...,...,...,...,...,...,...,...
4582,20435,1,1,24116,49501.0,None,NaN,F,F,F,F,None
4583,20438,1,1,24116,49501.0,None,NaN,F,F,F,F,None
4584,20438,1,3,24116,49501.0,None,NaN,F,F,F,F,None
4585,20423,1,1,24116,49501.0,None,NaN,F,F,F,F,None


#### ABW_TEMP_GRADIENT_LOAD

In [181]:
get_table("BRIDGEWARE", "ABW_TEMP_GRADIENT_LOAD")

,bridge_id,struct_def_id,temp_gradient_load_id,temp_diff_at_deck,temp_diff_at_100,temp_diff_bot_super_struct,dist_t,dist_a,spec_id,load_case_id
0,16421,3,1,None,None,None,None,None,None,None
1,16755,1,1,None,None,None,None,None,None,None
2,16756,2,1,None,None,None,None,None,None,None
3,16420,2,1,None,None,None,None,None,None,None
4,16423,1,1,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
2247,17626,14,1,None,None,None,None,None,None,None
2248,17630,1,1,None,None,None,None,None,None,None
2249,17646,1,1,None,None,None,None,None,None,None
2250,17655,1,1,None,None,None,None,None,None,None


#### ABW_STL_BEAM_DEF

In [182]:
get_table("BRIDGEWARE", "ABW_STL_BEAM_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id,stl_beam_def_type,lrfd_allow_moment_redist_ind,lrfd_allow_plastic_anal_ind,lrfd_use_appa6_flex_resist_ind,lrfr_allow_moment_redist_ind,lrfr_allow_plastic_anal_ind,lrfr_use_appa6_flex_resist_ind,...,lrfr_con_lat_bend_stres_lg_ind,lrfd_cons_conc_mom_cb_calc_ind,lrfr_cons_conc_mom_cb_calc_ind,lrfd_ltb_gamma_e_mthd_type,lrfr_ltb_gamma_e_mthd_type,lrfd_cons_oly_fuly_dev_cp_ind,lrfr_cons_oly_fuly_dev_cp_ind,lfr_cons_oly_fuly_dev_cp_ind,asr_cons_oly_fuly_dev_cp_ind,lrfr_comp_web_alt_cb_calc_ind
0,1625,1,1,24106,F,F,F,F,F,F,...,None,None,None,64601,64601,None,None,None,None,None
1,1633,1,1,24106,F,F,F,F,F,F,...,None,None,None,64601,64601,None,None,None,None,None
2,1629,1,1,24106,F,F,F,F,F,F,...,None,None,None,64601,64601,None,None,None,None,None
3,1629,1,2,24106,F,F,F,F,F,F,...,None,None,None,64601,64601,None,None,None,None,None
4,1625,1,2,24106,F,F,F,F,F,F,...,None,None,None,64601,64601,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28090,20432,1,6,24107,F,F,F,F,F,F,...,F,None,None,64601,64601,None,None,None,None,None
28091,20437,3,2,24107,F,F,F,F,F,F,...,F,None,None,64601,64601,None,None,None,None,None
28092,20437,3,3,24107,F,F,F,F,F,F,...,F,None,None,64601,64601,None,None,None,None,None
28093,20437,3,4,24107,F,F,F,F,F,F,...,F,None,None,64601,64601,None,None,None,None,None


#### ABW_SUPER_STRUCT_DEF

In [183]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_DEF")

,bridge_id,struct_def_id,super_struct_def_type,nbi_struct_matl_type,nbi_struct_const_type,num_spans,super_struct_service_life,impact_factor_adjustment,impact_factor_override,max_girder_spacing,...,curve_start_tan_length,curve_radius,curve_direction_type,curve_end_tan_length,curve_dist_last_supportline_pt,curve_design_speed,curve_superelevation,min_num_elements_per_span,max_element_central_angle,three_d_fe_node_gen_tol_type
0,8449,1,25407,29804,30102,None,None,NaN,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64001
1,8910,1,25407,29804,30102,None,None,NaN,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64001
2,9465,1,25407,29804,30102,None,None,NaN,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64001
3,4424,1,25407,29804,30102,None,None,NaN,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64001
4,5457,1,25407,29804,30102,None,None,NaN,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14906,19271,3,25406,29801,30101,None,None,NaN,NaN,None,...,NaN,NaN,40201.0,NaN,NaN,NaN,NaN,NaN,NaN,64001
14907,19356,1,25406,29801,30101,None,None,NaN,NaN,None,...,NaN,NaN,40201.0,NaN,NaN,NaN,NaN,NaN,NaN,64001
14908,19356,2,25406,29801,30101,None,None,NaN,NaN,None,...,NaN,NaN,40201.0,NaN,NaN,NaN,NaN,NaN,NaN,64001
14909,19356,3,25406,29801,30101,None,None,NaN,NaN,None,...,NaN,NaN,40201.0,NaN,NaN,NaN,NaN,NaN,NaN,64001


#### ABW_SYS_LINE_GRD_ENG_DEFAULT

In [184]:
get_table("BRIDGEWARE", "ABW_SYS_LINE_GRD_ENG_DEFAULT")

,engine_default_id,member_type,culvert_type,default_rating_method_type,asr_analysis_module_guid,lfr_analysis_module_guid,lrfr_analysis_module_guid,lrfd_analysis_module_guid
0,1,24103.0,NaN,33902,75BA513B-0C0A-4E92-8F5D-FE4233875977,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,BE4A3542-9731-4FC3-B1D1-66F409C03827,553254C5-C4CC-43B8-A6E6-FF99FB6330B3
1,2,24110.0,NaN,33902,75BA513B-0C0A-4E92-8F5D-FE4233875977,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,BE4A3542-9731-4FC3-B1D1-66F409C03827,553254C5-C4CC-43B8-A6E6-FF99FB6330B3
2,3,24114.0,NaN,33902,75BA513B-0C0A-4E92-8F5D-FE4233875977,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,BE4A3542-9731-4FC3-B1D1-66F409C03827,553254C5-C4CC-43B8-A6E6-FF99FB6330B3
3,4,24128.0,NaN,33902,75BA513B-0C0A-4E92-8F5D-FE4233875977,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,BE4A3542-9731-4FC3-B1D1-66F409C03827,553254C5-C4CC-43B8-A6E6-FF99FB6330B3
4,5,24105.0,NaN,33901,13D8F039-C740-45F3-B145-79CFF7256416,None,B85BEA26-000A-4548-8D5D-EAD3F5E72133,None
5,6,24127.0,NaN,33902,75BA513B-0C0A-4E92-8F5D-FE4233875977,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,BE4A3542-9731-4FC3-B1D1-66F409C03827,553254C5-C4CC-43B8-A6E6-FF99FB6330B3
6,7,24101.0,NaN,33902,None,96FD58B4-7F01-46FA-A8E4-88D3EA6D3254,F97A6574-26F7-4ABA-9161-720EAC089C87,None
7,8,24125.0,NaN,33902,75BA513B-0C0A-4E92-8F5D-FE4233875977,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,BE4A3542-9731-4FC3-B1D1-66F409C03827,553254C5-C4CC-43B8-A6E6-FF99FB6330B3
8,9,NaN,50601.0,33904,None,E351F506-F3B3-432F-AE76-EAD6B64F90E5,EA699976-0293-412E-8A94-13467BB631C5,76A73D96-23ED-4FD1-8FA7-C29FAF07DE32
9,10,NaN,50608.0,33904,None,EB91281C-9CB8-4B3F-B58F-1E267370680F,1184D45A-8840-48CB-93C0-BD2CE982D9D2,None


#### ABW_SYS_MODULE_SPEC_DEFAULT

In [185]:
get_table("BRIDGEWARE", "ABW_SYS_MODULE_SPEC_DEFAULT")

,module_guid,default_spec_edition_guid,default_lib_lfr_factor_id,default_lib_lrfd_factor_id,default_lib_lrfr_factor_id
0,0217E0F6-A21A-4133-A9A6-0AB35AF282F0,E695393F-25FA-4F9D-B35A-6D661E5B251D,NaN,NaN,2.0
1,245496D6-0A35-46B8-A97F-66CA334E9702,EE11A0B0-323B-4D6B-B431-1516A27A2E77,NaN,NaN,NaN
2,7DF95E9D-6326-4DD7-800C-A04B3AB3191D,8B564F41-BEA5-4DC6-9C68-663891734163,2.0,NaN,NaN
3,D3D3505F-72FB-4F5E-B13A-7EFC26E3D49D,6F9B56BA-D0F2-48E5-B139-693A46DE3052,NaN,5.0,NaN
4,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,F9E28C1B-6942-488E-AB06-CDFA396512FE,NaN,12.0,NaN
5,75BA513B-0C0A-4E92-8F5D-FE4233875977,8E450923-9513-4CE3-B3D4-AB58C3287560,NaN,NaN,NaN
6,76A73D96-23ED-4FD1-8FA7-C29FAF07DE32,F9E28C1B-6942-488E-AB06-CDFA396512FE,NaN,12.0,NaN
7,96FD58B4-7F01-46FA-A8E4-88D3EA6D3254,8E450923-9513-4CE3-B3D4-AB58C3287560,2.0,NaN,NaN
8,BE4A3542-9731-4FC3-B1D1-66F409C03827,B51A3690-1C49-4E2F-A22D-78DAE730A760,NaN,NaN,12.0
9,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,8E450923-9513-4CE3-B3D4-AB58C3287560,2.0,NaN,NaN


#### ABW_SYS_THREE_D_ENG_DEFAULT

In [186]:
get_table("BRIDGEWARE", "ABW_SYS_THREE_D_ENG_DEFAULT")

,engine_default_id,structure_type,culvert_type,asr_analysis_module_guid,lfr_analysis_module_guid,lrfr_analysis_module_guid,lrfd_analysis_module_guid
0,1,25406.0,NaN,75BA513B-0C0A-4E92-8F5D-FE4233875977,DAF6E1A1-2B96-49E9-A99D-9625BFB527CC,BE4A3542-9731-4FC3-B1D1-66F409C03827,553254C5-C4CC-43B8-A6E6-FF99FB6330B3
1,2,25404.0,NaN,None,None,None,None
2,3,25408.0,NaN,None,None,None,None
3,4,25411.0,NaN,None,None,None,None
4,5,25431.0,NaN,None,None,None,None
5,6,NaN,50601.0,None,None,None,None
6,7,NaN,50608.0,None,None,None,None


#### ABW_BMDEF_INT_SUPPORT_DETAIL

In [187]:
get_table("BRIDGEWARE", "ABW_BMDEF_INT_SUPPORT_DETAIL")

,bridge_id,struct_def_id,spng_mbr_def_id,int_support_detail_id,support_num,left_support_dist,right_support_dist,left_beam_proj,right_beam_proj,top_out_flng_conn_width,...,bot_out_flng_conn_width,bot_out_flng_conn_thick,bot_out_flng_conn_left_length,bot_out_flng_conn_right_length,bot_out_conn_stl_matl_id,bot_in_flng_conn_width,bot_in_flng_conn_thick,bot_in_flng_conn_left_length,bot_in_flng_conn_right_length,bot_in_conn_stl_matl_id
0,4423,1,3,1,2,215.9,215.9,203.2,203.2,None,...,None,None,None,None,1,None,None,None,None,1
1,4423,1,3,2,3,215.9,215.9,203.2,203.2,None,...,None,None,None,None,1,None,None,None,None,1
2,4423,1,4,1,2,215.9,215.9,203.2,203.2,None,...,None,None,None,None,1,None,None,None,None,1
3,4423,1,4,2,3,215.9,215.9,203.2,203.2,None,...,None,None,None,None,1,None,None,None,None,1


#### ABW_BMDEF_INTERMEDIATE_SUPPORT

In [188]:
get_table("BRIDGEWARE", "ABW_BMDEF_INTERMEDIATE_SUPPORT")

,bridge_id,struct_def_id,spng_mbr_def_id,intermediate_support_id,dist,x_translation_type,y_translation_type,z_rotation_type
0,18126,2,5,1,0.957255,25502,25502,25501
1,18126,2,5,2,1.947864,25502,25502,25501
2,18126,2,5,3,2.938272,25502,25502,25501
3,18126,2,5,4,3.928872,25502,25502,25501
4,18126,2,5,5,4.919472,25502,25502,25501
5,18126,2,5,6,5.910072,25502,25502,25501
6,18126,2,5,7,6.900672,25502,25502,25501
7,18126,2,5,8,7.891272,25502,25502,25501
8,18126,2,5,9,0.000000,25502,25502,25501
9,18315,2,1,1,4.114800,25502,25502,25501


#### ABW_SYS_SPEC_EDITION_SOIL

In [189]:
get_table("BRIDGEWARE", "ABW_SYS_SPEC_EDITION_SOIL")

,sys_spec_soil_id,soil_name,pci_range_low,pci_range_high,pci_rating_value,specification_edition_guid
0,9,Loose sand,0.004072,0.016287,0.008143,B51A3690-1C49-4E2F-A22D-78DAE730A760
1,15,Clayey soils 4 ksf <= qu <= 8ksf,0.023073,0.046146,0.042074,B51A3690-1C49-4E2F-A22D-78DAE730A760
2,17,Loose sand,0.004072,0.016287,0.008143,B51A3690-1C49-4E2F-A22D-78DAE730A760
3,18,Medium dense sand,0.009501,0.078720,0.031216,B51A3690-1C49-4E2F-A22D-78DAE730A760
4,19,Dense sand,0.062433,0.124866,0.078720,B51A3690-1C49-4E2F-A22D-78DAE730A760
5,20,Clayey medium dense sand,0.031216,0.078720,0.054289,B51A3690-1C49-4E2F-A22D-78DAE730A760
6,21,silty medium dense sand,0.023073,0.046146,0.039360,B51A3690-1C49-4E2F-A22D-78DAE730A760
7,22,Clayey soils qu <= 4ksf,0.010858,0.023073,0.016287,B51A3690-1C49-4E2F-A22D-78DAE730A760
8,23,Clayey soils 4 ksf <= qu <= 8ksf,0.023073,0.046146,0.042074,B51A3690-1C49-4E2F-A22D-78DAE730A760
9,24,Clayey soils qu > 8 ksf,0.046146,0.046146,0.046146,B51A3690-1C49-4E2F-A22D-78DAE730A760


#### ABW_FL_FLOORBEAM_STRINGER_SPAN

In [190]:
get_table("BRIDGEWARE", "ABW_FL_FLOORBEAM_STRINGER_SPAN")

,bridge_id,struct_def_id,super_struct_mbr_id,floor_beam_id,stringer_support_type,offset_or_cantilever_length,left_offset_cantilever_length,right_offset_cantilever_length,relative_floor_beam_loc,current_floor_beam_ind
0,8322,4,1,9,38001,None,None,None,0.000000,F
1,8322,4,1,10,38002,None,None,None,13.869985,T
2,8322,4,1,11,38002,None,None,None,31.254698,F
3,8322,4,1,12,38002,None,None,None,48.625128,F
4,8322,4,1,13,38002,None,None,None,65.998728,F
5,8322,4,1,14,38002,None,None,None,83.372328,F
6,8322,4,1,15,38002,None,None,None,100.745928,F
7,8322,4,1,16,38001,None,None,None,114.614328,F
8,11883,2,4,65,38001,None,None,None,0.000000,F
9,11883,2,4,66,38002,None,None,None,6.062655,F


#### ABW_MBR_CONN_LOCATION

In [191]:
get_table("BRIDGEWARE", "ABW_MBR_CONN_LOCATION")

,bridge_id,struct_def_id,super_struct_mbr_id,mbr_conn_id,mbr_conn_def_id,conn_mbr_id,framed_ind,vert_dist,conn_type,x_translation_type,y_translation_type,z_translation_type,z_rotation_type,x_rotation_type,y_rotation_type


#### ABW_MBR_DISTRIB_LOAD

In [192]:
get_table("BRIDGEWARE", "ABW_MBR_DISTRIB_LOAD")

,bridge_id,struct_def_id,super_struct_mbr_id,mbr_distr_load_id,direction_type,load_case_id,dist,length,load_start,load_end,local_axis_ind,ws_field_measured_ind,flexure_percent_of_load,shear_percent_of_load,description,apply_to_full_box_only_ind
0,4689,1,1,1,22701,4,0.0,84.734400,0.729696,0.729696,None,F,NaN,NaN,None,None
1,4689,1,2,1,22701,4,0.0,84.734400,0.729696,0.729696,None,F,NaN,NaN,None,None
2,4689,1,3,1,22701,4,0.0,84.734400,0.729696,0.729696,None,F,NaN,NaN,None,None
3,4689,1,4,1,22701,4,0.0,84.734400,0.729696,0.729696,None,F,NaN,NaN,None,None
4,50,1,1,1,22702,1,0.0,36.880800,37.331239,37.331239,None,None,NaN,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20172,12836,1,3,1,22701,4,0.0,24.304752,1.167513,1.167513,None,F,NaN,NaN,None,None
20173,12836,1,3,2,22701,2,0.0,24.304752,1.459392,1.459392,None,F,NaN,NaN,None,None
20174,12836,1,4,1,22701,4,0.0,24.304752,1.167513,1.167513,None,F,NaN,NaN,None,None
20175,12836,1,4,2,22701,2,0.0,24.304752,1.459392,1.459392,None,F,NaN,NaN,None,None


#### ABW_MBR_SIDEWALK_LL

In [193]:
get_table("BRIDGEWARE", "ABW_MBR_SIDEWALK_LL")

,bridge_id,struct_def_id,super_struct_mbr_id,mbr_sidewalk_ll_id,sidewalk_load,load_case_id


#### ABW_TIMBTRUSS_ELEMENT_XSECTION

In [194]:
get_table("BRIDGEWARE", "ABW_TIMBTRUSS_ELEMENT_XSECTION")

,bridge_id,struct_def_id,spng_mbr_def_id,element_xsection_id


#### ABW_TRUSS_DEF_ELEMENT_LOAD

In [195]:
get_table("BRIDGEWARE", "ABW_TRUSS_DEF_ELEMENT_LOAD")

,bridge_id,struct_def_id,spng_mbr_def_id,truss_def_element_id,element_load_id,truss_def_element_load_type,load_case_id,reference_panel_point_id


#### ABW_BEARING_DEF

In [196]:
get_table("BRIDGEWARE", "ABW_BEARING_DEF")

,bridge_id,struct_def_id,bearing_def_id,bearing_def_type,name,descr


#### ABW_BRIDGE_EXPLORER_SETTING

In [197]:
get_table("BRIDGEWARE", "ABW_BRIDGE_EXPLORER_SETTING")

,profile_id,profile_column_id,column_id,column_order,sort_type
0,2,627,1,1,57701
1,2,628,103,2,57701
2,2,629,104,3,57701
3,2,630,107,4,57701
4,2,631,105,5,57701
...,...,...,...,...,...
143,4,961,118,15,57701
144,4,962,20,16,57701
145,4,963,4,17,57701
146,4,964,5,18,57701


#### ABW_EVENT_COMPONENT

In [198]:
get_table("BRIDGEWARE", "ABW_EVENT_COMPONENT")

,event_id,component_id,analysis_module_guid,analysis_module_component_guid,component_properties
0,32711,212,7DF95E9D-6326-4DD7-800C-A04B3AB3191D,69FCCB96-DA3C-4DE6-95FB-A194B9DAFD08,"b'Brass Std Analysis Event Properties,5.04.00...."
1,32711,213,D3D3505F-72FB-4F5E-B13A-7EFC26E3D49D,F579D61D-89F7-4FC2-84C7-822E473E6519,"b'Brass LRFD Event Properties,5.03.01.01,1,1,0..."
2,33425,215,245496D6-0A35-46B8-A97F-66CA334E9702,69FCCB96-DA3C-4DE6-95FB-A194B9DAFD08,"b'Brass Std Analysis Event Properties,2.00.01...."
3,33425,217,7DF95E9D-6326-4DD7-800C-A04B3AB3191D,69FCCB96-DA3C-4DE6-95FB-A194B9DAFD08,"b'Brass Std Analysis Event Properties,2.00.01...."
4,33425,218,D3D3505F-72FB-4F5E-B13A-7EFC26E3D49D,F579D61D-89F7-4FC2-84C7-822E473E6519,"b'Brass LRFD Event Properties,2.00.01.01,1,1,0..."
...,...,...,...,...,...
293,641983,1197,715B7539-65AB-4843-A02F-4B51D71CA785,E4C3BA4A-9DCB-42C9-8174-290B7B91A7F2,b'<BRASS_GIRDER_ENGINE_PROPERTIES>\n <DESCRI...
294,642020,1200,715B7539-65AB-4843-A02F-4B51D71CA785,E4C3BA4A-9DCB-42C9-8174-290B7B91A7F2,b'<BRASS_GIRDER_ENGINE_PROPERTIES>\n <DESCRI...
295,1156212,1787,0217E0F6-A21A-4133-A9A6-0AB35AF282F0,F579D61D-89F7-4FC2-84C7-822E473E6519,"b'Brass LRFD Event Properties,5.03.01.01,1,1,0..."
296,708857,1269,715B7539-65AB-4843-A02F-4B51D71CA785,E4C3BA4A-9DCB-42C9-8174-290B7B91A7F2,b'<BRASS_GIRDER_ENGINE_PROPERTIES>\n <DESCRI...


#### ABW_EVENT_COMPONENT_TEMPLATE

In [199]:
get_table("BRIDGEWARE", "ABW_EVENT_COMPONENT_TEMPLATE")

,template_id,component_id,analysis_module_guid,analysis_module_component_guid,properties
0,85,843,245496D6-0A35-46B8-A97F-66CA334E9702,69FCCB96-DA3C-4DE6-95FB-A194B9DAFD08,"b'Brass Std Analysis Event Properties,5.04.00...."
1,86,848,715B7539-65AB-4843-A02F-4B51D71CA785,E4C3BA4A-9DCB-42C9-8174-290B7B91A7F2,b'<BRASS_GIRDER_ENGINE_PROPERTIES>\n <DESCRI...
2,85,844,715B7539-65AB-4843-A02F-4B51D71CA785,E4C3BA4A-9DCB-42C9-8174-290B7B91A7F2,b'<BRASS_GIRDER_ENGINE_PROPERTIES>\n <DESCRI...
3,63,351,715B7539-65AB-4843-A02F-4B51D71CA785,E4C3BA4A-9DCB-42C9-8174-290B7B91A7F2,b'<BRASS_GIRDER_ENGINE_PROPERTIES>\n <DESCRI...
4,97,943,0217E0F6-A21A-4133-A9A6-0AB35AF282F0,F579D61D-89F7-4FC2-84C7-822E473E6519,"b'Brass LRFD Event Properties,5.03.01.01,1,1,0..."
...,...,...,...,...,...
70,96,925,715B7539-65AB-4843-A02F-4B51D71CA785,E4C3BA4A-9DCB-42C9-8174-290B7B91A7F2,b'<BRASS_GIRDER_ENGINE_PROPERTIES>\n <DESCRI...
71,103,1032,D3D3505F-72FB-4F5E-B13A-7EFC26E3D49D,F579D61D-89F7-4FC2-84C7-822E473E6519,"b'Brass LRFD Event Properties,5.03.01.01,1,1,0..."
72,103,1033,715B7539-65AB-4843-A02F-4B51D71CA785,E4C3BA4A-9DCB-42C9-8174-290B7B91A7F2,b'<BRASS_GIRDER_ENGINE_PROPERTIES>\n <DESCRI...
73,102,1030,245496D6-0A35-46B8-A97F-66CA334E9702,69FCCB96-DA3C-4DE6-95FB-A194B9DAFD08,"b'Brass Std Analysis Event Properties,5.04.00...."


#### ABW_GROUP_ACCESS_PRIVILEGE

In [200]:
get_table("BRIDGEWARE", "ABW_GROUP_ACCESS_PRIVILEGE")

,group_id,group_access_privilege_id,access_privilege_id,can_create_ind,can_delete_ind,can_read_ind,can_write_ind
0,2,1,1,T,T,T,T
1,2,2,2,T,T,T,T
2,2,3,3,T,T,T,T
3,2,4,4,T,T,T,T
4,2,5,5,T,T,T,T
...,...,...,...,...,...,...,...
150,6,151,27,T,F,T,T
151,6,152,28,F,F,F,F
152,6,153,29,F,F,F,F
153,6,154,30,F,F,F,F


#### ABW_LC_SETTING_WATERLEVEL_TEMP

In [201]:
get_table("BRIDGEWARE", "ABW_LC_SETTING_WATERLEVEL_TEMP")

,load_combo_template_id,water_level_id,sys_design_water_level_id,include_water_level_ind


#### ABW_LIB_MATL_TIMBER

In [202]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TIMBER")

,timber_matl_id,name,descr,library_type,si_or_us_type,lib_matl_timber_type,grading_method_type
0,1,Douglas Fir-Larch,Douglas Fir-Larch,22901,10401,35801,36401
1,2,Eastern Softwoods,Eastern Softwoods,22901,10401,35801,36401
2,3,Hem-Fir,Hem-Fir,22901,10401,35801,36401
3,4,Mixed Southern Pine,Mixed Southern Pine,22901,10401,35801,36401
4,5,Mixed Southern Pine (Dry or Wet),Mixed Southern Pine (Dry or Wet),22901,10401,35801,36401
5,6,Northern Red Oak,Northern Red Oak,22901,10401,35801,36401
6,7,Red Maple,Red Maple,22901,10401,35801,36401
7,8,Red Oak,Red Oak,22901,10401,35801,36401
8,9,Southern Pine,Southern Pine,22901,10401,35801,36401
9,10,Southern Pine (Dry or Wet),Southern Pine (Dry or Wet),22901,10401,35801,36401


#### ABW_LIB_PS_SHAPE

In [203]:
get_table("BRIDGEWARE", "ABW_LIB_PS_SHAPE")

,ps_shape_id,name,spec_id,description,year,library_type,si_or_us_type,lib_ps_shape_type,user_defined_properties_ind
0,152,OH CB17-36 (1960 - 1994),NaN,"Ohio 17"" x 36"" Composite Box Beam",1994,22902,10401,23002,F
1,153,OH CB21-36 (1960 - 1994),NaN,"Ohio 21"" x 36"" Composite Box Beam",1994,22902,10401,23002,F
2,154,OH CB27-36 (1960 - 1994),NaN,"Ohio 27"" x 36"" Composite Box Beam",1960,22902,10401,23002,F
3,155,OH CB33-36 (1960 - 1994),NaN,"Ohio 33"" x 36"" Composite Box Beam",1994,22902,10401,23002,F
4,156,OH CB42-36 (1960 - 1994),NaN,"Ohio 42"" x 36"" Composite Box Beam",1994,22902,10401,23002,F
...,...,...,...,...,...,...,...,...,...
188,257,OH-CB42-36,NaN,"Ohio 42"" x 36"" Composite Box Beam",2002,22902,10401,23002,None
189,258,OH-CB42-48,NaN,"Ohio 42"" x 48"" Composite Box Beam",2002,22902,10401,23002,None
190,259,Beam 24-36,NaN,Precast Beam 24 inches D - 36 inches W,2008,22902,10401,23002,F
191,260,OH B12-36 No-Void,NaN,Ohio Non-composite B12-36,2007,22902,10401,23002,F


#### ABW_LIB_STL_SHAPE

In [204]:
get_table("BRIDGEWARE", "ABW_LIB_STL_SHAPE")

,stl_shape_id,name,spec_id,description,year,library_type,si_or_us_type,lib_stl_shape_type
0,541,WT 5x16.5,3.0,WT 5x16.5 Imported from AISC Tables (1994),1994,22901,10401,23405
1,542,WT 5x15,3.0,WT 5x15 Imported from AISC Tables (1994),1994,22901,10401,23405
2,543,WT 5x13,3.0,WT 5x13 Imported from AISC Tables (1994),1994,22901,10401,23405
3,544,WT 5x11,3.0,WT 5x11 Imported from AISC Tables (1994),1994,22901,10401,23405
4,545,WT 5x9.5,3.0,WT 5x9.5 Imported from AISC Tables (1994),1994,22901,10401,23405
...,...,...,...,...,...,...,...,...
3322,3346,WF 36X182,NaN,36WF182,1957,22902,10401,23401
3323,3345,WF 27X94,NaN,27WF94,1957,22902,10401,23401
3324,3342,WF 33x132,NaN,33WF132,1940,22902,10401,23401
3325,3344,L 6 x 6 11/16,NaN,"Historic Steel Shape, Carnegie Steel",1931,22902,10401,23402


#### ABW_LIB_TIMBER_BEAM_SHAPE

In [205]:
get_table("BRIDGEWARE", "ABW_LIB_TIMBER_BEAM_SHAPE")

,timber_beam_shape_id,name,descr,library_type,si_or_us_type,lib_timber_beam_shape_type
0,2,Timber 12 inches wide,One foot wide timber slab,22902,10401,35901


#### ABW_PIER_WALL_TOP_CONCENT_LOAD

In [206]:
get_table("BRIDGEWARE", "ABW_PIER_WALL_TOP_CONCENT_LOAD")

,bridge_id,struct_def_id,pier_wall_top_id,concent_load_id,load_case_type,load_direction_type,name,dist,concent_force,concent_moment


#### ABW_PIER_WALL_TOP_DISTRIB_LOAD

In [207]:
get_table("BRIDGEWARE", "ABW_PIER_WALL_TOP_DISTRIB_LOAD")

,bridge_id,struct_def_id,pier_wall_top_id,distrib_load_id,load_case_type,load_direction_type,name,dist,length,start_force,end_force,start_moment,end_moment


#### ABW_PIER_WEB_WALL_CONCENT_LOAD

In [208]:
get_table("BRIDGEWARE", "ABW_PIER_WEB_WALL_CONCENT_LOAD")

,bridge_id,struct_def_id,pier_web_wall_id,concent_load_id,name,load_case_type,load_direction_type,dist,offset,concent_force,concent_moment


#### ABW_PIER_WEB_WALL_DISTRIB_LOAD

In [209]:
get_table("BRIDGEWARE", "ABW_PIER_WEB_WALL_DISTRIB_LOAD")

,bridge_id,struct_def_id,pier_web_wall_id,distrib_load_id,name,load_direction_type,load_case_type,dist,offset,length,start_force,end_force


#### ABW_STLTRUSS_ELEMENT_XSECTION

In [210]:
get_table("BRIDGEWARE", "ABW_STLTRUSS_ELEMENT_XSECTION")

,bridge_id,struct_def_id,spng_mbr_def_id,element_xsection_id,stltruss_element_xsection_type,bolt_rivet_hole_size,bolt_rivet_hole_effective_num,bolt_rivet_eff_area_deduction
0,14631,2,2,1,48701,None,None,None
1,14631,2,2,9,48702,None,None,None
2,14631,2,8,1,48701,None,None,None
3,14631,2,8,2,48702,None,None,None
4,14631,2,9,1,48701,None,None,None
5,14631,2,9,2,48702,None,None,None
6,14631,2,10,1,48701,None,None,None
7,14631,2,10,2,48702,None,None,None
8,14631,2,11,1,48701,None,None,None
9,14631,2,11,2,48702,None,None,None


#### ABW_MCB_TEND_PROF_ANCH_REINF

In [211]:
get_table("BRIDGEWARE", "ABW_MCB_TEND_PROF_ANCH_REINF")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,tendon_profile_id,anchorage_reinf_id,num_legs,rebar_id,start_distance,num_spaces,spacing


#### ABW_RATING_TOOL_ANAL_TEMPLATE

In [212]:
get_table("BRIDGEWARE", "ABW_RATING_TOOL_ANAL_TEMPLATE")

,template_id,name,structural_analysis_type,analysis_method_type,poi_stl_gen_tenth_points_ind,poi_stl_gen_xsec_chng_pts_ind,poi_stl_gen_userdef_pts_ind,poi_stl_gen_stiffeners_ind,poi_conc_gen_ten_pts_nosup_ind,poi_conc_gen_support_ind,poi_conc_gen_suppf_critshr_ind,poi_conc_gen_xsec_chng_pts_ind,poi_conc_gen_userdef_pts_ind,overwrite_exist_prcmp_data_ind,stop_on_first_error_ind,override_poi_ind
0,1,Default,46401,32602,T,T,T,T,T,T,T,T,T,F,F,F


#### ABW_CONC_BEAM_DEF

In [213]:
get_table("BRIDGEWARE", "ABW_CONC_BEAM_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id,conc_beam_def_type,side_cover,top_cover,bot_cover,lrfd_shear_comp_method_type,lfr_ignore_shear_ind,bot_crack_control_param_z,...,lfr_poi_gen_suppf_critshr_ind,lrfd_poi_gen_suppf_critshr_ind,lrfr_poi_gen_suppf_critshr_ind,asr_poi_gen_ten_pts_nosup_ind,lfr_poi_gen_ten_pts_nosup_ind,lrfd_poi_gen_ten_pts_nosup_ind,lrfr_poi_gen_ten_pts_nosup_ind,lrfr_allow_moment_redist_ind,lrfr_cons_iter_shear_rtng_ind,lrfr_modify_mcft_theta_ind
0,2119,2,2,24111,None,None,None,34403.0,F,0.0,...,T,T,T,None,T,T,T,None,None,None
1,2119,2,3,24111,None,None,None,34403.0,F,0.0,...,T,T,T,None,T,T,T,None,None,None
2,2125,1,1,24116,None,None,None,34403.0,F,0.0,...,T,T,T,T,T,T,T,None,None,None
3,2131,2,7,24111,None,None,None,34403.0,F,0.0,...,T,T,T,None,T,T,T,None,None,None
4,2131,2,8,24111,None,None,None,34403.0,F,0.0,...,T,T,T,None,T,T,T,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20978,20407,1,4,24116,None,None,None,34403.0,F,NaN,...,T,T,T,T,T,T,T,None,None,None
20979,20400,1,1,24111,None,None,None,34403.0,F,NaN,...,T,T,T,None,T,T,T,F,F,F
20980,20400,1,2,24111,None,None,None,34403.0,F,NaN,...,T,T,T,None,T,T,T,F,F,F
20981,20401,1,1,24111,None,None,None,34403.0,F,NaN,...,T,T,T,None,T,T,T,None,None,None


#### ABW_CULVERT_ALT

In [214]:
get_table("BRIDGEWARE", "ABW_CULVERT_ALT")

,bridge_id,struct_def_id,culvert_alt_id,culvert_def_id,name,descr,units_type,slab_exposure_factor,default_analysis_method_type,construction_type,...,lrfr_field_meas_sect_prop_ind,cons_cond_factor_ind,side_fill_condition_type,lfd_soil_struct_interact_fact,bottom_slab_exposure_factor,wall_ext_slab_exposure_factor,int_sur_slab_exposure_factor,lrfr_vert_earth_ld_mod_factor,lrfr_ll_load_modifier_factor,lrfr_depth_fill_dns_known_ind
0,2830,1,1,1,Culvert Alt 1,Single-cell reinforced concrete box culvert trial,10401,0.75,33904,20701,...,F,F,53301,NaN,NaN,NaN,NaN,NaN,NaN,None
1,2831,1,1,1,Culvert Alt 1,Single-cell reinforced concrete box culvert trial,10401,0.75,33904,20701,...,F,F,53301,NaN,NaN,NaN,NaN,NaN,NaN,None
2,2832,1,1,1,Culvert Alt 1,Single-cell reinforced concrete box culvert trial,10401,0.75,33904,20701,...,F,F,53301,NaN,NaN,NaN,NaN,NaN,NaN,None
3,2833,1,1,1,Culvert Alt 1,Single-cell reinforced concrete box culvert trial,10401,0.75,33904,20701,...,F,F,53301,NaN,NaN,NaN,NaN,NaN,NaN,None
4,2834,1,1,1,Culvert Alt 1,Single-cell reinforced concrete box culvert trial,10401,0.75,33904,20701,...,F,F,53301,NaN,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1292,20402,1,1,1,Culvert 12'x8',None,10401,NaN,33904,20701,...,F,None,53301,NaN,NaN,NaN,NaN,NaN,NaN,F
1293,20248,1,1,1,RC Precast Culvert 12x4,None,10401,NaN,33904,20701,...,F,None,53301,NaN,NaN,NaN,NaN,NaN,NaN,F
1294,20256,1,1,1,Box Culvert,None,10401,1.00,33904,20701,...,F,None,53301,NaN,1.0,1.0,1.0,NaN,NaN,F
1295,20422,3,1,1,Twin Metal Pipe Arch,Span = 9.5'\r\nRise = 6.4167',10401,NaN,33904,20702,...,F,None,53301,NaN,NaN,NaN,NaN,1.05,1.0,F


#### ABW_CULVERT_DEF

In [215]:
get_table("BRIDGEWARE", "ABW_CULVERT_DEF")

,bridge_id,struct_def_id,culvert_def_id,culvert_def_type,default_conc_matl_id,default_stl_reinf_id,lrfd_gen_tenth_points_ind,lrfd_gen_xsec_chng_pts_ind,lrfd_gen_userdef_pts_ind,lrfd_shear_comp_method_type,...,lrfr_cons_mod_shr_strn_cap_ind,lrfr_ignore_effect_negl_ll_ind,lfr_gen_tenth_points_ind,lfr_gen_xsec_chng_pts_ind,lfr_gen_userdef_pts_ind,lfr_ignore_shear_ind,lfr_exclude_bottom_slab_ind,lfr_incl_hnch_stif_fe_mod_ind,lfr_ignore_effect_negl_ll_ind,lrfd_ignore_effects_neg_ll_ind
0,2888,1,1,50603,1.0,1.0,T,F,T,34402,...,None,None,T,F,T,F,F,None,None,None
1,2889,1,1,50603,1.0,1.0,T,F,T,34402,...,None,None,T,F,T,F,F,None,None,None
2,2890,1,1,50603,1.0,1.0,T,F,T,34402,...,None,None,T,F,T,F,F,None,None,None
3,2891,1,1,50603,1.0,1.0,T,F,T,34402,...,None,None,T,F,T,F,F,None,None,None
4,3303,3,1,50603,1.0,1.0,T,F,T,34403,...,None,None,T,F,T,F,F,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1299,20102,1,2,50603,1.0,4.0,T,F,T,34403,...,F,F,T,F,T,F,F,F,F,F
1300,20402,1,1,50603,1.0,1.0,T,F,T,34403,...,F,F,T,F,T,F,F,F,F,F
1301,20393,1,2,50603,1.0,1.0,T,F,T,34403,...,F,F,T,F,T,F,F,F,F,F
1302,20436,1,1,50603,1.0,1.0,T,F,T,34403,...,F,F,T,F,T,F,F,F,F,None


#### ABW_CULVERT_DEF_SEG_ANAL_PT

In [216]:
get_table("BRIDGEWARE", "ABW_CULVERT_DEF_SEG_ANAL_PT")

,bridge_id,struct_def_id,culvert_def_id,culvert_def_segment_id,anal_pt_id,culvert_def_seg_anal_pt_type,lrfd_shear_comp_method_type,lrfr_shear_comp_method_type,lfr_ignore_shear_ind
0,1724,1,1,1,1,51601,34401.0,49301.0,F
1,2593,1,1,1,1,51601,NaN,NaN,None
2,2594,1,1,1,1,51601,NaN,NaN,None
3,2595,1,1,1,1,51601,NaN,NaN,None
4,2596,1,1,1,1,51601,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...
486,13448,1,1,1,1,51601,NaN,NaN,None
487,13535,1,1,1,1,51601,NaN,NaN,None
488,15571,1,1,1,1,51601,NaN,NaN,None
489,2635,1,1,1,2,51601,34401.0,49301.0,F


#### ABW_CULVERT_DEF_SEGMENT

In [217]:
get_table("BRIDGEWARE", "ABW_CULVERT_DEF_SEGMENT")

,bridge_id,struct_def_id,culvert_def_id,culvert_def_segment_id,culvert_def_segment_type,name,descr,dist_left_end_to_start_seg,length_seg,impact_factor_type,lrfd_dla_type,depth_fill_start_edge,depth_fill_end_edge,wear_surface_unit_load,wear_surface_thick,lrfd_live_load_sur_height,lfr_live_load_sur_height,water_height,pavement_reduct_factor,pavement_reduct_fact_comment
0,2805,1,1,1,51703,Culvert Seg 1,,NaN,40.8432,31401,51001,3.176016,3.176016,2322.716258,406.4,0.6096,0.6096,0.0000,NaN,None
1,2806,1,1,1,51703,Culvert Seg 1,,NaN,40.8432,31401,51001,0.734568,0.734568,2322.716258,406.4,0.6096,0.6096,0.0000,NaN,None
2,2807,1,1,1,51703,Culvert Seg 1,,NaN,40.8432,31401,51001,0.606552,0.606552,2322.716258,279.4,0.6096,0.6096,0.0000,NaN,None
3,2808,1,1,1,51703,Culvert Seg 1,,NaN,40.8432,31401,51001,0.000000,0.000000,2322.716258,0.0,0.6096,0.6096,0.0000,NaN,None
4,2809,1,1,1,51703,Culvert Seg 1,,NaN,40.8432,31401,51001,0.128016,0.128016,2322.716258,127.0,0.6096,0.6096,0.0000,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1263,20402,1,1,1,51703,Culvert 12'x8',None,0.0,27.4320,31401,51001,0.762000,0.762000,2322.716258,254.0,0.6096,0.6096,NaN,NaN,None
1264,20248,1,1,1,51703,RC Precast Culvert segment 12x4,None,NaN,NaN,31401,51001,0.609600,0.609600,2322.716258,254.0,0.6096,0.6096,NaN,NaN,None
1265,20256,1,1,1,51703,Box Culvert,None,0.0,13.4112,31401,51001,0.304800,0.304800,2322.716258,203.2,0.7620,0.7620,1.3716,NaN,None
1266,20356,1,1,1,51703,C1,None,NaN,NaN,31401,51001,0.457200,0.457200,2322.716258,254.0,0.6096,0.6096,NaN,NaN,None


#### ABW_HAUNCH_RANGE

In [218]:
get_table("BRIDGEWARE", "ABW_HAUNCH_RANGE")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,haunch_range_id,dist,length,y1,y2,y3,z1,z2,z3,z4
0,1733,1,6,1,1,0.0,53.6448,77.597,77.597,NaN,147.32,375.92,838.2,838.2
1,1753,1,1,1,1,0.0,85.3440,133.350,133.350,NaN,228.60,457.20,1066.8,1066.8
2,1753,1,3,1,1,0.0,85.3440,133.350,NaN,NaN,228.60,457.20,NaN,NaN
3,1753,2,3,1,1,0.0,85.3440,133.350,NaN,NaN,228.60,457.20,NaN,NaN
4,1753,2,1,1,1,0.0,85.3440,133.350,133.350,NaN,228.60,457.20,1066.8,1066.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45486,16598,1,2,1,1,0.0,20.5486,0.000,NaN,NaN,0.00,228.60,NaN,NaN
45487,16598,1,3,1,1,0.0,20.5486,0.000,NaN,NaN,0.00,228.60,NaN,NaN
45488,16598,1,4,1,1,0.0,20.5486,0.000,NaN,NaN,0.00,228.60,NaN,NaN
45489,16598,1,5,1,1,0.0,20.5486,0.000,NaN,NaN,0.00,228.60,NaN,NaN


#### ABW_COMMAND_TIMBER_TRUSS_DEF

In [219]:
get_table("BRIDGEWARE", "ABW_COMMAND_TIMBER_TRUSS_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id


#### ABW_COMMAND_TRUSS_DEF

In [220]:
get_table("BRIDGEWARE", "ABW_COMMAND_TRUSS_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id,command_truss_def_type
0,1904,1,1,24101
1,1904,1,2,24101
2,2163,3,1,24101
3,2163,3,2,24101
4,2867,1,1,24101
...,...,...,...,...
96,5062,1,2,24101
97,5179,2,1,24101
98,13880,5,2,24101
99,13052,2,1,24101


#### ABW_DETAILED_STL_TRUSS_DEF

In [221]:
get_table("BRIDGEWARE", "ABW_DETAILED_STL_TRUSS_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id,univ_mill_plate_not_pres_ind
0,14631,2,2,F
1,14631,2,8,F
2,14631,2,9,F
3,14631,2,10,F
4,14631,2,11,F
5,14631,2,12,F
6,14631,2,13,F
7,14631,2,14,F
8,14631,2,15,F
9,14631,2,16,F


#### ABW_DETAILED_TIMBER_TRUSS_DEF

In [222]:
get_table("BRIDGEWARE", "ABW_DETAILED_TIMBER_TRUSS_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id


#### ABW_MA_STRS_IELEM_LOSS_RANGE

In [223]:
get_table("BRIDGEWARE", "ABW_MA_STRS_IELEM_LOSS_RANGE")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,truss_mbr_alt_element_id,truss_element_loss_range_id,ma_strs_ielem_loss_range_type


#### ABW_STL_TRUSS_ROLLED_XSECTION

In [224]:
get_table("BRIDGEWARE", "ABW_STL_TRUSS_ROLLED_XSECTION")

,bridge_id,struct_def_id,spng_mbr_def_id,element_xsection_id,stl_shape_id,stl_structural_id,cover_plate_attachment_type
0,14631,2,2,1,1,1,48402
1,14631,2,8,1,1,1,48402
2,14631,2,9,1,1,1,48402
3,14631,2,10,1,1,1,48402
4,14631,2,11,1,1,1,48402
5,14631,2,12,1,1,1,48402
6,14631,2,13,1,1,1,48402
7,14631,2,14,1,1,1,48402
8,14631,2,15,1,1,1,48402
9,14631,2,16,1,1,1,48402


#### ABW_FLOORSYS_STRUCT_DEF_FK

In [225]:
get_table("BRIDGEWARE", "ABW_FLOORSYS_STRUCT_DEF_FK")

,bridge_id,struct_def_id,wearing_surface_load_case_id
0,806,1,NaN
1,806,4,NaN
2,806,5,NaN
3,1904,1,NaN
4,2163,3,NaN
...,...,...,...
113,13878,3,5.0
114,14631,3,NaN
115,16040,8,NaN
116,13880,5,NaN


#### ABW_FLRBM_MBR

In [226]:
get_table("BRIDGEWARE", "ABW_FLRBM_MBR")

,bridge_id,struct_def_id,super_struct_mbr_id


#### ABW_FRAME_COLUMN_MBR_FK

In [227]:
get_table("BRIDGEWARE", "ABW_FRAME_COLUMN_MBR_FK")

,bridge_id,struct_def_id,super_struct_mbr_id,fline_node_id


#### ABW_LIB_DEFAULT

In [228]:
get_table("BRIDGEWARE", "ABW_LIB_DEFAULT")

,default_units_system_type,default_analysis_template_id,default_rating_template_id,default_analysis_method_type,default_agency_name,check_out_enabled_ind,default_average_humidity,ps_conc_tens_stress_coef_si,ps_conc_tens_stress_coef_us,prelim_sub_lrfd_ds_id,...,def_striii_threesec_gust_speed,load_rating_tool_repository,load_rating_tool_deny_code,load_rating_tool_nr_code,rating_lldf_extbm_uselever_ind,rating_lldf_simpbm_comp_type,multimedia_folder_path,brm_endpoint_url,brm_endpoint_ignore_cert_ind,brm_endpoint_db_id
0,10401,None,None,33902,Ohio Department of Transportation,F,70.0,0.5,0.19,1,...,185.074905,X:\Bridge Management\BrDR_Virtis_precomputed\,X,NA,F,48201,C:\,None,F,None


#### ABW_SUB_PILE_DEF_DISTRIB_LOAD

In [229]:
get_table("BRIDGEWARE", "ABW_SUB_PILE_DEF_DISTRIB_LOAD")

,bridge_id,struct_def_id,found_def_id,pile_def_id,distrib_load_id,name,load_direction_type,load_case_type,dist,offset,length,start_force,end_force


#### ABW_SUB_PILE_FOUND_DEF

In [230]:
get_table("BRIDGEWARE", "ABW_SUB_PILE_FOUND_DEF")

,bridge_id,struct_def_id,found_def_id,cap_width,cap_length,cap_thick,cap_bot_elevation,cap_conc_id,lock_pile_layout_ind,consider_pile_group_action_ind,group_fact_compr_resistance,group_fact_tens_resistance
0,18675,5,1,None,None,None,None,None,None,None,None,None
1,4904,3,1,None,None,None,None,None,None,None,None,None
2,3717,2,1,None,None,None,None,None,None,None,None,None


#### ABW_SUB_PILE_FOUND_DEF

In [231]:
get_table("BRIDGEWARE", "ABW_SUB_PILE_FOUND_DEF")

,bridge_id,struct_def_id,found_def_id,cap_width,cap_length,cap_thick,cap_bot_elevation,cap_conc_id,lock_pile_layout_ind,consider_pile_group_action_ind,group_fact_compr_resistance,group_fact_tens_resistance
0,18675,5,1,None,None,None,None,None,None,None,None,None
1,4904,3,1,None,None,None,None,None,None,None,None,None
2,3717,2,1,None,None,None,None,None,None,None,None,None


#### ABW_SUB_SPREAD_FOUND_DEF

In [232]:
get_table("BRIDGEWARE", "ABW_SUB_SPREAD_FOUND_DEF")

,bridge_id,struct_def_id,found_def_id,width,length,thick,bot_elevation,conc_id
0,5564,3,1,1.8288,10.3632,0.762,225.8568,1
1,5564,4,1,1.8288,10.3632,0.762,225.8568,1
2,5564,5,1,1.8288,10.3632,0.762,225.8568,1
3,5564,6,1,1.8288,10.3632,0.762,225.8568,1


#### ABW_SUB_STRUCT

In [233]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT")

,bridge_id,bridge_design_alt_id,sub_struct_id,sub_struct_type,sub_struct_ref_line_id,name,descr,station,offset,angle,...,ice_jam_pressure,finished_ground_line_elevation,long_load_length_percent,trans_load_length_percent,stream_flow_direction_type,skew_angle_display_type,long_load_dist_super_length,long_load_dist_super_pctlength,super_struct_defined_opis_ind,soil_density
0,4191,1,4,39702,7,Abut 2,None,NaN,None,None,...,NaN,NaN,None,None,40301,NaN,NaN,NaN,None,NaN
1,4216,1,1,39702,4,Abut 1,None,NaN,None,None,...,NaN,NaN,None,None,40301,NaN,NaN,NaN,None,NaN
2,4216,1,2,39701,5,Pier 1,None,NaN,None,None,...,9.576053,NaN,None,None,40301,43701.0,NaN,NaN,None,NaN
3,4216,1,3,39701,6,Pier 2,None,NaN,None,None,...,9.576053,NaN,None,None,40301,43701.0,NaN,NaN,None,NaN
4,4216,1,4,39702,7,Abut 2,None,NaN,None,None,...,NaN,NaN,None,None,40301,NaN,NaN,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3863,19449,2,1,39702,79,Abut 1,None,0.000000,None,None,...,NaN,NaN,None,None,40301,43701.0,NaN,NaN,None,NaN
3864,19449,2,2,39701,80,Pier 1,None,24.886920,None,None,...,0.960000,NaN,None,None,40301,43701.0,NaN,NaN,None,NaN
3865,19676,1,1,39702,4,Abut 1,None,0.000000,None,None,...,NaN,NaN,None,None,40301,43701.0,NaN,NaN,None,NaN
3866,19676,1,2,39701,5,Pier 1,None,26.490077,None,None,...,0.960000,NaN,None,None,40301,43701.0,NaN,NaN,None,NaN


#### ABW_SUB_STRUCT_ALT

In [234]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_ALT")

,bridge_id,bridge_design_alt_id,sub_struct_id,sub_struct_alt_id,name,descr,struct_def_id,last_modified_event_id,creation_event_id,comment_id,...,rep_int_beam_detail_cap_ind,rep_int_beam_summary_cap_ind,rep_int_cw_summary_cap_ind,rep_int_cw_combo_csv_env_ind,rep_int_cw_int_diag_csv_ind,rep_int_cw_capacity_csv_ind,rep_int_fnd_det_brng_pile_ind,rep_int_fnd_smry_brng_pile_ind,rep_int_fnd_det_capacity_ind,rep_int_fnd_smry_capacity_ind
0,5564,1,2,1,FRA-270-4884,None,3,None,None,None,...,F,F,F,None,None,F,F,T,F,F
1,3717,1,2,1,Pier 1,None,2,None,None,None,...,F,F,F,None,None,F,F,T,F,F
2,5564,1,3,1,Copy of FRA-270-4884,None,4,None,None,None,...,F,F,F,None,None,F,F,T,F,F
3,5564,1,4,1,Copy of FRA-270-4884,None,5,None,None,None,...,F,F,F,None,None,F,F,T,F,F
4,5564,1,5,1,Copy of FRA-270-4884,None,6,None,None,None,...,F,F,F,None,None,F,F,T,F,F
5,13291,1,3,1,Copy of pier1,None,5,None,None,None,...,F,F,F,None,None,F,F,T,F,F
6,13291,1,2,1,pier1,None,4,None,None,None,...,F,F,F,None,None,F,F,T,F,F
7,13291,1,4,1,Copy of pier1,None,6,None,None,None,...,F,F,F,None,None,F,F,T,F,F


#### ABW_SUB_STRUCT_DEF

In [235]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_DEF")

,bridge_id,struct_def_id,sub_struct_def_type,combined_footing_ind,ce_force_load_display_type,ce_use_override_values_ind,ce_left_super_struct_height,ce_right_super_struct_height,ce_force_dist_above_deck,br_force_load_display_type,...,super_trans_vehicle_incr_lane,super_trans_move_rightleft_ind,super_long_load_backspan_ind,super_long_load_aheadspan_ind,super_long_load_backahead_ind,super_struct_load_pattern_type,scan_control_super_loadpos_ind,dist_left_cap_left_travelway,travelway_width,wind_load_basis_type
0,18675,5,25424,F,42101,None,None,None,None,42101,...,0.6096,None,None,None,None,46802,None,None,None,60202
1,7533,2,25420,T,42101,None,None,None,None,42101,...,0.6096,None,None,None,None,46802,T,None,None,60202
2,7534,3,25420,T,42101,None,None,None,None,42101,...,0.6096,None,None,None,None,46802,T,None,None,60202
3,4904,3,25424,None,42101,None,None,None,None,42101,...,0.6096,None,None,None,None,46802,T,None,None,60202
4,5564,3,25420,T,42101,None,None,None,None,42101,...,0.6096,F,None,None,None,46802,T,None,None,60202
5,3717,2,25424,None,42101,None,None,None,None,42101,...,0.6096,None,None,None,None,46802,T,None,None,60202
6,5564,4,25420,T,42101,None,None,None,None,42101,...,0.6096,F,None,None,None,46802,T,None,None,60202
7,5564,5,25420,T,42101,None,None,None,None,42101,...,0.6096,F,None,None,None,46802,T,None,None,60202
8,5564,6,25420,T,42101,None,None,None,None,42101,...,0.6096,F,None,None,None,46802,T,None,None,60202
9,6370,2,25420,F,42101,None,None,None,None,42101,...,0.6096,None,None,None,None,46802,T,None,None,60202


#### ABW_SUB_STRUCT_DEF_PEDESTAL

In [236]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_DEF_PEDESTAL")

,bridge_id,sub_struct_def_id,pedestal_id,spng_mbr_bearing_loc_id,width,length,elevation


#### ABW_RESULTS_CONTROL_UNF_ACTION

In [237]:
get_table("BRIDGEWARE", "ABW_RESULTS_CONTROL_UNF_ACTION")

,spng_mbr_alt_event_id,point_id,control_unf_action_id,stage_id,moment_max,moment_max_cvehicle_id,moment_min,moment_min_cvehicle_id,axial_max,axial_max_cvehicle_id,...,shear_min,shear_min_cvehicle_id,reaction_max,reaction_max_cvehicle_id,reaction_min,reaction_min_cvehicle_id,y_defl_max,y_defl_max_cvehicle_id,y_defl_min,y_defl_min_cvehicle_id


#### ABW_LRFD_FACTOR_SPEC_EDITION

In [238]:
get_table("BRIDGEWARE", "ABW_LRFD_FACTOR_SPEC_EDITION")

,bridge_id,lrfd_factor_id,factor_spec_edition_id,specification_edition_guid
0,18221,1,1,70A01C5A-21D9-4C12-9A23-1FD3431D583B
1,19468,1,1,70A01C5A-21D9-4C12-9A23-1FD3431D583B
2,1041,1,4,6F9B56BA-D0F2-48E5-B139-693A46DE3052
3,18222,1,1,70A01C5A-21D9-4C12-9A23-1FD3431D583B
4,18224,1,1,70A01C5A-21D9-4C12-9A23-1FD3431D583B
...,...,...,...,...
3050,19705,1,1,70A01C5A-21D9-4C12-9A23-1FD3431D583B
3051,20417,1,1,70A01C5A-21D9-4C12-9A23-1FD3431D583B
3052,20429,2,1,F9E28C1B-6942-488E-AB06-CDFA396512FE
3053,20432,2,1,70A01C5A-21D9-4C12-9A23-1FD3431D583B


#### ABW_LRFR_FACTOR_SPEC_EDITION

In [239]:
get_table("BRIDGEWARE", "ABW_LRFR_FACTOR_SPEC_EDITION")

,bridge_id,lrfr_factor_id,factor_spec_edition_id,specification_edition_guid
0,19488,1,1,8D0781BB-A593-4537-9A1A-5093E83E03EB
1,123,1,5,E695393F-25FA-4F9D-B35A-6D661E5B251D
2,19488,1,2,BFDF7135-0C38-4698-86A8-6A056FC557F6
3,339,1,5,E695393F-25FA-4F9D-B35A-6D661E5B251D
4,905,1,5,E695393F-25FA-4F9D-B35A-6D661E5B251D
...,...,...,...,...
13610,15343,1,6,025EC606-7B38-430A-95C3-C84F253C7CF7
13611,325,1,3,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E
13612,15347,1,1,8D0781BB-A593-4537-9A1A-5093E83E03EB
13613,15347,1,2,BFDF7135-0C38-4698-86A8-6A056FC557F6


#### ABW_SYS_ANAL_MODULE_HELP_MENU

In [274]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MODULE_HELP_MENU")

,module_guid,help_submenu_id,help_submenu_name,help_file_path,main_engine_help_ind,component_guid
0,0217E0F6-A21A-4133-A9A6-0AB35AF282F0,7,Engine Help,Help\BrassLRFDEngine.chm,T,None
1,0217E0F6-A21A-4133-A9A6-0AB35AF282F0,20,Main Help,Help\GIRDER(LRFD).HLP,F,None
2,0217E0F6-A21A-4133-A9A6-0AB35AF282F0,21,Command Help,Help\GIRDLRFD.HLP,F,None
3,0217E0F6-A21A-4133-A9A6-0AB35AF282F0,22,Technical Help,Help\GIRDER(LRFD) TECHNICAL.HLP,F,None
4,0217E0F6-A21A-4133-A9A6-0AB35AF282F0,23,Engine Properties,Help\girder(lrfd)-properties.chm,F,None
5,245496D6-0A35-46B8-A97F-66CA334E9702,3,Engine Help,Help\BrassLFDEngine.chm,T,None
6,245496D6-0A35-46B8-A97F-66CA334E9702,27,Command Help,Help\Brass.hlp,F,None
7,245496D6-0A35-46B8-A97F-66CA334E9702,28,Capabilities Help,Help\BRASS-GIRDER(STD) Capabilities.hlp,F,None
8,245496D6-0A35-46B8-A97F-66CA334E9702,29,Engine Properties,Help\girder(lfd)-properties.chm,F,None
9,7DF95E9D-6326-4DD7-800C-A04B3AB3191D,2,Engine Help,Help\BrassLFDEngine.chm,T,None


#### ABW_SUB_PILE_FOUND_DEF

In [275]:
get_table("BRIDGEWARE", "ABW_SUB_PILE_FOUND_DEF")

,bridge_id,struct_def_id,found_def_id,cap_width,cap_length,cap_thick,cap_bot_elevation,cap_conc_id,lock_pile_layout_ind,consider_pile_group_action_ind,group_fact_compr_resistance,group_fact_tens_resistance
0,18675,5,1,None,None,None,None,None,None,None,None,None
1,4904,3,1,None,None,None,None,None,None,None,None,None
2,3717,2,1,None,None,None,None,None,None,None,None,None


#### ABW_SYS_SPECIFICATION

In [276]:
get_table("BRIDGEWARE", "ABW_SYS_SPECIFICATION")

,specification_guid,name,descr,analysis_method_type,secondary_analysis_method_type
0,1B7FD1D8-CAC0-4147-A5A9-31FDE5EE6412,AASHTO Std,AASHTO Standard Specification,21702,21701.0
1,2C5AE28F-D0E5-407D-AB21-974F98C67915,AASHTO LRFD,AASHTO LRFD Specification,21703,NaN
2,F3173513-9FB6-43DC-9573-8F6E0712FC57,AASHTO LRFR,AASHTO LRFR Specification,21704,NaN


#### ABW_SYS_SPECIFICATION_EDITION

In [277]:
get_table("BRIDGEWARE", "ABW_SYS_SPECIFICATION_EDITION")

,specification_edition_guid,specification_guid,name,descr,edition_order_number
0,025EC606-7B38-430A-95C3-C84F253C7CF7,F3173513-9FB6-43DC-9573-8F6E0712FC57,"MBE 2nd, LRFD 5th 2010i","AASHTO MBE - 2nd Edition, LRFD 5th Edition wit...",6
1,16340513-4151-4EFE-9318-C45294900922,F3173513-9FB6-43DC-9573-8F6E0712FC57,"MBE 1st 2010i, LRFD 5th","AASHTO MBE - 1st Edition with 2010 Interims, L...",3
2,1AEE14A2-B9B6-496A-8779-267CC0382FFD,F3173513-9FB6-43DC-9573-8F6E0712FC57,"MBE 1st, LRFD 4th 2009i","AASHTO MBE - 1st Edition, LRFD 4th Edition wit...",2
3,1B3E09E6-6861-445E-98FE-3E5AFB66F3F2,F3173513-9FB6-43DC-9573-8F6E0712FC57,"MBE 1st, LRFD 4th 2008i","AASHTO MBE - 1st Edition, LRFD 4th Edition wit...",1
4,4BCE9530-4A05-4AA5-BC70-B06319E80F52,1B7FD1D8-CAC0-4147-A5A9-31FDE5EE6412,"MCEB 1st, Std 16th","AASHTO Standard - 16th Edition, MCEB 1st Edition",3
5,55959202-DD8D-482C-911E-1FB1F8E75CED,2C5AE28F-D0E5-407D-AB21-974F98C67915,LRFD 4th 2009i,AASHTO LRFD Specification - 4th Edition with 2...,2
6,6AA0EF00-FF16-42FD-866D-936BC9BADD8E,F3173513-9FB6-43DC-9573-8F6E0712FC57,"MBE 2nd, LRFD 5th","AASHTO MBE - 2nd Edition, LRFD 5th Edition",5
7,6F9B56BA-D0F2-48E5-B139-693A46DE3052,2C5AE28F-D0E5-407D-AB21-974F98C67915,LRFD 5th 2010i,AASHTO LRFD Specification - 5th Edition with 2...,4
8,7C621221-8E5A-45D7-919B-88A5AA2023B3,2C5AE28F-D0E5-407D-AB21-974F98C67915,LRFD 5th,AASHTO LRFD Specification - 5th Edition,3
9,8B564F41-BEA5-4DC6-9C68-663891734163,1B7FD1D8-CAC0-4147-A5A9-31FDE5EE6412,"MBE 1st 2010i, Std 17th 2003i",AASHTO Standard - 17th Edition with 2003 Inter...,7


#### ABW_BRIDGE_DIAPHRAGM_DEF_LOC

In [278]:
get_table("BRIDGEWARE", "ABW_BRIDGE_DIAPHRAGM_DEF_LOC")

,bridge_id,diaphragm_def_id,location_id,name,stl_shape_id,section_orientation_type,section_loc_type,stl_structural_id,lrfdr_axl_rgd_coef_non_comp,lrfdr_axl_rgd_coef_comp_lt,lrfdr_axl_rgd_coef_comp_st
0,18141,1,13,AB,NaN,NaN,NaN,NaN,None,None,None
1,18141,1,14,CD,1.0,52001.0,52101.0,2.0,None,None,None
2,18141,1,15,AD,1.0,52001.0,52101.0,2.0,None,None,None
3,17952,1,25,AB,NaN,NaN,NaN,NaN,None,None,None
4,17952,1,26,CD,3.0,52001.0,52101.0,2.0,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
33196,20437,1,20,CB,1.0,52001.0,52101.0,1.0,None,None,None
33197,20437,2,5,AB,1.0,52001.0,52101.0,1.0,None,None,None
33198,20437,2,6,CD,1.0,52001.0,52101.0,1.0,None,None,None
33199,20437,2,7,CE,1.0,52001.0,52101.0,1.0,None,None,None


#### ABW_CULVERT_DEF_RC_BOX_SEG

In [279]:
get_table("BRIDGEWARE", "ABW_CULVERT_DEF_RC_BOX_SEG")

,bridge_id,struct_def_id,culvert_def_id,culvert_def_segment_id
0,216,2,1,1
1,219,2,1,1
2,439,2,1,1
3,477,2,1,1
4,483,2,1,1
...,...,...,...,...
1263,20402,1,1,1
1264,20403,2,1,1
1265,20404,1,1,1
1266,20409,2,1,1


#### ABW_MULTIPLE_PRESENCE_FACTORS

In [280]:
get_table("BRIDGEWARE", "ABW_MULTIPLE_PRESENCE_FACTORS")

,bridge_id,multiple_presence_factor_id,num_lanes,num_lanes_greater_than_ind,multiple_presence_factor
0,290,2,2,F,1.00
1,290,3,3,F,0.85
2,290,4,3,T,0.65
3,291,1,1,F,1.20
4,295,2,2,F,1.00
...,...,...,...,...,...
52783,19621,3,3,F,0.85
52784,19621,4,3,T,0.65
52785,19622,1,1,F,1.20
52786,19622,2,2,F,1.00


#### ABW_NAIL_DEF

In [281]:
get_table("BRIDGEWARE", "ABW_NAIL_DEF")

,bridge_id,nail_id,name,descr,length,diameter,pennyweight,last_change_timestamp
0,25,1,20 Pennyweight,None,101.6,3.7592,20d,2003-01-01 12:00:00
1,806,1,20 Penny,None,101.6,3.7592,20d,2010-11-19 20:32:31
2,2095,1,60 d,None,NaN,NaN,60d,2015-06-23 14:19:04
3,13852,1,20 Pennyweight,None,101.6,3.7592,20d,2022-03-10 14:14:33


#### ABW_PIER_BOT_CAP_SEG

In [282]:
get_table("BRIDGEWARE", "ABW_PIER_BOT_CAP_SEG")

,bridge_id,struct_def_id,pier_cap_id,bot_cap_seg_id,seg_type,column_bay_config_type,cantilever_config_type,dist,length,start_elevation,end_elevation,radius,radius_center_dist,radius_center_elevation,segment_loc_type
0,18675,5,1,25,41801,NaN,40601.0,-6.85800,0.00000,0.0000,0.0000,None,None,None,41702
1,18675,5,1,26,41801,NaN,40601.0,-6.85800,0.00000,0.0000,0.0000,None,None,None,41701
2,18675,5,1,27,41801,NaN,40601.0,-6.85800,1.37160,0.0000,0.0000,None,None,None,41702
3,18675,5,1,28,41801,NaN,40601.0,-6.85800,1.37160,0.0000,0.0000,None,None,None,41701
4,18675,5,1,29,41802,41101.0,NaN,-5.48640,1.37160,0.0000,0.0000,None,None,None,41702
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,13291,6,1,11,41802,41101.0,NaN,-4.08432,5.51688,-1.2192,-1.2192,None,None,None,41702
160,13291,6,1,12,41802,41101.0,NaN,1.43256,5.51688,-1.2192,-1.2192,None,None,None,41702
161,13291,6,1,13,41801,NaN,40601.0,6.94944,0.45720,-1.2192,-1.2192,None,None,None,41702
162,13291,6,1,14,41801,NaN,40601.0,7.40664,0.00000,-1.2192,0.0000,None,None,None,41702


#### ABW_PIER_COL_REINF

In [283]:
get_table("BRIDGEWARE", "ABW_PIER_COL_REINF")

,bridge_id,struct_def_id,pier_column_id,reinf_set_id,start_dist,straight_length,reinf_pattern_def_id,hook_at_start_ind,hook_at_end_ind,developed_at_start_ind,developed_at_end_ind,reinf_set_num,follows_profile_ind,head_at_start_ind,head_at_end_ind
0,5564,3,1,1,0.0,4.78789,1,F,F,T,T,1,T,None,None
1,5564,3,2,1,0.0,4.78789,1,F,F,T,T,1,T,None,None
2,5564,3,3,1,0.0,4.78789,1,F,F,T,T,1,T,None,None
3,5564,4,1,1,0.0,4.89585,1,F,F,T,T,1,T,None,None
4,5564,4,2,1,0.0,4.89585,1,F,F,T,T,1,T,None,None
5,5564,4,3,1,0.0,4.89585,1,F,F,T,T,1,T,None,None
6,5564,5,1,1,0.0,4.98793,1,F,F,T,T,1,T,None,None
7,5564,5,2,1,0.0,4.98793,1,F,F,T,T,1,T,None,None
8,5564,5,3,1,0.0,4.98793,1,F,F,T,T,1,T,None,None
9,5564,6,1,1,0.0,4.95300,1,F,F,T,T,1,T,None,None


#### ABW_PIER_COL_REINF_PAT_DETAIL

In [284]:
get_table("BRIDGEWARE", "ABW_PIER_COL_REINF_PAT_DETAIL")

,bridge_id,struct_def_id,pier_column_id,reinf_pattern_def_id,reinf_bar_id,rebar_id,stl_reinf_id,bundle_type,x_coordinate,y_coordinate
0,5564,3,3,1,16,18,1,47601,353.246817,-170.114697
1,5564,3,3,1,17,18,1,47601,244.454401,-306.536115
2,5564,3,3,1,18,18,1,47601,87.244758,-382.244270
3,5564,3,3,1,19,18,1,47601,-87.244758,-382.244270
4,5564,3,3,1,20,18,1,47601,-244.454401,-306.536115
...,...,...,...,...,...,...,...,...,...,...
163,5564,6,3,1,10,18,1,47601,-244.454401,306.536115
164,5564,6,3,1,11,18,1,47601,-87.244758,382.244270
165,5564,6,3,1,12,18,1,47601,87.244758,382.244270
166,5564,6,3,1,13,18,1,47601,244.454401,306.536115


#### ABW_PIER_COL_SHEAR_REINF

In [285]:
get_table("BRIDGEWARE", "ABW_PIER_COL_SHEAR_REINF")

,bridge_id,struct_def_id,pier_column_id,shear_reinf_id,shear_reinf_type,rebar_id,stl_reinf_id,trans_num_legs,long_num_legs,start_dist,num_spaces,bar_spacing,pitch,straight_length
0,5564,3,2,3,47201,13,1,None,None,0.0,None,None,114.3,4.78789
1,5564,3,3,2,47201,13,1,None,None,0.0,None,None,114.3,4.78789
2,5564,3,1,4,47201,13,1,None,None,0.0,None,None,114.3,4.78789
3,5564,4,3,2,47201,13,1,None,None,0.0,None,None,114.3,4.89585
4,5564,5,2,2,47201,13,1,None,None,0.0,None,None,114.3,4.98793
5,5564,5,3,2,47201,13,1,None,None,0.0,None,None,114.3,4.98793
6,5564,6,2,2,47201,13,1,None,None,0.0,None,None,114.3,4.95300
7,5564,6,3,2,47201,13,1,None,None,0.0,None,None,114.3,4.95300
8,5564,4,1,2,47201,13,1,None,None,0.0,None,None,114.3,4.89585
9,5564,4,2,2,47201,13,1,None,None,0.0,None,None,114.3,4.89585


#### ABW_PIER_DEF_CAP

In [286]:
get_table("BRIDGEWARE", "ABW_PIER_DEF_CAP")

,bridge_id,struct_def_id,pier_cap_id,name,cap_config_type,material_type,composition_type,conc_id,stl_structural_id,ahead_span_topelev_config_type,...,back_span_ledge_width,exposure_factor,lock_reinf_ind,lock_geometry_ind,long_skin_reinf_rebar_id,long_skin_reinf_bar_spacing,long_skin_stl_reinf_id,flex_reinf_follows_profile_ind,stirrup_clear_cover,flex_reinf_input_method_type
0,18675,5,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,NaN,NaN,1.0,None,NaN,47901
1,7533,2,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,NaN,NaN,1.0,None,NaN,47901
2,7534,3,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,NaN,NaN,1.0,F,50.8,47901
3,4904,3,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,NaN,NaN,1.0,None,NaN,47901
4,5564,3,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,14.0,457.2,1.0,F,50.8,47901
5,3717,2,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,NaN,NaN,1.0,T,50.8,47901
6,5564,4,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,14.0,457.2,NaN,F,50.8,47901
7,5564,5,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,14.0,457.2,NaN,F,50.8,47901
8,5564,6,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,14.0,457.2,NaN,F,50.8,47901
9,6370,2,1,Cap,40801,29801,40901,1,None,40502,...,None,None,None,None,NaN,NaN,1.0,None,NaN,47901


#### ABW_PIER_DEF_CAP_CONCENT_LOAD

In [287]:
get_table("BRIDGEWARE", "ABW_PIER_DEF_CAP_CONCENT_LOAD")

,bridge_id,struct_def_id,pier_cap_id,concent_load_id,load_direction_type,load_case_type,name,dist,concent_force,concent_moment


#### ABW_PIER_DEF_CAP_DISTRIB_LOAD

In [288]:
get_table("BRIDGEWARE", "ABW_PIER_DEF_CAP_DISTRIB_LOAD")

,bridge_id,struct_def_id,pier_cap_id,distrib_load_id,load_direction_type,load_case_type,name,dist,length,start_force,end_force


#### ABW_MCB_SEG_DECK_RANGE

In [289]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_DECK_RANGE")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,segment_deck_range_id,start_distance,length,distance_at_start,distance_at_end
0,3496,1,1,1,50,0.000000,1.176338,0.0000,0.0000
1,3496,1,1,1,51,1.176338,2.895600,0.0000,0.0000
2,3496,1,1,1,52,4.071937,2.895600,0.0000,0.0000
3,3496,1,1,1,53,6.967538,2.895600,0.0000,0.0000
4,3496,1,1,1,54,9.863138,2.895600,0.0000,0.0000
...,...,...,...,...,...,...,...,...,...
206,4275,2,1,1,147,112.816000,35.480000,6.8580,6.8580
207,4275,2,1,1,148,148.296000,2.440000,6.8580,6.8580
208,4275,2,1,1,149,150.735990,25.325000,6.8580,6.8580
209,4275,2,1,1,150,176.060990,1.220010,6.8580,6.8580


#### ABW_BRIDGE_REF_LINE

In [290]:
get_table("BRIDGEWARE", "ABW_BRIDGE_REF_LINE")

,bridge_id,bridge_ref_line_id,line_type,x,y,z,direction_angle_x,direction_angle_y,direction_angle_z,straight_line_length
0,19690,1,25301,0.0,NaN,0.0,0.0,1.570796,1.570796,13.716000
1,721,2,25302,0.0,NaN,0.0,0.0,1.570796,1.570796,NaN
2,702,2,25301,0.0,NaN,0.0,0.0,1.570796,1.570796,432.467614
3,725,1,25301,0.0,NaN,0.0,0.0,1.570796,1.570796,12.496800
4,17339,1,25301,0.0,NaN,0.0,0.0,1.570796,1.570796,NaN
...,...,...,...,...,...,...,...,...,...,...
46843,19781,1,25301,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46844,19781,2,25302,0.0,NaN,0.0,0.0,1.570796,1.570796,NaN
46845,19781,4,25301,0.0,NaN,0.0,0.0,1.570796,1.570796,22.016618
46846,20409,2,25301,0.0,NaN,0.0,0.0,1.570796,1.570796,NaN


#### ABW_MCB_TENDON_PROFILE_DEF

In [291]:
get_table("BRIDGEWARE", "ABW_MCB_TENDON_PROFILE_DEF")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,tendon_profile_id,name,start_dist_into_start,end_dist_from_end,inflection_point_method_type,profile_assigned_to_type,...,anch_spalling_rebar_id,starting_span,ending_span,sl_std_prior_to_seating,sl_std_anchor_and_couplers,sl_std_elsewhere_length,sl_std_service_limit,distribute_jforce_equal_ind,stage_id,profile_assigned_type_web_id
0,5236,1,1,1,1,Profile,0.0,0.0,56902,57001,...,None,1,1,1390.603539,1396.188292,1303.109073,1340.340761,T,2,None
1,3958,1,1,1,1,Span 1,0.0,0.0,56902,57001,...,None,1,1,1390.603539,1396.188292,1303.109073,1340.340761,T,2,None
2,4275,1,1,1,1,Ramp K (Span 1),0.0,0.0,56901,57001,...,None,1,1,1390.603539,1396.188292,1303.109073,1340.340761,T,2,None
3,4275,1,1,1,2,Ramp K (Spans 2-5),0.0,0.0,56901,57001,...,None,2,5,1390.603539,1396.188292,1303.109073,1340.340761,T,2,None
4,4275,2,1,1,1,Ramp LL,0.0,0.0,56901,57001,...,None,1,5,1390.603539,1396.188292,1303.109073,1340.340761,T,2,None


#### ABW_STL_TRUSS_BUILTUP_XSECTION

In [292]:
get_table("BRIDGEWARE", "ABW_STL_TRUSS_BUILTUP_XSECTION")

,bridge_id,struct_def_id,spng_mbr_def_id,element_xsection_id,web_depth,web_thick,top_offset,bot_offset,web_lacing_ind,top_angle_horz_leg_thick,...,bot_angle_stl_id,web_stl_id,top_angle_stl_shape_id,bot_angle_stl_shape_id,builtup_section_type,struct_tee_shape_id,stl_channel_shape_id,ex_stl_channel_top_ind,channel_stl_id,tee_stl_id
0,14631,2,2,9,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
1,14631,2,8,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
2,14631,2,9,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
3,14631,2,10,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
4,14631,2,11,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
5,14631,2,12,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
6,14631,2,13,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
7,14631,2,14,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
8,14631,2,15,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None
9,14631,2,16,2,NaN,0.0000,NaN,NaN,None,13.716,...,1.0,NaN,NaN,NaN,50002,None,None,None,None,None


#### ABW_MC_BOX_WEB_DISTFACT_RANGE

In [293]:
get_table("BRIDGEWARE", "ABW_MC_BOX_WEB_DISTFACT_RANGE")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,web_id,range_id,dist,length,single_lane_ll_distfactor,multi_lane_ll_distfactor,...,start_multi_lane_distfactor,end_multi_lane_distfactor,dist_factor_variation_type,skew_correction_factor_start,skew_correction_factor_end,start_sngl_lane_distfact_skew,end_sngl_lane_distfact_skew,start_multi_lane_distfact_skew,end_multi_lane_distfact_skew,reference_point_type
0,3496,1,1,1,2,1,0.000000,25.770166,1.070908,1.070908,...,1.070908,1.070908,47001,None,None,None,None,None,None,64301
1,3496,1,1,1,2,2,25.770166,15.562895,1.070908,1.070908,...,1.070908,1.070908,47001,None,None,None,None,None,None,64301
2,3496,1,1,1,2,3,41.333060,18.574581,1.070908,1.070908,...,1.070908,1.070908,47001,None,None,None,None,None,None,64301
3,3496,1,1,1,2,4,59.907641,21.297052,1.070908,1.070908,...,1.070908,1.070908,47001,None,None,None,None,None,None,64301
4,3496,1,1,1,2,5,81.204693,35.492433,1.070908,1.070908,...,1.070908,1.070908,47001,None,None,None,None,None,None,64301
5,3496,1,1,1,2,6,0.000000,32.794575,1.118054,1.173356,...,1.173356,1.173356,47001,None,None,None,None,None,None,64301
6,3496,1,1,1,2,7,32.794575,39.627175,1.118054,1.173356,...,1.173356,1.173356,47001,None,None,None,None,None,None,64301
7,3496,1,1,1,2,8,72.421750,44.275375,1.118054,1.173356,...,1.173356,1.173356,47001,None,None,None,None,None,None,64301
8,3496,1,1,1,2,9,0.000000,116.697125,0.600000,1.000000,...,1.000000,1.000000,47001,None,None,None,None,None,None,64301
9,3496,1,1,1,1,10,0.000000,25.770166,0.910273,0.872732,...,0.872732,0.872732,47001,None,None,None,None,None,None,64301


#### ABW_MBRALT_STLFLNG_LOSS_RANGE

In [294]:
get_table("BRIDGEWARE", "ABW_MBRALT_STLFLNG_LOSS_RANGE")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,range_id,mbralt_stlflng_loss_range_type,top_location_ind
0,2031,1,10,1,1,36505,T
1,2031,1,10,1,2,36505,F
2,5909,3,15,1,2,36505,T
3,5909,3,15,1,3,36505,F
4,5909,3,16,1,2,36505,T
...,...,...,...,...,...,...,...
1386,20260,7,172,1,5,36506,F
1387,20416,14,28,1,2,36505,T
1388,20416,14,28,1,3,36505,F
1389,20416,18,21,1,1,36505,F


#### ABW_MATL_STL_REINF

In [295]:
get_table("BRIDGEWARE", "ABW_MATL_STL_REINF")

,bridge_id,stl_reinf_id,name,descr,si_or_us_type,yield_strength,mod_of_elast,reinf_bar_type,ultimate_strength,last_change_timestamp
0,405,1,Grade 33,Name based on Year-of-Construction,10401,227.526981,199947.953000,28501,NaN,2009-04-14 13:32:48.000000
1,18857,1,Reinforcing Steel 1984 to Now,60 ksi reinforcing steel,10401,413.685420,199947.981999,28501,482.63306,2024-10-30 00:41:44.407000
2,434,1,Grade 60,Name based on Fy from Card Type 14,10401,413.685420,199947.953000,28501,NaN,2009-04-15 12:43:02.000000
3,439,1,Reinforcing Wire Grade 65,for Precast RC Boxes and Frames,10401,448.159205,199947.982000,28501,NaN,2009-04-15 14:53:35.000000
4,440,1,Grade 60,Name based on Year-of-Construction,10401,413.685420,199947.953000,28501,NaN,2009-04-15 15:00:02.000000
...,...,...,...,...,...,...,...,...,...,...
12612,20244,2,Reinforcing Steel 1984 to Now PLAIN,60 ksi reinforcing steel,10401,413.685420,199947.981999,28501,482.63306,2025-02-28 18:50:42.170000
12613,20248,1,Reinforcing Steel 60 ksi,60 ksi reinforcing steel,10401,413.685420,199947.981999,28502,482.63306,2025-05-14 17:40:26.236607
12614,5058,2,Grade 60,60 ksi reinforcing steel,10401,413.685480,199947.981999,28501,620.52822,2025-05-28 18:18:52.710658
12615,20256,1,Grade 60,60 ksi reinforcing steel,10401,413.685480,199947.981999,28502,620.52822,2025-05-27 12:24:13.803351


#### ABW_MATL_STL_STRUCTURAL

In [296]:
get_table("BRIDGEWARE", "ABW_MATL_STL_STRUCTURAL")

,bridge_id,stl_structural_id,name,descr,si_or_us_type,yield_strength,tens_strength,toughness,thermal_exp_coeff,density,mod_of_elast,last_change_timestamp
0,19030,2,ASTM A709,ASTM A 709 50 ksi,10401,344.737850,448.159205,None,0.000012,7849.179078,199947.981999,2024-10-29 11:22:32.580
1,69,1,ASTM A709,ASTM A 709 50 ksi,10401,344.737850,448.159205,None,0.000012,7849.179078,199947.982000,2020-03-31 12:14:22.000
2,19031,2,Structural Steel (1966 to 1990),ASTM A 36,10401,248.211252,399.895906,None,0.000012,7849.179078,199947.981999,2024-11-13 21:04:01.660
3,19032,2,Structural Steel (1966 to 1990),ASTM A 36,10401,248.211252,399.895906,None,0.000012,7849.179078,199947.982000,2020-04-21 12:31:13.000
4,19033,2,ASTM A709,ASTM A 709 50 ksi,10401,344.737850,448.159205,None,0.000012,7849.179078,199947.981999,2024-11-07 13:05:14.240
...,...,...,...,...,...,...,...,...,...,...,...,...
7653,19155,2,Steel - Corrugated,"Structural plate (thickness 0.176""-0.250"")",10401,227.527000,310.264100,None,0.000012,7849.179078,199948.000002,2024-11-19 14:44:33.480
7654,19156,2,Steel - Corrugated,"Structural plate (thickness 0.176""-0.250"")",10401,227.527000,310.264100,None,0.000012,7849.179078,199948.000002,2024-11-19 14:44:33.480
7655,19157,2,Steel - Corrugated,"Structural plate (thickness 0.176""-0.250"")",10401,227.527000,310.264100,None,0.000012,7849.179078,199948.000002,2024-11-19 14:44:33.480
7656,19162,1,Structural Steel (1931 To 1965),ASTM A 570,10401,227.526981,358.527364,None,0.000012,7849.179078,199947.981999,2024-11-04 15:15:05.747


#### ABW_MATL_TIMBER_SAWN

In [297]:
get_table("BRIDGEWARE", "ABW_MATL_TIMBER_SAWN")

,bridge_id,timber_id,timber_species,timber_grading_rule_agency,timber_size_classifcation,timber_commercial_grade,unit_asd_bending_stress,unit_asd_tens_stress_parallel,unit_asd_shear_stress_parallel,unit_asd_comp_stress_perp,unit_asd_comp_stress_parallel,mod_of_elast,unit_lrfd_bending_stress,unit_lrfd_tens_stress_parallel,unit_lrfd_shear_stres_parallel,unit_lrfd_comp_stress_perp,unit_lrfd_comp_stress_parallel,lrfd_mod_of_elast,asd_notes,lrfd_notes
0,16762,1,Douglas Fir-Larch,WWPA,"2"" - 4"" thick, 2"" & wider",Select Structural,9.997398,6.894757,0.655002,4.309223,11.721087,13100.0383,10.342135,6.894757,1.241056,4.309223,11.721087,13100.038300,None,None
1,20225,1,Southern Pine,SPIB,Beams and Stringers,No. 2,12.065825,8.273708,2.068427,4.481592,13.100038,9652.6598,12.065825,8.273708,1.792637,4.481592,13.100038,9652.659800,None,None
2,20225,4,Southern Pine,SPIB,Posts and Timbers,Select Structural,12.065825,8.273708,2.068427,4.481592,13.100038,9652.6598,12.065825,8.273708,1.792637,4.481592,13.100038,9652.659800,None,None
3,18396,1,Douglas Fir-Larch,WCLIB,Beams and Stringers,Select Structural,11.031611,6.550019,5.860543,4.309223,7.584233,11031.6112,11.031611,6.550019,5.860543,4.309223,7.584233,11031.611200,None,None
4,806,1,Southern Pine,SPIB,"2"" - 4"" thick, 2"" - 4"" wide",No. 2,10.342135,5.688175,0.620528,3.895538,11.376349,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,20016,1,Southern Pine,SPIB,Beams and Stringers,No. 2,15.857941,10.686873,1.792637,5.102120,15.857941,13100.0383,15.857941,10.686873,1.792637,5.102120,15.857941,13100.038300,None,None
81,20016,4,Southern Pine,SPIB,Posts and Timbers,Select Structural,15.857941,10.686873,1.792637,5.102120,15.857941,13100.0383,15.857941,10.686873,1.792637,5.102120,15.857941,13100.038300,None,None
82,17434,2,Douglas Fir-Larch,WCLIB,Posts and Timbers,No. 2,5.171068,3.275010,0.586054,4.309223,4.826330,8963.1841,5.171068,3.275010,1.172109,4.309223,4.826330,8963.184100,None,None
83,17759,1,Mixed Southern Pine,SPIB,"2"" - 4"" thick, 10"" wide",No. 2,6.377650,3.792116,0.620528,3.895538,9.997398,9652.6598,NaN,NaN,NaN,NaN,NaN,NaN,None,None


#### ABW_BRIDGE_SUB_LRFD_DS_LS_VEH

In [298]:
get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFD_DS_LS_VEH")

,bridge_id,lrfd_design_setting_id,design_setting_ls_info_id,ds_ls_vehicle_info_id,design_setting_vehicle_info_id,adjacent_lane_vehicle_info_id
0,5564,1,1,1,1,None
1,5564,1,3,1,1,None
2,5564,1,4,1,1,None
3,5564,1,5,1,1,None
4,5564,1,6,1,1,None
5,5564,1,7,1,1,None
6,5564,1,8,1,1,None
7,5564,1,9,1,1,None
8,5564,1,10,1,2,None
9,5564,1,11,1,1,None


#### ABW_BRIDGE_SUB_LRFD_DS_VEHICLE

In [299]:
get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFD_DS_VEHICLE")

,bridge_id,lrfd_design_setting_id,design_setting_vehicle_info_id,vehicle_id,consider_dsgn_tandem_pair_ind
0,5564,1,1,2,F
1,5564,1,2,11,None
2,3717,1,1,2,F
3,3717,1,2,11,None
4,5478,1,3,2,F
5,5478,1,4,11,None
6,13095,1,1,2,F
7,13095,1,2,11,None
8,18262,1,7,2,T
9,18262,1,8,11,None


#### ABW_MULTI_CELL_BOX_XSEC_CELL

In [300]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_XSEC_CELL")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,xsection_id,cell_id,cell_num,s_dimension,top_right_web_thick,bot_right_web_thick,top_slab_thick
0,5236,1,1,1,1,1,1,0.8382,152.4,152.4,215.9
1,5236,1,1,1,1,2,2,0.7620,152.4,152.4,215.9
2,5236,1,1,1,1,3,3,0.7620,152.4,152.4,215.9
3,5236,1,1,1,1,4,4,0.7620,152.4,152.4,215.9
4,5236,1,1,1,1,5,5,0.7620,152.4,152.4,215.9
...,...,...,...,...,...,...,...,...,...,...,...
77,4275,2,1,1,3,2,2,3.8680,380.0,380.0,475.0
78,4275,2,1,1,4,1,1,4.5380,380.0,380.0,475.0
79,4275,2,1,1,4,2,2,4.3050,380.0,380.0,475.0
80,4275,2,1,1,6,1,1,4.3050,380.0,380.0,475.0


#### ABW_MULTI_CELL_BOX_XSECT_RANGE

In [301]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_XSECT_RANGE")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,xsection_range_id,start_xsection_id,end_xsection_id,depth_vary_type,solid_section_ind,start_distance,length
0,5236,1,1,1,1,1,1,56201,T,0.000000,12.788900
1,3496,1,1,1,1,1,1,56201,F,0.000000,1.176338
2,3496,1,1,1,2,2,2,56201,F,1.176338,2.895600
3,3496,1,1,1,3,2,2,56201,F,4.071937,2.895600
4,3496,1,1,1,4,2,2,56201,F,6.967538,2.895600
...,...,...,...,...,...,...,...,...,...,...,...
206,4275,2,1,1,9,6,6,56201,T,148.296000,2.440000
207,4275,2,1,1,10,6,6,56201,F,150.735990,25.325000
208,4275,2,1,1,11,6,6,56201,T,176.060990,1.220010
209,3958,1,1,1,53,22,23,56202,F,255.422400,4.876800


#### ABW_FLNG_LAT_BEND_STRESS_LOC

In [302]:
get_table("BRIDGEWARE", "ABW_FLNG_LAT_BEND_STRESS_LOC")

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,flng_lat_bend_stress_id,flng_lat_bend_stress_loc_id,girder_sys_diaph_id,girder_line_diaph_id,support_num,start_distance,top_flng_lat_bending_stress,bot_flng_lat_bending_stress
0,16601,2,2,1,1,19,10.0,NaN,2,16.509992,0.000000,0.000000
1,16601,2,2,1,1,20,25.0,NaN,2,16.509992,0.000000,0.000000
2,16601,2,2,1,1,21,26.0,NaN,3,0.558811,0.000000,0.000000
3,16601,2,2,1,1,22,11.0,NaN,3,0.558811,0.000000,0.000000
4,16601,2,2,1,1,23,27.0,NaN,3,3.860810,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
6029,20338,1,4,1,6,9,NaN,9.0,2,13.614441,35.990632,-35.990632
6030,20338,1,4,1,6,10,NaN,10.0,3,2.102541,26.131129,-26.131129
6031,20338,1,4,1,6,11,NaN,11.0,3,6.109381,-28.613242,28.613242
6032,20338,1,4,1,6,12,NaN,12.0,3,10.116221,-36.955898,36.955898


#### ABW_MULTI_CELL_BOX_XSECTION

In [268]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_XSECTION")

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,xsection_id,name,input_method_type,section_entry_method_type,top_slab_conc_id,other_parts_conc_id,...,ext_top_web_fillet_ind,ext_bot_web_fillet_ind,int_top_web_fillet_ind,int_bot_web_fillet_ind,fillet_top_horz,fillet_bot_horz,fillet_top_vert,fillet_bot_vert,slope_one,slope_two
0,5236,1,1,1,1,Slab Section,56301,56401,1,1,...,F,F,F,F,NaN,NaN,NaN,NaN,None,None
1,3496,1,1,1,1,Exp. Joint Segment,56302,56401,1,1,...,T,F,F,F,203.2,NaN,203.2,NaN,None,None
2,3496,1,1,1,2,"9' 6"" Segment",56302,56401,1,1,...,T,F,F,F,203.2,NaN,203.2,NaN,None,None
3,3496,1,1,1,3,"8' 8"" Segment",56302,56401,1,1,...,T,F,F,F,203.2,NaN,203.2,NaN,None,None
4,3496,1,1,1,4,"6' 4"" Segment",56302,56401,1,1,...,T,F,F,F,203.2,NaN,203.2,NaN,None,None
5,3496,1,1,1,5,"5' 0"" Segment",56302,56401,1,1,...,T,F,F,F,203.2,NaN,203.2,NaN,None,None
6,3958,1,1,1,15,1B,56301,56401,1,1,...,T,T,F,F,1981.2,1524.0,406.4,266.7,None,None
7,3958,1,1,1,16,2B,56301,56401,1,1,...,T,T,F,F,1981.2,1524.0,406.4,266.7,None,None
8,3958,1,1,1,17,3B,56301,56401,1,1,...,T,T,F,F,1981.2,1524.0,406.4,266.7,None,None
9,3958,1,1,1,18,4B,56301,56401,1,1,...,T,T,F,F,1981.2,1524.0,406.4,266.7,None,None


#### ABW_TENDON_PROFILE_DEF_WEBDUCT

In [269]:
get_table("BRIDGEWARE", "ABW_TENDON_PROFILE_DEF_WEBDUCT")

,bridge_id,struct_def_id,spng_mbr_def_id,tendon_profile_id,web_duct_id,duct_num,strands_per_duct
0,18358,1,1,1,39,None,12
1,18358,1,1,3,8,None,12
2,18358,1,1,4,6,None,12
3,18358,1,1,9,15,None,11
4,18358,1,1,9,16,None,11
...,...,...,...,...,...,...,...
61,18796,3,2,1,3,None,9
62,18796,3,2,2,3,None,9
63,18796,3,2,3,5,None,9
64,18796,3,2,4,27,None,12


#### ABW_CULVDEF_SEG_ANAL_PT_RC

In [303]:
get_table("BRIDGEWARE", "ABW_CULVDEF_SEG_ANAL_PT_RC")

,bridge_id,struct_def_id,culvert_def_id,culvert_def_segment_id,anal_pt_id,rcbox_component_type,rcbox_cell_id,rcbox_dist_left_edge_bot_wall
0,18450,1,1,1,1,51401,NaN,0.0
1,1724,1,1,1,1,51404,1.0,NaN
2,2593,1,1,1,1,51401,NaN,0.0
3,2594,1,1,1,1,51401,NaN,0.0
4,2595,1,1,1,1,51401,NaN,0.0
...,...,...,...,...,...,...,...,...
486,13448,1,1,1,2,51401,1.0,0.0
487,13535,1,1,1,1,51401,NaN,0.0
488,18090,1,1,1,1,51401,1.0,0.0
489,17288,1,1,1,1,51401,NaN,0.0


#### ABW_PS_SHAPE

In [304]:
get_table("BRIDGEWARE", "ABW_PS_SHAPE")

,bridge_id,ps_shape_id,name,description,si_or_us_type,ps_shape_type,last_change_timestamp,user_defined_properties_ind
0,1877,1,OH-B33-36,"Ohio 33"" x 36"" Non-composite Box Beam",10401,23002,2013-07-11 13:18:56.000000,None
1,19464,1,CB27-48,"Prestressed Concrete Box Beam 27""x48"".",10401,23002,2024-09-16 14:58:40.997429,F
2,19468,3,CB33-48,"Prestressed Concrete Box Beam 33""x48""",10401,23002,2024-09-25 13:20:57.878000,F
3,19470,3,CB33-48,"Prestressed Concrete Box Beam 33""x48""",10401,23002,2024-09-25 13:20:57.878000,F
4,1876,1,OH-B17-36,"Ohio 17"" x 36"" Non-composite Box Beam",10401,23002,2013-07-10 13:18:54.000000,None
...,...,...,...,...,...,...,...,...
3730,18719,1,CB21-48,"Ohio 21"" x 48"" Composite Box Beam",10401,23002,2024-09-03 17:49:04.246000,F
3731,18721,1,B42-48,"Ohio 42"" x 48"" Non-composite Box Beam",10401,23002,2024-10-15 16:28:09.673000,F
3732,18723,2,B12-48,"Ohio 12"" x 48"" Non-composite Box Beam",10401,23002,2024-10-31 00:06:51.667000,F
3733,19462,1,CB27-48,"Prestressed Concrete Box Beam 27""x48"".",10401,23002,2024-09-16 14:58:40.997429,F


#### 

In [272]:
get_table("BRIDGEWARE", "ABW_DECK_CONC_PANEL")

,bridge_id,struct_def_id,deck_panel_id,conc_id,eff_thick,actual_thick,left_overhang_x1,left_overhang_x2,left_overhang_x3,left_overhang_x4,...,right_overhang_x2,right_overhang_x3,right_overhang_x4,right_overhang_x5,right_overhang_y1,right_overhang_y2,right_overhang_y3,right_overhang_y4,right_overhang_y5,precast_ind
0,3898,1,1,1.0,NaN,190.50,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,3908,1,1,2.0,NaN,215.90,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,3808,3,1,1.0,NaN,184.15,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,3912,1,1,NaN,NaN,NaN,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,3917,1,1,1.0,NaN,NaN,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13135,20434,1,1,NaN,NaN,NaN,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
13136,20435,1,1,NaN,NaN,NaN,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
13137,20437,3,1,1.0,245.0,245.00,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
13138,20438,1,1,NaN,NaN,NaN,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


#### 

In [273]:
get_table("BRIDGEWARE", "ABW_SPNG_MBR_DEF")

,bridge_id,timber_id,combination_type,timber_species_outer,timber_species_core,asd_perp_tens_zone_stress_tens,asd_perp_comp_zone_stress_tens,asd_perp_tension_face,asd_perp_compression_face,asd_perp_shear_parallel_gain,...,lrfd_para_shear_para_grain,lrfd_para_mod_of_elasticity,lrfd_para_tens_para_grain,lrfd_axial_tens_para_grain,lrfd_axial_comp_para_grain,lrfd_axial_mod_of_elasticity,lrfd_fast_side_face,lrfd_fast_top_bottom_face,asd_notes,lrfd_notes


#### 

In [ ]:
get_table("BRIDGEWARE", "ABW_DESIGNRATIO_RSLTS_SUMMARY")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_SHAPE")

In [ ]:
get_table("BRIDGEWARE", "ABW_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_FOUND_ALT")

In [ ]:
get_table("BRIDGEWARE", "ABW_STRIPED_LANE_POSITION")

In [ ]:
get_table("BRIDGEWARE", "ABW_STRUCT_DEF_DISTRIB_LOAD")

In [ ]:
get_table("BRIDGEWARE", "ABW_STRUCT_DEF_REF_LINE")

In [ ]:
get_table("BRIDGEWARE", "ABW_STRUCT_DEF_REF_LINE_HORZ")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_ANAL_OPTION_SUBS_LOAD")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_DR_SHAFT_ANAL_AXIAL")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_DR_SHAFT_ANAL_ELEV")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_DR_SHAFT_CURVE_ELEV")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_DR_SHAFT_FLEX_REINF")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_DR_SHAFT_FOUND_LAYER")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_DR_SHAFT_PY")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBRALT_FLNGANGL_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBRALT_FLNGFLAT_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBRALT_STL_TRUSS_ELEMENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBRALT_STLBM_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBRALT_STLCPLT_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBRALT_STLWEB_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSALT_LRFD_LOAD_FACTOR")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_BEARING_LINE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_BEARING_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_CLOSED_BOX_BEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_LRFR_LOAD_FACTOR_TABLE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_BRDR_COLUMN")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_BRDR_FOREIGN_KEY")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPSTRUCTDEF_BRACING")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFD_RANGE_APP_VALUE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_FACTOR")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_LEGAL_LOADS")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_LEGAL_LOADS_HAUL")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_LF_TABLE_VALUE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_LOAD_FACTOR")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_LS")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_PERMIT_LIMITED")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_PERMIT_ROUTINE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_CONC")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_IRON")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_PS_BAR")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_STL_PS")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_STL_REINF")

In [ ]:
get_table("BRIDGEWARE", "ABW_TIMBER_RECT_BEAM_SHAPE")

In [ ]:
get_table("BRIDGEWARE", "ABW_TOP_FLNG_LAT_SUP_LOC_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_TOP_FLNG_LAT_SUPPORT_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_TRUSS_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_MBR_SPAN")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_SPNG_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_BDEF_WEB")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_CONC_RAILING_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_STL_RAILING_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_WINDBRAC_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_SUB_STRUCT_INTERFACE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_ENVIR_LOAD_DETAIL")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_ENVIRONMENTAL_LOAD")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_FR_FORCE_SUB_DETAIL")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_LL_DIST_SUB_WDLA")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_LL_DIST_SUB_WODLA")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_LL_PATTERN_SUB_ITEM")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_CONC_DECK_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_CONTRAFLEX_POINTS")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_DECK_REINF_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_DIAPH_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_GENERIC_DECK_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_GLINE_SUPPORT")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_LEG_BRACE_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_SHEAR_CONN_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MA_STRS_IE_ANG_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MA_STRS_IE_CHAN_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MA_STRS_IE_CPLT_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_BEARING_SPHERICAL")

In [ ]:
get_table("BRIDGEWARE", "ABW_MA_STRS_IE_FLNG_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MA_STRS_IE_WEB_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBRALT_STLIBM_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_WELD")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_NAIL")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_PS_BOX_BEAM")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_PS_IBEAM")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_PS_SHAPE_MILD_STEEL")

In [ ]:
get_table("BRIDGEWARE", "PARAMTRS")

In [ ]:
get_table("BRIDGEWARE", "ROADWAY")

In [ ]:
get_table("BRIDGEWARE", "USERS")

In [ ]:
get_table("BRIDGEWARE", "ABW_AGENCY_STREAM_FLOW_EFFECT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SLAB_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_LRFD_RANGE_OF_APP_ROW")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_LRFD_RANGE_APP_LABEL")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_SUPER_STRUCT_STAGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_TABLE_RELATIONSHIPS")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_TYPE_CATEGORY")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_TEND_PROF_DEF_WEBDUCT")

In [ ]:
get_table("BRIDGEWARE", "ABW_TFS_FLOORLINE_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_TFS_FLOORSYS_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_TF_FLOORLINE_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_TF_FLOORSYS_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_TRUSS_DEF_COMMAND")

In [ ]:
get_table("BRIDGEWARE", "ABW_TRUSS_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_WALL_SUB_STRUCT")

In [ ]:
get_table("BRIDGEWARE", "ABW_WALL_SUB_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_WIND_BRACE_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_PLATE")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_RAILING")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_RAILING_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_RIVET_BOLT_CONN")

In [ ]:
get_table("BRIDGEWARE", "ABW_MULTICELL_BOX_TENDON_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_LRFD_OVERRIDE_CAP")

In [ ]:
get_table("BRIDGEWARE", "ABW_FRAME_SYS_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_FLOOR_BEAM_MBR_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_STRGRP_DIAPH")

In [ ]:
get_table("BRIDGEWARE", "ABW_BEAM_SHEAR_REINF_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANALYSIS_MODULE")

In [ ]:
get_table("BRIDGEWARE", "ABW_BEARING_TIMBER")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_COMPONENT_ERRORS")

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_ANALPT_LRFD_OVRD_CAP")

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_ANALPT_LRFR_OVRD_CAP")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFD_FACT_SPEC_EDITION")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_FACT_SPEC_EDITION")

In [ ]:
get_table("BRIDGEWARE", "ABW_TRUSS_ELEMENT_XSECTION")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_RC_PILE_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_SETTLEMENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_SH_LOAD")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_SPNG_MBR_BRNG_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_STREAM_FLOW")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_ANALYSIS_EVENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_ANAL_EVENT_TEMPLATE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_ANAL_OPTION_LS_VEHICLE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_ANAL_REPORTS_TEMPLATE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_DEF_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_DEF_PILE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_FK")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_STL_PILE_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_DEF_WALL_SHAFT_XSECT")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_ENCASEMENT_WALL_XSEC")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_ENCSW_USERDEF_XSECT")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_RC_COL_USERDEF_XSEC")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_SUBSDEF_COLUMN_XSEC")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_SUB_STRUCT")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_WALL_SHAFT_COL")

In [ ]:
get_table("BRIDGEWARE", "ABW_FS_FLOORLINE_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_FS_FLOORSYS_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_GFS_FLOORLINE_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_GFS_FLOORSYS_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_GF_FLOORLINE_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_GF_FLOORSYS_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_GIRDER_FLRBM_STRUCT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_GIRDER_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_GIRDER_SYS_STRUCT_DEF_FK")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_FLNG_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_IBEAM_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_LEG_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_NON_DETAILED_BEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_PLATE_IBEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_ROLLED_IBEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_SCH_CPLATE_LOSS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_WSHFT_ROUNDNOSE_XSECT")

In [ ]:
get_table("BRIDGEWARE", "ABW_GENPREF_TEMPLATE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_METAL_BOX_CULVERT")

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_SUPPORT")

In [ ]:
get_table("BRIDGEWARE", "ABW_BRACKET_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_COMMENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_FK")

In [ ]:
get_table("BRIDGEWARE", "ABW_DETAIL_TRUSS_DEF_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SPNG_MBR_ALT_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_LRFR_OVERRIDE_CAP")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TIMBER_SAWN")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SPEC")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SPEC_ARTICLE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_STANDARD_LOAD_GROUP")

In [ ]:
get_table("BRIDGEWARE", "ABW_STRUCT_DEF_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SUB_ANALYSIS_OPTIONS")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SUB_LRFD_DS_LS_VEHICLE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SUB_LRFD_DS_VEHICLE")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_WSHFT_USERXSECT_COORD")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_EVENT_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_EVENT_COMP_TEMPLATE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MOD_SPEC_EDITION")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MODULE_FEATURE")

In [ ]:
get_table("BRIDGEWARE", "ABW_BOT_FLANGE_LAT_BRACING_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_TENDON_DEF_WEB_JFORCE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LFR_FACTOR")

In [ ]:
get_table("BRIDGEWARE", "ABW_LFR_FACTOR_SPEC_EDITION")

In [ ]:
get_table("BRIDGEWARE", "ABW_LFR_LOADING")

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_TIMBER")

In [ ]:
get_table("BRIDGEWARE", "ABW_ANALYSIS_EVENT_TEMPLATE")

In [ ]:
get_table("BRIDGEWARE", "ABW_BAR_MARK_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_STL")

In [ ]:
get_table("BRIDGEWARE", "ABW_GUSSET_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_FRAME_MBR_ALT_LEG")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_STRGRPDEF_TEMPLATE_FK")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_STRINGER_UNIT_LAYOUT")

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_SPNG_MBR_ALT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MODULE_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_BRIDGE_EXPLORER_COLUMN")

In [ ]:
get_table("BRIDGEWARE", "ABW_USER_PROFILE")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_ROLLED_SHAPE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_UNIT_CATEGORY")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_SECTION_PROPERTY")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_ANALYSIS_OPTIONS_LS")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_ANAL_OPTION_LS_VEHICLE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_ANAL_OPTION_VEHICLE")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_CONC_COORD_SHAPE")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_SPLICE_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_SPLICE_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_AGENCY_BRIDGE_DESIGN_PARAM")

In [ ]:
get_table("BRIDGEWARE", "ABW_ADV_CONC_BEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVDEF_RCBOX_SEG_REINPROF")

In [ ]:
get_table("BRIDGEWARE", "ABW_ADV_CONC_PT_BEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT")

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_PS_CONC")

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_ANAL_PT")

In [ ]:
get_table("BRIDGEWARE", "ABW_TRUSSDEF_ELEMENT_DIST_LOAD")

In [ ]:
get_table("BRIDGEWARE", "ABW_UNIT_TOLERANCE")

In [ ]:
get_table("BRIDGEWARE", "ABW_WEARING_SURFACE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCELL_BOX_STRUCT_DEF_FK")

In [ ]:
get_table("BRIDGEWARE", "ABW_WELD_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_WIND_LOAD")

In [ ]:
get_table("BRIDGEWARE", "ABW_EVENT_LRFD_FACTOR")

In [ ]:
get_table("BRIDGEWARE", "ABW_EVENT_LRFD_LS")

In [ ]:
get_table("BRIDGEWARE", "ABW_SPNG_MBR_ALT_EVENTS")

In [ ]:
get_table("BRIDGEWARE", "ABW_RESULTS_DL_CASE")

In [ ]:
get_table("BRIDGEWARE", "ABW_DECK_PANEL_ANALYSIS_EVENTS")

In [ ]:
get_table("BRIDGEWARE", "ABW_INTEREST_PT")

In [ ]:
get_table("BRIDGEWARE", "ABW_LRFD_SPEC_CHECK")

In [ ]:
get_table("BRIDGEWARE", "ABW_LRFD_SPEC_CHECK_COMMENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_ALT_EVENTS_EXPORT_LOG")

In [ ]:
get_table("BRIDGEWARE", "ABW_SPNG_MBR_ALT_EVENT_ERRORS")

In [ ]:
get_table("BRIDGEWARE", "ABW_SPNG_MBR_ALT_EVENTS_REPORT")

In [ ]:
get_table("BRIDGEWARE", "ABW_RESULTS_GENERIC_REPORT")

In [ ]:
get_table("BRIDGEWARE", "ABW_RESULTS_STL_XSEC_REPORT")

In [ ]:
get_table("BRIDGEWARE", "ABW_RESULTS_STL_LS_SUMMARY")

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_WIND_EFFECT")

In [ ]:
get_table("BRIDGEWARE", "ABW_CONC_BMDEF_VOID_SEC_PAT")

In [ ]:
get_table("BRIDGEWARE", "ABW_AGENCY_ENVIRONMENTAL_PARAM")

In [ ]:
get_table("BRIDGEWARE", "ABW_SECONDARY_EVENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_FRAME_LEG_BEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_FRAME_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_FRAME_SPNG_MBR")

In [ ]:
get_table("BRIDGEWARE", "ABW_FRAME_SYS_STRUCT_DEF_FK")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_GIRDER_MBR_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_STRINGER_GROUP_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_STRINGER_MBR_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_SUPPORT_LINE")

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_TRUSS_MBR_LOC")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_ENCSW_USERXSECT_COORD")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_RC_COL_CIRC_XSEC")

In [ ]:
get_table("BRIDGEWARE", "ABW_COMMAND_STL_TRUSS_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUP_STR_THR_D_NODE_GEN_TOL")

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_WSHFT_USERDEF_XSECT")

In [ ]:
get_table("BRIDGEWARE", "ABW_PROJECT")

In [ ]:
get_table("BRIDGEWARE", "ABW_PROJECT_DETAIL")

In [ ]:
get_table("BRIDGEWARE", "ABW_PROJECT_ENGINEER")

In [ ]:
get_table("BRIDGEWARE", "ABW_PROJECT_GROUP")

In [ ]:
get_table("BRIDGEWARE", "ABW_PROJECT_GROUP_DETAIL")

In [ ]:
get_table("BRIDGEWARE", "ABW_PS_PRECAST_BOX_BEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_PS_PRECAST_IBEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_PS_PRECAST_SLAB_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_PS_PRECAST_TEE_BEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_PS_PRECAST_UBEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFD_RANGE_OF_APP")

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFD_RANGE_OF_APP_ROW")

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_ARCH_CULVERT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_PIPE_CULVERT_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_DR_SHAFT_REINF_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_TRUSS_GUSSET_PLATE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_MSEG_TND_DEF_JFORCE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_MSEG_TND_PRF_ANCH_REIN")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_MSEG_TND_PRF_DEF_DET")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_MSEG_TND_PRF_DEF_WBDCT")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_MULTI_SEG_TND_PRF_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_MULTISEG_SLAB_REINF")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_MULTISEG_TENDON_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_LL_DISTFACT_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_TEND_PROF_DEF_DET")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_TENDON_PROFILE_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_TENDON_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_THICKNESS_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_TND_PRF_ANCH_REINF")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_TND_PRF_DEF_WEBDCT")

In [ ]:
get_table("BRIDGEWARE", "ABW_STRUCT_DEF_EXC_ANAL_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_BRM_VEHICLE")

In [ ]:
get_table("BRIDGEWARE", "ABW_BOT_FLNG_LAT_SUP_LOC_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_ANAL_PT")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_ANAL_PT_COMPONENT")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_ANAL_PT_CONC_REINF")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_ANAL_PT_PS_CONC")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_WEB_ANAL_PT_REINF_CONC")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCBW_ANAL_PT_LRFD_OVR_CAP")

In [ ]:
get_table("BRIDGEWARE", "ABW_MCBW_ANAL_PT_LRFR_OVR_CAP")

In [ ]:
get_table("BRIDGEWARE", "ABW_STLBD_BOTFL_LSUP_LOC_RANGE")

In [ ]:
get_table("BRIDGEWARE", "ABW_TIMBER_GLULAM_BEAM_DEF")

In [ ]:
get_table("BRIDGEWARE", "ABW_MATL_TIMBER_GLULAM")

### CTXSYS

In [ ]:
Tables in schema CTXSYS:

DR$NUMBER_SEQUENCE
DR$OBJECT_ATTRIBUTE
DR$POLICY_TAB
DR$THS
DR$THS_PHRASE

### MDSYS

In [ ]:
Tables in schema MDSYS:
NTV2_XML_DATA
OGIS_GEOMETRY_COLUMNS
OGIS_SPATIAL_REFERENCE_SYSTEMS
SDO_COORD_AXES
SDO_COORD_AXIS_NAMES
SDO_COORD_OPS
SDO_COORD_OP_METHODS
SDO_COORD_OP_PARAMS
SDO_COORD_OP_PARAM_USE
SDO_COORD_OP_PARAM_VALS
SDO_COORD_OP_PATHS
SDO_COORD_REF_SYS
SDO_COORD_SYS
SDO_CRS_GEOGRAPHIC_PLUS_HEIGHT
SDO_CS_SRS
SDO_DATUMS
SDO_DATUMS_OLD_SNAPSHOT
SDO_DIST_METADATA_TABLE
SDO_ELLIPSOIDS
SDO_ELLIPSOIDS_OLD_SNAPSHOT
SDO_FEATURE_USAGE
SDO_GEOR_PLUGIN_REGISTRY
SDO_GEOR_XMLSCHEMA_TABLE
SDO_INDEX_HISTOGRAM_TABLE
SDO_PREFERRED_OPS_SYSTEM
SDO_PREFERRED_OPS_USER
SDO_PRIME_MERIDIANS
SDO_PROJECTIONS_OLD_SNAPSHOT
SDO_SRIDS_BY_URN
SDO_SRIDS_BY_URN_PATTERN
SDO_TIN_PC_SEQ
SDO_TIN_PC_SYSDATA_TABLE
SDO_UNITS_OF_MEASURE
SDO_WS_CONFERENCE
SDO_WS_CONFERENCE_PARTICIPANTS
SDO_WS_CONFERENCE_RESULTS
SDO_XML_SCHEMAS
SRSNAMESPACE_TABLE
RDF_PARAMETER
SDO_CS_CONTEXT_INFORMATION
SDO_GEOR_DDL__TABLE$$
SDO_GR_MOSAIC_0
SDO_GR_MOSAIC_1
SDO_GR_MOSAIC_2
SDO_GR_MOSAIC_3
SDO_GR_MOSAIC_CB
SDO_GR_PARALLEL
SDO_GR_RDT_1
SDO_ST_TOLERANCE
SDO_TOPO_DATA$
SDO_TOPO_RELATION_DATA
SDO_TOPO_TRANSACT_DATA
SDO_TTS_METADATA_TABLE
SDO_TXN_IDX_EXP_UPD_RGN
SDO_TXN_JOURNAL_GTT
SDO_TXN_JOURNAL_REG
SDO_WFS_LOCAL_TXNS

### ODOTREF

In [ ]:
Checking for tables in schema: ODOTREF

Tables in schema ODOTREF:
DISTRICT_ADDRESS

### SYS

In [ ]:
Checking for tables in schema: SYS

Tables in schema SYS:
AV_DUAL
DUAL
HS$_PARALLEL_METADATA
HS_BULKLOAD_VIEW_OBJ
HS_PARTITION_COL_NAME
HS_PARTITION_COL_TYPE
SCHEDULER_FILEWATCHER_QT
STMT_AUDIT_OPTION_MAP
SYSTEM_PRIVILEGE_MAP
TABLE_PRIVILEGE_MAP
USER_PRIVILEGE_MAP
AW$AWCREATE
AW$AWCREATE10G
AW$AWMD
AW$AWREPORT
AW$AWXML
AW$EXPRESS
WRR$_REPLAY_CALL_FILTER
ALL_UNIFIED_AUDIT_ACTIONS
AUDIT_ACTIONS
DATA_PUMP_XPL_TABLE$
FINALHIST$
IMPDP_STATS
KU$NOEXP_TAB
KU$XKTFBUE
KU$_DATAPUMP_MASTER_10_1
KU$_DATAPUMP_MASTER_11_1
KU$_DATAPUMP_MASTER_11_1_0_7
KU$_DATAPUMP_MASTER_11_2
KU$_DATAPUMP_MASTER_12_0
KU$_DATAPUMP_MASTER_12_2
KU$_LIST_FILTER_TEMP
KU$_LIST_FILTER_TEMP_2
KU$_SHARD_DOMIDX_NAMEMAP
MODELGTTRAW$
ODCI_PMO_ROWIDS$
ODCI_SECOBJ$
ODCI_WARNINGS$
PLAN_TABLE$
SAM_SPARSITY_ADVICE
SPD_SCRATCH_TAB
WRI$_ADV_ASA_RECO_DATA
WRI$_HEATMAP_TOPN_DEP1
WRI$_HEATMAP_TOPN_DEP2
XS$VALIDATION_TABLE

### SYSTEM

In [ ]:
Checking for tables in schema: SYSTEM

Tables in schema SYSTEM:
HELP
MVIEW$_ADV_INDEX
MVIEW$_ADV_PARTITION
PBCATCOL
PBCATEDT
PBCATFMT
PBCATTBL
PBCATVLD
MVIEW$_ADV_OWB
OL$
OL$HINTS
OL$NODES

### XDB

In [ ]:
Checking for tables in schema: XDB

Tables in schema XDB:
XDB$IMPORT_NM_INFO
XDB$IMPORT_PT_INFO
XDB$IMPORT_QN_INFO
XDB$IMPORT_TT_INFO
XDB_INDEX_DDL_CACHE
XDB$XIDX_IMP_T

### ODOTREF

In [ ]:
Tables in schema ODOTREF:
DISTRICT_ADDRESS